# Dictionary Discovery v19: Unified Embedding Space**Changes from v18**:- **Unified embedding approach**: Vocabulary and seed terms embedded together in same context- **Direct cosine similarity**: No query-time re-encoding, uses pre-computed embeddings- **Better consistency**: More reliable similarity scores and reproducible resultsa marker weights: 0.70 -> 0.55 (reduced uniform baseline)- Weight tiers expanded: 5 -> 7 (better differentiation)- Added +38 problem-specific terms- Expected: Max scores 0.72-0.80, Std 0.14-0.17, High confidence ~27%

## Weighted Dictionary Curation Note

**IMPORTANT**: When curating the dictionary, preserve seed weights!

The seed dictionary contains weights (0.7-1.0) that indicate semantic importance.
During curation, these weights must be preserved and extended to discovered terms.

### Weight Preservation Code

Add this to your curation code:

```python
# Load seed weights from original dictionary
seed_df = pd.read_csv(CONFIG["paths"]["dictionary_excel"])
seed_weights = {}
if "weight" in seed_df.columns:
    seed_weights = dict(zip(seed_df["keyword"], seed_df["weight"]))

# After curating candidates, add weight column
default_discovered = CONFIG["weights"]["default_discovered_weight"]  # 0.8
default_core = CONFIG["weights"]["default_core_weight"]  # 1.0

def assign_weight(term):
    if term in seed_weights:
        return seed_weights[term]  # Preserve original seed weight
    else:
        return default_discovered  # New discovered term

curated_df["weight"] = curated_df["term"].apply(assign_weight)

# Save with weights preserved
curated_df.to_csv(output_path, index=False)
```

### Weight System

- **Seed terms**: Keep original weights from dictionary (0.7-1.0)
- **Discovered terms**: Get default weight (0.8)
- **Vector building**: Combines seed weight × SIF weight

This ensures your manually curated semantic importance is preserved through the entire pipeline!


# Dictionary Discovery v10 - Multi-Output Ordinal Classification

## Major Architecture Change: Binary to Ordinal

### Why v10?
v8 had a critical bug: training with soft labels (0.18-0.59) but evaluating as binary.
v9 fixed this with binary hard labels (0/1 at 0.35 threshold).

**v10 takes a more sophisticated approach:**

### Multi-Output Ordinal Classification

Instead of binary (relevant/not relevant), we predict **3 ordinal classes per topic**:

- **Low**: cosine < 0.30 (weak relevance)
- **Medium**: cosine 0.30-0.40 (moderate relevance)
- **High**: cosine >= 0.40 (strong relevance)

### Architecture:



Each chunk gets a **pattern signature**: e.g., "Med-High-Med"

### Benefits over Binary:

1. **More expressive**: Captures 27 possible patterns (observed: 17 in training data)
2. **Data efficient**: 366 samples train each topic head independently
3. **Ordinal structure**: "Low vs High" penalized more than "Med vs High"
4. **Interpretable**: "High Colonial + Medium Slavery" clearer than binary
5. **Better balance**: Each topic has 3 classes with 10-52% distribution

### Soft Ordinal Loss:

Uses **Mean Squared Error on ordinal encodings**:
- Low = 0, Medium = 1, High = 2
- Predicting Med when truth is High: loss = (1-2)^2 = 1
- Predicting Low when truth is High: loss = (0-2)^2 = 4 (larger penalty!)

This respects the ordered nature of relevance levels.

### Label Distribution (0.30/0.40 thresholds):

- **Colonialism**: Low=13%, Med=47%, High=40%
- **Historical Slavery**: Low=19%, Med=50%, High=31%
- **Modern Racism**: Low=10%, Med=38%, High=52%

### Expected Performance:

- Per-topic accuracy: 50-70% (vs 33% random baseline)
- Pattern exact match: 20-40%
- Mean Absolute Error: 0.3-0.5 (in 0-2 scale)

### Modified Cells:
- Cell 7.1: SBERT with 3 separate ordinal heads
- Cell 7.2: MultiOrdinalDataset (3-class labels per topic)
- Cell 7.3: OrdinalTrainer with Soft Ordinal Loss
- Cell 7.4: Ordinal metrics (accuracy, MAE, pattern match)
- Cell 7.5: MultiOrdinalDataCollator

---


---
# CHECKPOINT 0: Initial Setup
---

In [1]:
# ============================================================
# CELL 0.1: IMPORTS
# ============================================================
import os
import re
import json
import hashlib
import shutil
import warnings
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

from transformers import AutoTokenizer, TrainingArguments
import torch
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

# NLTK
import nltk
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

# ML libraries
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sentence_transformers import SentenceTransformer

def st_embed(texts: list, batch_size: int = 256) -> np.ndarray:
    return st_model.encode(
        texts, 
        batch_size=batch_size, 
        show_progress_bar=False, 
        normalize_embeddings=False
    )

# ============================================================
# Data Collator
# ============================================================

from dataclasses import dataclass
from typing import Any, Dict, List
from dataclasses import dataclass, field
from typing import List, Tuple
from typing import List, Tuple, Optional

@dataclass
class ContinuousDataCollator:
    tokenizer: Any
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # Extract labels
        labels = [f.pop("labels") for f in features]
        
        # Pad inputs
        batch = self.tokenizer.pad(
            features,
            padding=True,
            return_tensors="pt"
        )
        
        # Add labels as tensor
        batch["labels"] = torch.tensor(labels, dtype=torch.float32)
        
        return batch

print("V13: ContinuousDataCollator defined")


# Suppress warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

print("✓ All imports successful")

V13: ContinuousDataCollator defined
✓ All imports successful


In [5]:
# ============================================================
# CONFIGURATION WITH CORPUS FILTERING
# ============================================================

import os
import sys
from datetime import datetime

CONFIG = {
  # ==========================================
    # WORKFLOW NAMING (4-component system)
    # ==========================================
    "workflow": {
        # 1. CORPUS: What documents are you analyzing?
        #    Options: policy, historical, parliament, education, media, legal, mixed, dutch-caribbean
        "corpus": "slavery",

        # 2. VECTOR SOURCE: What dictionary creates topic vectors?
        #    Options: pretrained, slavery-dict, policy-dict, combined-dict, curated-v1, curated-v2
        "vector_source": "Short-slavdict",

        # 3. BERTJE TRAINING: How was BERTJE trained?
        #    Options: pretrained, ft-policy, ft-slavery, ft-historical, ft-mixed, multitask
        "bertje_training": "pretrained",

        # 4. TARGET: What are you identifying?
        #    Options: slavery, colonialism, racism, heritage, slavery-policy, 4topic, 5topic
        "target": "slavery",

        # VERSION: Auto-increments if None
        "version": None,

        # METADATA: Optional additional information
        "metadata": {
            "dictionary_file": "C:\\Users\\Home\\policy-analysis\\dutch_slavery_legacy_5topics_seed_v3_caribbean.csv",
            "dictionary_size": 563,
            "dictionary_description": "dictionary with colonial systems, ",
            "corpus_description": "Articles and books about slavery",
            "topics": ["Colonial Systems", "Heritage & Memory", "Historical Slavery", "Modern Racism"],
            "base_dictionaries": [
                "pretrained-Slavery_11.10.25_succes/Curated_dictionary.csv",
                "Finetuned_slaverypolicy-Slavery-Policy_11.13.25_v1/expanded_candidates.csv"
            ],
            "notes": "Policy analysis with enhanced colonial terminology from fine-tuned model"
        }
    },
    # =====================
    # PATHS
    # =====================
    "paths": {
        "corpus_dir": "PDF sources",
        "dictionary_excel": "C:\\Users\\Home\\policy-analysis\\A__dutch_slavery_legacy_5topics_seed_v4_weighted.csv",
        "workflow_base": "workflow_data",
        "pretrained_model_path": "C:\\Users\\Home\\policy-analysis\\workflow_data\\slavery_Slavdict_pretraining_slavery_v25\\Model_finetuning",
    },
    
    # =====================
    # CORPUS FILTERING (NEW SECTION)
    # =====================
    "corpus_filter": {
        # Enable/disable filtering
        "enabled": False,
        
        # Year filtering options:
        # Option 1: Specific years as a list
        "years": [2021, 2022, 2023],  # e.g., [2015, 2016, 2017]
        
        # Option 2: Year range (overrides "years" if set)
        # Set to None to disable, or use tuple like (2015, 2023)
        "year_range": None,  # e.g., (2015, 2023) for 2015 through 2023
        
        # Option 3: Only documents after/before certain year
        "year_min": None,  # e.g., 2000 for documents from 2000 onwards
        "year_max": None,  # e.g., 2023 for documents up to 2023
        
        # Document type filtering
        # List of document types to include (None = all types)
        "doc_types": ["ambtsberichten","beleidsnotas","besluiten","brieven","convenanten","jaarplannen","jaarverslagen","kamerstukken"],  # e.g., ["policy", "report", "legislation"]
        
        # Filename pattern matching (uses regex)
        # List of patterns - documents matching ANY pattern are included
        "filename_patterns": None,  # e.g., [".*gemeente.*", ".*ministry.*"]
        
        # Exclude patterns (uses regex)
        # Documents matching ANY exclude pattern are skipped
        "exclude_patterns": None,  # e.g., [".*draft.*", ".*concept.*"]
    },
    
    # =====================
    # MODEL SETTINGS
    # =====================
    "model": {
        "base_model_name": "NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers",
        "use_pretrained": False,
        "max_length": 512,
    },
    "sbert": {
        "use_mean_pooling": True,
        "dropout": 0.1,
        "use_multi_label": True
    },

    "multi_label": {
        "use_soft_targets": True,
        "loss_type": "bce",
        "clip_cosine_min": 0.0,
        "clip_cosine_max": 1.0,
        "use_focal_loss": False,
        "focal_alpha": 0.25,
        "focal_gamma": 2.0
    },

    
    # =====================
    # DICTIONARY SETTINGS
    # =====================
    "dictionary": {
        "use_excel": True,
        "topic_column": "topic",
        "keyword_column": "keyword",
        "sheet_name": 0,
        "default_topics": {
            "Historical slavery": ["slavernij", "tot-slaaf-gemaakte", "dwangarbeid", "zweep"],
            "Colonialism": ["kolonie", "koloniaal", "voc", "wic", "exploitatie"],
            "Modern racism& inequality": ["racisme", "discriminatie", "ongelijkheid"],
        },
    },
    
    # =====================
    # TEXT PROCESSING
    # =====================
    "chunking": {
        "sentences_per_chunk": 30,
        "min_sentences_to_keep": 3,
        "drop_likely_english": True,
        "remove_stopwords": True,
        "use_stemming": False,
        "use_token_aware": True,
        "max_tokens": 500,
    },

    "pdf_processing": {
        "enabled": True,                    # If False, assume preprocessed text files
        "min_sentences_per_page": 3,        # Remove pages with fewer sentences
        "max_english_ratio": 0.5,           # Remove pages with >50% English words
        "max_numeric_ratio": 0.3,           # Remove pages with >30% numeric characters
        "min_word_count": 50,               # Remove pages with fewer words
        "detect_layout_pages": True,        # Remove pages with layout/formatting (high whitespace ratio)
        "remove_headers_footers": True,     # Remove repeated text across pages
        "remove_reference_sections": True,  # Remove pages that are primarily citations
        "save_intermediate_text": True,     # Save preprocessed pages as .txt files
        "attempt_ocr_fallback": True,       # Try OCR if text extraction fails
        "layout_whitespace_threshold": 0.7, # Pages with >70% whitespace considered layout
        "layout_avg_line_length": 30,       # Pages with avg line length <30 chars considered layout
    },

    "pdf_processing": {
        "enabled": True,                    # If False, assume preprocessed text files
        "min_sentences_per_page": 3,        # Remove pages with fewer sentences
        "max_english_ratio": 0.5,           # Remove pages with >50% English words
        "max_numeric_ratio": 0.3,           # Remove pages with >30% numeric characters
        "min_word_count": 50,               # Remove pages with fewer words
        "detect_layout_pages": True,        # Remove pages with layout/formatting (high whitespace ratio)
        "remove_headers_footers": True,     # Remove repeated text across pages
        "remove_reference_sections": True,  # Remove pages that are primarily citations
        "save_intermediate_text": True,     # Save preprocessed pages as .txt files
        "attempt_ocr_fallback": True,       # Try OCR if text extraction fails
        "layout_whitespace_threshold": 0.7, # Pages with >70% whitespace considered layout
        "layout_avg_line_length": 30,       # Pages with avg line length <30 chars considered layout
    },
    
    "tokenize": {
        "lower": True,
        "keep_hyphen": True,
        "min_len": 2,
        "max_len": 30,
        "pattern": r"[0-9A-Za-zÀ-ÖØ-öø-ÿ\-]+",
    },
    
    # =====================
    # VOCABULARY SETTINGS
    # =====================
    "vocab": {
        "min_df": 2,           # Minimum document frequency (can be int or float 0-1 for percentage)
        "max_df": 0.8,         # Maximum document frequency (can be int or float 0-1 for percentage)
                               # Remove terms appearing in > 80% of chunks (likely stop words)
        "max_vocab": 100000,   # Maximum vocabulary size (truncate if needed)
    },
    
    # =====================
    # EXPANSION SETTINGS
    # =====================
    "expand": {
        "k_nearest": 50,
        "topN_per_topic": 300,
        "min_cosine": 0.55,
    },
    
    # =====================
    # SCORING SETTINGS
    # =====================
    "scoring": {
        "use_sif": True,
        "sif_a": 1e-3,
        "high_confidence_score": 0.40,
        "high_confidence_margin": 0.05,
        "low_confidence_score": 0.20,
        "low_confidence_margin": 0.02,
    },

    # Weight configuration for hybrid weighting system
    "weights": {
        "default_core_weight": 1.0,        # Default for seed terms without explicit weight
        "default_discovered_weight": 0.8,  # Default for expanded terms not in seed
        "weighting_scheme": "multiplicative",  # Options: "multiplicative", "additive", "seed_dominant", "geometric"
        "additive_seed_ratio": 0.7,        # If using "additive"/"seed_dominant": ratio for seed weight
    },

    # Sampling limits (balance with labeled data)
    "sampling": {
        "unlabeled_multiplier": 50,  # Max unlabeled = labeled_size * 30
        "pseudo_multiplier": 50       # Max pseudo = labeled_size * 20
    },

    
    # =====================
    # TRAINING SETTINGS
    # =====================
    "training": {
        "num_epochs": 5,
        "batch_size_train": 16,
        "batch_size_eval": 32,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "dataset_option": "option4",
        "apply_confidence_sampling": True,  # Apply stratified sampling by confidence level
        # Sampling reduces score compression by oversampling high-confidence (single-topic)
        # and undersampling none-confidence (ambiguous/noise) chunks
    },
}

# ============================================================
# GENERATE WORKFLOW BASE NAME (without version)
# ============================================================

import sys
sys.path.insert(0, os.getcwd())

# Generate base workflow name (components only, no version yet)
corpus = CONFIG['workflow']['corpus']
vector = CONFIG['workflow']['vector_source']
bertje = CONFIG['workflow']['bertje_training']
target = CONFIG['workflow']['target']

# Base name without version
workflow_base_name = f"{corpus}_{vector}_{bertje}_{target}"
CONFIG['workflow']['workflow_base_name'] = workflow_base_name

# Version will be set later by WorkflowFileSystem when creating/loading folder
# For now, just store placeholders
CONFIG['workflow']['workflow_name'] = None  # Will be set in Cell 55
CONFIG['workflow']['workflow_dir'] = None   # Will be set in Cell 55

print("✓ Configuration loaded")
print(f"\nWorkflow base name: {workflow_base_name}")
print(f"  Corpus: {CONFIG['workflow']['corpus']}")
print(f"  Vector: {CONFIG['workflow']['vector_source']}")
print(f"  BERTJE: {CONFIG['workflow']['bertje_training']}")
print(f"  Target: {CONFIG['workflow']['target']}")
print(f"\n  ⚠️  Version will be assigned when you create/load workflow in Cell 55")

# Display active filters
if CONFIG['corpus_filter']['enabled']:
    print("\n📂 Corpus Filtering ENABLED:")
    
    # Year filters
    if CONFIG['corpus_filter']['year_range']:
        print(f"  - Year range: {CONFIG['corpus_filter']['year_range'][0]} to {CONFIG['corpus_filter']['year_range'][1]}")
    elif CONFIG['corpus_filter']['years']:
        print(f"  - Specific years: {CONFIG['corpus_filter']['years']}")
    elif CONFIG['corpus_filter']['year_min'] or CONFIG['corpus_filter']['year_max']:
        if CONFIG['corpus_filter']['year_min']:
            print(f"  - Year minimum: {CONFIG['corpus_filter']['year_min']}")
        if CONFIG['corpus_filter']['year_max']:
            print(f"  - Year maximum: {CONFIG['corpus_filter']['year_max']}")
    
    # Document type filters
    if CONFIG['corpus_filter']['doc_types']:
        print(f"  - Document types: {CONFIG['corpus_filter']['doc_types']}")
    
    # Pattern filters
    if CONFIG['corpus_filter']['filename_patterns']:
        print(f"  - Include patterns: {CONFIG['corpus_filter']['filename_patterns']}")
    if CONFIG['corpus_filter']['exclude_patterns']:
        print(f"  - Exclude patterns: {CONFIG['corpus_filter']['exclude_patterns']}")
else:
    print("\n📂 Corpus Filtering DISABLED - loading all documents")

✓ Configuration loaded

Workflow base name: slavery_Short-slavdict_pretrained_slavery
  Corpus: slavery
  Vector: Short-slavdict
  BERTJE: pretrained
  Target: slavery

  ⚠️  Version will be assigned when you create/load workflow in Cell 55

📂 Corpus Filtering DISABLED - loading all documents


In [6]:
# ============================================================
# CELL 0.3: FILE SYSTEM UTILITIES
# ============================================================

class WorkflowFileSystem:
    """Manages structured folder system for workflow data."""
    
    def __init__(self, config):
        self.config = config
        self.root = None
        self.folders = {}
    
    def create_workflow_folder(self):
        """Create main workflow folder with subfolders. Auto-increments version."""
        # Get base name from CONFIG
        base_name = self.config["workflow"]["workflow_base_name"]
        base_dir = Path(self.config["paths"]["workflow_base"])

        # Auto-increment version
        version = self._get_next_version(base_name, base_dir)

        # Create folder name with version
        folder_name = f"{base_name}_v{version}"

        # Update CONFIG with the actual version and names
        self.config["workflow"]["version"] = version
        self.config["workflow"]["workflow_name"] = folder_name
        self.config["workflow"]["workflow_dir"] = str(base_dir / folder_name)

        self.root = base_dir / folder_name
        self.root.mkdir(parents=True, exist_ok=True)
        
        subfolder_names = [
            "config",
            "Dictionary",
            "Dictionary/Dictionary_suggestions",
            "Model_finetuning",
            "Cosine_labeling",
            "Bertje_labeling",
            "Visuals",
            "Preprocessed_text",
            "Other_data",
        ]
        
        for subfolder in subfolder_names:
            path = self.root / subfolder
            path.mkdir(parents=True, exist_ok=True)
            key = subfolder.split("/")[-1]
            self.folders[key] = path
        
        self.folders["Dictionary"] = self.root / "Dictionary"
        self.folders["Preprocessed_text"] = self.root / "Preprocessed_text"
        
        print(f"\n{'='*60}")
        print("WORKFLOW FOLDER CREATED")
        print(f"{'='*60}")
        print(f"Location: {self.root}")
        print(f"\nSubfolders:")
        for name in subfolder_names:
            print(f"  ✓ {name}/")
        
        return self.root

    def _get_next_version(self, base_name, base_dir):
        """Auto-increment version number based on existing folders."""
        if not base_dir.exists():
            return 1

        prefix = f"{base_name}_v"
        existing = [d.name for d in base_dir.iterdir()
                   if d.is_dir() and d.name.startswith(prefix)]

        if not existing:
            return 1

        versions = []
        for folder in existing:
            try:
                version_str = folder.split("_v")[-1]
                versions.append(int(version_str))
            except ValueError:
                continue

        if versions:
            return max(versions) + 1
        return 1

    def load_existing_workflow(self, folder_path):
        """Load existing workflow folder and update CONFIG."""
        self.root = Path(folder_path)
        if not self.root.exists():
            raise ValueError(f"Workflow folder not found: {folder_path}")

        # Extract version from folder name and update CONFIG
        folder_name = self.root.name
        if "_v" in folder_name:
            try:
                version = int(folder_name.split("_v")[-1])
                self.config["workflow"]["version"] = version
                self.config["workflow"]["workflow_name"] = folder_name
                self.config["workflow"]["workflow_dir"] = str(self.root)
            except ValueError:
                pass  # Keep existing CONFIG values if parsing fails
        
        subfolder_names = [
            "config", "Dictionary", "Dictionary_suggestions",
            "Model_finetuning", "Cosine_labeling", "Bertje_labeling",
            "Visuals", "Preprocessed_text", "Other_data"
        ]
        
        for name in subfolder_names:
            if name == "Dictionary_suggestions":
                path = self.root / "Dictionary" / name
            else:
                path = self.root / name
            if path.exists():
                self.folders[name] = path
        
        print(f"✓ Loaded existing workflow: {self.root.name}")
        return self.root
    
    def save_config(self, checkpoint_name=None):
        """Save CONFIG to config folder."""
        if self.root is None:
            raise ValueError("Workflow folder not initialized")
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"config_{checkpoint_name}_{timestamp}.json" if checkpoint_name else f"config_{timestamp}.json"
        config_path = self.folders["config"] / filename
        
        config_data = {
            "metadata": {
                "timestamp": timestamp,
                "checkpoint": checkpoint_name,
                "workflow_folder": str(self.root),
            },
            "config": self.config
        }
        
        with open(config_path, 'w', encoding='utf-8') as f:
            json.dump(config_data, f, indent=2, ensure_ascii=False)
        
        print(f"✓ Config saved: {config_path.name}")
        return config_path
    
    def save_data(self, data, filename, folder_key, file_format="csv"):
        """Save data to specific folder."""
        if self.root is None:
            raise ValueError("Workflow folder not initialized")
        
        folder = self.folders.get(folder_key)
        if folder is None:
            raise ValueError(f"Unknown folder key: {folder_key}")
        
        full_filename = f"{filename}.{file_format}"
        filepath = folder / full_filename
        
        if file_format == "csv":
            if not isinstance(data, pd.DataFrame):
                raise ValueError("CSV format requires DataFrame")
            data.to_csv(filepath, index=False, encoding='utf-8')
        elif file_format == "json":
            with open(filepath, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
        elif file_format == "npy":
            np.save(filepath, data, allow_pickle=True)
        else:
            raise ValueError(f"Unsupported format: {file_format}")
        
        print(f"✓ Saved: {folder_key}/{full_filename}")
        return filepath
    
    def copy_file_to_folder(self, source_path, folder_key, new_name=None):
        """Copy external file to workflow folder."""
        if self.root is None:
            raise ValueError("Workflow folder not initialized")
        
        folder = self.folders.get(folder_key)
        if folder is None:
            raise ValueError(f"Unknown folder key: {folder_key}")
        
        source = Path(source_path)
        if not source.exists():
            raise FileNotFoundError(f"Source file not found: {source_path}")
        
        dest_name = new_name if new_name else source.name
        dest_path = folder / dest_name
        
        shutil.copy2(source, dest_path)
        print(f"✓ Copied: {source.name} → {folder_key}/{dest_name}")
        return dest_path

    def get_source_workflow(self, workflow_path):
        """
        Load a different workflow as a source for reading data.

        Args:
            workflow_path: Path to workflow folder (e.g., "workflow_data/pretrained-Slavery_11.10.25_v2")

        Returns:
            WorkflowFileSystem instance pointing to the source workflow
        """
        if workflow_path is None:
            return self

        # Create a new WorkflowFileSystem instance
        source_fs = WorkflowFileSystem(self.config)

        # Load the existing workflow
        source_fs.load_existing_workflow(workflow_path)

        return source_fs

print("✓ WorkflowFileSystem class defined")


✓ WorkflowFileSystem class defined


In [7]:
# ============================================================
# CELL 0.4: CREATE OR LOAD WORKFLOW
# ============================================================

# Choose one:
CREATE_NEW = False   # Set to False to load existingr"C:\Users\Home\policy-analysis\workflow_data\slavery_Slavdict_pretraining_slavery_v13"
EXISTING_FOLDER = r"C:\Users\Home\policy-analysis\workflow_data\slavery_Short-slavdict_pretrained_slavery_v4"  # Set path if loading existing

fs = WorkflowFileSystem(CONFIG)

if CREATE_NEW:
    workflow_root = fs.create_workflow_folder()
    fs.save_config("initial_setup")
else:
    if EXISTING_FOLDER is None:
        raise ValueError("EXISTING_FOLDER must be set when CREATE_NEW=False")
    workflow_root = fs.load_existing_workflow(EXISTING_FOLDER)

print(f"\n✓ Workflow initialized: {workflow_root}")

✓ Loaded existing workflow: slavery_Short-slavdict_pretrained_slavery_v4

✓ Workflow initialized: C:\Users\Home\policy-analysis\workflow_data\slavery_Short-slavdict_pretrained_slavery_v4


In [8]:
# ============================================================
# CELL 0.5: LOAD DICTIONARY
# ============================================================

def load_dictionary_from_excel(excel_path, config):
    """Load topics, keywords, and optional weights from Excel or CSV."""
    if not Path(excel_path).exists():
        print(f"⚠ Dictionary file not found: {excel_path}")
        return config["dictionary"]["default_topics"]
    
    try:
        # Support both Excel and CSV
        if str(excel_path).endswith('.csv'):
            df = pd.read_csv(excel_path)
        else:
            df = pd.read_excel(excel_path, sheet_name=config["dictionary"]["sheet_name"])
        
        topic_col = config["dictionary"]["topic_column"]
        keyword_col = config["dictionary"]["keyword_column"]
        
        # Check for weight column
        has_weights = "weight" in df.columns
        default_core_weight = config["weights"]["default_core_weight"]
        
        if has_weights:
            print(f"✓ Found weight column in seed dictionary")
            # Return dict with weights
            topics_dict = {}
            for topic in df[topic_col].unique():
                topic_df = df[df[topic_col] == topic]
                # Store as list of (keyword, weight) tuples
                # Use explicit weight if present, otherwise use default_core_weight
                topics_dict[topic] = [
                    (row[keyword_col], row.get("weight", default_core_weight))
                    for _, row in topic_df.iterrows()
                ]
        else:
            print(f"⚠ No weight column found - using default_core_weight ({default_core_weight})")
            # Return dict with default weights
            topics_dict = {}
            for topic in df[topic_col].unique():
                keywords = df[df[topic_col] == topic][keyword_col].tolist()
                topics_dict[topic] = [(kw, default_core_weight) for kw in keywords]
        
        print(f"✓ Loaded {len(topics_dict)} topics with {sum(len(v) for v in topics_dict.values())} keywords")
        return topics_dict
    
    except Exception as e:
        print(f"❌ Failed to load dictionary: {e}")
        return config["dictionary"]["default_topics"]
if CONFIG["dictionary"]["use_excel"]:
    topics = load_dictionary_from_excel(CONFIG["paths"]["dictionary_excel"], CONFIG)
    CONFIG["topics"] = topics

    # Extract weights for later use and create keywords-only version
    topic_seed_weights = {}
    topics_keywords_only = {}
    
    for topic, terms in topics.items():
        if isinstance(terms[0], tuple):
            # Has weights - extract them
            keywords = [kw for kw, w in terms]
            weights = {kw: w for kw, w in terms}
            topic_seed_weights[topic] = weights
            topics_keywords_only[topic] = keywords
        else:
            # No weights (shouldn't happen with new function, but fallback)
            topics_keywords_only[topic] = terms
            topic_seed_weights[topic] = {kw: 1.0 for kw in terms}
    
    # Use keywords-only version for expansion
    topics = topics_keywords_only
    CONFIG["topics"] = topics
    CONFIG["topic_seed_weights"] = topic_seed_weights  # Store for later
    
    print(f"✓ Extracted seed weights for {len(topic_seed_weights)} topics")
    if Path(CONFIG["paths"]["dictionary_excel"]).exists():
        fs.copy_file_to_folder(
            CONFIG["paths"]["dictionary_excel"],
            "Dictionary",
            "input_dictionary.xlsx"
        )
else:
    CONFIG["topics"] = CONFIG["dictionary"]["default_topics"]

print(f"\n{'='*60}")
print("TOPICS LOADED")
print(f"{'='*60}")
for topic, keywords in CONFIG["topics"].items():
    print(f"  {topic}: {len(keywords)} keywords")

fs.save_config("with_dictionary")

✓ Found weight column in seed dictionary
✓ Loaded 4 topics with 134 keywords
✓ Extracted seed weights for 4 topics
✓ Copied: A__dutch_slavery_legacy_5topics_seed_v4_weighted.csv → Dictionary/input_dictionary.xlsx

TOPICS LOADED
  Colonial Systems: 30 keywords
  Heritage & Memory: 31 keywords
  Historical Slavery: 39 keywords
  Modern Racism & Discrimination: 34 keywords
✓ Config saved: config_with_dictionary_20260102_122832.json


WindowsPath('C:/Users/Home/policy-analysis/workflow_data/slavery_Short-slavdict_pretrained_slavery_v4/config/config_with_dictionary_20260102_122832.json')

✅ **CHECKPOINT 0 COMPLETE** - Folder structure created, dictionary loaded

---
# CHECKPOINT 1: Text Processing
---

Chunks corpus into sentence-based segments with cleaning.

## Consolidated Text Processing Pipeline (Checkpoint 1)

This section implements a streamlined 4-stage workflow that processes documents from PDFs or text files into a chunked corpus with comprehensive filtering and statistics reporting.

**Stages:**
1. Configuration & Source Loading
2. Document Loading & Text Filtering
3. Chunking & Dual Text Processing
4. Validation & Statistics Reporting

Each stage reports detailed filtering statistics showing what was removed and why.

In [8]:
# ========== CELL 12: Imports and Dependencies ==========

import os
import re
import json
import hashlib
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Tuple, Optional, Dict, Any
from dataclasses import dataclass, field
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import fitz  # PyMuPDF
except ImportError:
    print("⚠️  PyMuPDF not installed. PDF processing will not be available.")
    fitz = None

print("✓ All imports loaded successfully")

✓ All imports loaded successfully


In [9]:
# ============================================================
# CELL 1.1: CHECKPOINT 1 SOURCE OVERRIDES
# ============================================================
CP1_SOURCE = None  # e.g., r"workflow_data/AltSource_2024_v1"
CP1_SOURCE_CHUNKS = None  # e.g., r"workflow_data/AltSource_2024_v1/Other_data/chunked_corpus.csv"
CP1_FORCE_MODE = 'pdf'  # Options: 'pdf', 'text', 'chunked'

source_fs = fs.get_source_workflow(CP1_SOURCE) if CP1_SOURCE else fs

print("? CP1 overrides ready")
print(f"  Source override: {{CP1_SOURCE or 'None (use CONFIG)'}}")
print(f"  Pre-chunked CSV: {{CP1_SOURCE_CHUNKS or 'None'}}")
print(f"  Force mode: {{CP1_FORCE_MODE or 'Auto-detect'}}")


? CP1 overrides ready
  Source override: {CP1_SOURCE or 'None (use CONFIG)'}
  Pre-chunked CSV: {CP1_SOURCE_CHUNKS or 'None'}
  Force mode: {CP1_FORCE_MODE or 'Auto-detect'}


In [10]:
# ============================================================
# CELL 1.2: PDF/TEXT TO CLEAN TEXT FILES
# ============================================================
# ========== CELL 13: Configuration Dataclass ==========

@dataclass
class ProcessingConfig:
    """Unified configuration for all processing stages"""

    # Source configuration
    corpus_source: Path = None
    corpus_mode: str = 'auto'  # 'pdf', 'text', 'chunked', 'auto'

    # Document filtering
    corpus_filter_enabled: bool = False
    filter_years: List[int] = field(default_factory=list)
    filter_year_range: Tuple[int, int] = None
    filter_doc_types: List[str] = field(default_factory=list)
    filter_filename_patterns: List[str] = field(default_factory=list)
    filter_exclude_patterns: List[str] = field(default_factory=list)

    # PDF page filtering
    pdf_min_word_count: int = 50
    pdf_min_sentences: int = 3
    pdf_max_english_ratio: float = 0.5
    pdf_max_numeric_ratio: float = 0.3
    pdf_detect_layout_pages: bool = True
    pdf_layout_whitespace_threshold: float = 0.7
    pdf_layout_avg_line_length: int = 30
    pdf_remove_reference_sections: bool = True
    pdf_remove_headers_footers: bool = True
    pdf_save_preprocessed: bool = True

    # Content filtering (footnotes/bibliography)
    content_remove_bibliography: bool = True
    content_remove_footnotes: bool = True
    content_aggressive_footnotes: bool = False
    content_skip_index_pages: bool = True

    # Chunking
    chunk_sentences_per_chunk: int = 30
    chunk_min_sentences: int = 3
    chunk_min_tokens: int = 80
    chunk_use_token_aware: bool = False
    chunk_max_tokens: int = 500

    # Text processing for scoring
    scoring_drop_english: bool = True
    scoring_remove_stopwords: bool = True
    scoring_use_stemming: bool = False

    # Output
    save_intermediate_text: bool = True

    @classmethod
    def from_config_dict(cls, config: dict, overrides: dict = None):
        """Create from CONFIG dict with optional overrides"""

        # Extract from nested CONFIG structure
        paths = config.get('paths', {})
        chunking = config.get('chunking', {})
        pdf_proc = config.get('pdf_processing', {})
        corpus_filter = config.get('corpus_filter', {})

        # Build config
        proc_config = cls(
            corpus_source=Path(paths.get('corpus_dir', 'PDF sources')),

            # Corpus filtering
            corpus_filter_enabled=corpus_filter.get('enabled', False),
            filter_years=corpus_filter.get('years', []),
            filter_year_range=corpus_filter.get('year_range'),
            filter_doc_types=corpus_filter.get('doc_types', []),
            filter_filename_patterns=corpus_filter.get('filename_patterns', []),
            filter_exclude_patterns=corpus_filter.get('exclude_patterns', []),

            # PDF filtering
            pdf_min_word_count=pdf_proc.get('min_word_count', 50),
            pdf_min_sentences=pdf_proc.get('min_sentences_per_page', 3),
            pdf_max_english_ratio=pdf_proc.get('max_english_ratio', 0.5),
            pdf_max_numeric_ratio=pdf_proc.get('max_numeric_ratio', 0.3),
            pdf_detect_layout_pages=pdf_proc.get('detect_layout_pages', True),
            pdf_layout_whitespace_threshold=pdf_proc.get('layout_whitespace_threshold', 0.7),
            pdf_layout_avg_line_length=pdf_proc.get('layout_avg_line_length', 30),
            pdf_remove_reference_sections=pdf_proc.get('remove_reference_sections', True),
            pdf_remove_headers_footers=pdf_proc.get('remove_headers_footers', True),
            pdf_save_preprocessed=pdf_proc.get('save_intermediate_text', True),

            # Chunking
            chunk_sentences_per_chunk=chunking.get('sentences_per_chunk', 10),
            chunk_min_sentences=chunking.get('min_sentences_to_keep', 2),
            chunk_use_token_aware=chunking.get('use_token_aware', False),
            chunk_max_tokens=chunking.get('max_tokens', 500),

            # Text processing
            scoring_drop_english=chunking.get('drop_likely_english', True),
            scoring_remove_stopwords=chunking.get('remove_stopwords', True),
            scoring_use_stemming=chunking.get('use_stemming', False),
        )

        # Apply overrides
        if overrides:
            for key, value in overrides.items():
                if hasattr(proc_config, key):
                    setattr(proc_config, key, value)

        return proc_config


def detect_corpus_mode(corpus_source: Path) -> str:
    """Auto-detect processing mode based on files in corpus source"""
    if not corpus_source.exists():
        raise ValueError(f"Corpus source does not exist: {corpus_source}")

    pdf_files = list(corpus_source.glob("**/*.pdf"))
    txt_files = list(corpus_source.glob("**/*.txt"))

    if pdf_files and not txt_files:
        return 'pdf'
    elif txt_files and not pdf_files:
        return 'text'
    elif pdf_files and txt_files:
        print(f"⚠️  Found both PDFs ({len(pdf_files)}) and TXT files ({len(txt_files)})")
        print(f"   Defaulting to PDF mode. Override with corpus_mode='text'")
        return 'pdf'
    else:
        raise ValueError(f"No PDF or TXT files found in {corpus_source}")


def load_configuration(config_dict: dict, cp1_source: str = None,
                       cp1_source_chunks: str = None, cp1_force_mode: str = None) -> ProcessingConfig:
    """
    STAGE 1: Load and validate configuration

    Args:
        config_dict: Main CONFIG dictionary
        cp1_source: Override corpus source path
        cp1_source_chunks: Path to pre-chunked CSV (skips all processing)
        cp1_force_mode: Force specific processing mode

    Returns:
        ProcessingConfig object
    """
    print("\n" + "="*80)
    print("STAGE 1: CONFIGURATION & SOURCE LOADING")
    print("="*80)

    overrides = {}

    # Handle pre-chunked CSV (highest priority)
    if cp1_source_chunks:
        print(f"✓ Loading pre-chunked data from: {cp1_source_chunks}")
        overrides['corpus_source'] = Path(cp1_source_chunks)
        overrides['corpus_mode'] = 'chunked'

    # Handle source override
    elif cp1_source:
        print(f"✓ Using overridden corpus source: {cp1_source}")
        overrides['corpus_source'] = Path(cp1_source)
        if cp1_force_mode:
            overrides['corpus_mode'] = cp1_force_mode
        else:
            overrides['corpus_mode'] = detect_corpus_mode(Path(cp1_source))

    # Use default from CONFIG
    else:
        default_source = Path(config_dict['paths']['corpus_dir'])
        print(f"✓ Using default corpus source: {default_source}")
        overrides['corpus_source'] = default_source
        if cp1_force_mode:
            overrides['corpus_mode'] = cp1_force_mode
        elif config_dict.get('pdf_processing', {}).get('enabled', True):
            overrides['corpus_mode'] = detect_corpus_mode(default_source)
        else:
            overrides['corpus_mode'] = 'text'

    # Create configuration
    proc_config = ProcessingConfig.from_config_dict(config_dict, overrides)

    print(f"\nConfiguration Summary:")
    print(f"  Source: {proc_config.corpus_source}")
    print(f"  Mode: {proc_config.corpus_mode}")
    print(f"  Corpus Filter: {'Enabled' if proc_config.corpus_filter_enabled else 'Disabled'}")
    print(f"  Chunking: {proc_config.chunk_sentences_per_chunk} sentences/chunk")
    print(f"  Text Processing: stopwords={proc_config.scoring_remove_stopwords}, stemming={proc_config.scoring_use_stemming}")

    return proc_config


print('✓ Cell 13: Configuration dataclass and loading functions loaded')

# ========== CELL 14: Content Filtering Patterns & Functions ==========

BIBLIOGRAPHY_HEADERS = [
    r'^\s*(?:bibliografie|bibliography|literatuur|literatuurlijst)',
    r'^\s*(?:bronnen|geraadpleegde\s+bronnen|bronvermelding)',
    r'^\s*(?:references|works\s+cited|works\s+consulted)',
    r'^\s*(?:noten|voetnoten|footnotes|endnotes)',
    r'^\s*(?:index|register|personenregister|namenregister)',
    r'^\s*(?:bijlagen|appendix|appendices)',
]

BIBLIOGRAPHY_PATTERNS = [
    (r'^[A-Z][a-z]+(?:-[A-Z][a-z]+)?,\s+[A-Z]', 'author_list'),
    (r'\(\d{4}[a-z]?\)', 'year_parens'),
    (r'(?:doi:|https?://|www\.)', 'url'),
]

INDEX_PATTERNS = [
    (r'^[A-Z][a-z]+,\s+[A-Z][a-z]+(?:\s+\d+(?:[,\s]+\d+)*)', 'index_entry'),
    (r'\d+(?:[,\s]+\d+){5,}', 'dense_page_nums'),
]


def detect_index_section(text: str, threshold: float = 0.4) -> bool:
    """Detect if text is an index page"""
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    if len(lines) < 5:
        return False

    if re.search(r'^\s*(?:index|register|personenregister|namenregister)',
                 text[:200], re.IGNORECASE | re.MULTILINE):
        return True

    matching_lines = 0
    for line in lines:
        for pattern, _ in INDEX_PATTERNS:
            if re.search(pattern, line, re.IGNORECASE):
                matching_lines += 1
                break

    return (matching_lines / len(lines)) >= threshold


def detect_bibliography_section(text: str, threshold: float = 0.3) -> bool:
    """Detect if text is a bibliography section"""
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    if len(lines) < 3:
        return False

    for pattern in BIBLIOGRAPHY_HEADERS:
        if re.search(pattern, text[:500], re.IGNORECASE | re.MULTILINE):
            return True

    matching_lines = 0
    for line in lines:
        for pattern, _ in BIBLIOGRAPHY_PATTERNS:
            if re.search(pattern, line, re.IGNORECASE):
                matching_lines += 1
                break

    return (matching_lines / len(lines)) >= threshold


def split_text_before_bibliography(text: str) -> Tuple[str, Optional[str]]:
    """Split text into main content and bibliography"""
    for pattern in BIBLIOGRAPHY_HEADERS:
        match = re.search(pattern, text, re.IGNORECASE | re.MULTILINE)
        if match:
            split_pos = match.start()
            main_text = text[:split_pos].strip()
            biblio_text = text[split_pos:].strip()

            if detect_bibliography_section(biblio_text):
                return main_text, biblio_text

    return text, None


def remove_footnote_markers(text: str, aggressive: bool = False) -> Tuple[str, int]:
    """Remove footnote markers and return (cleaned_text, num_removed)"""
    original_len = len(text)
    cleaned = text

    # Remove superscript-like numbers
    cleaned = re.sub(r'(?<=[a-z])\d{1,3}(?=[\s,;.\)])', '', cleaned)

    # Remove ibid, op. cit., etc.
    cleaned = re.sub(r'\b(?:ibid|ibidem|op\.\s*cit|loc\.\s*cit)\b\.?', '', cleaned, flags=re.IGNORECASE)

    # Remove cross-references
    cleaned = re.sub(r'\b(?:zie\s+ook|cf\.|see\s+also|vgl\.|vergelijk)\b', '', cleaned, flags=re.IGNORECASE)

    if aggressive:
        cleaned = re.sub(r'\([A-Z][a-z]+(?:\s+[a-z]+)*\s+\d{4}[a-z]?\)', '', cleaned)
        cleaned = re.sub(r'\bp{1,2}\.\s*\d+(?:[-–]\d+)?', '', cleaned)

    # Preserve line structure but normalize whitespace
    cleaned = cleaned.replace('\r\n', '\n').replace('\r', '\n')
    lines = []
    for raw_line in cleaned.split('\n'):
        normalized = re.sub(r'[ \t]+', ' ', raw_line).strip()
        if normalized:
            lines.append(normalized)
    cleaned = '\n'.join(lines)
    cleaned = re.sub(r'\n{3,}', '\n\n', cleaned)
    cleaned = re.sub(r'\(\s*\)', '', cleaned)

    num_removed = original_len - len(cleaned)
    return cleaned.strip(), num_removed


def filter_page_content(page_text: str, config: ProcessingConfig) -> Tuple[str, Dict[str, int]]:
    """
    Apply content-level filtering to page text.
    Returns (filtered_text, stats_dict)
    """
    stats = {
        'original_chars': len(page_text),
        'bibliography_chars_removed': 0,
        'footnote_chars_removed': 0,
        'is_index': False,
        'is_bibliography': False,
    }

    # Skip index pages entirely
    if config.content_skip_index_pages and detect_index_section(page_text):
        stats['is_index'] = True
        return "", stats

    # Remove bibliography sections
    if config.content_remove_bibliography:
        main_text, biblio = split_text_before_bibliography(page_text)
        if biblio:
            stats['bibliography_chars_removed'] = len(biblio)
            page_text = main_text
        elif detect_bibliography_section(page_text):
            stats['is_bibliography'] = True
            return "", stats

    # Remove footnote markers
    if config.content_remove_footnotes:
        page_text, footnote_chars = remove_footnote_markers(
            page_text, aggressive=config.content_aggressive_footnotes
        )
        stats['footnote_chars_removed'] = footnote_chars

    return page_text.strip(), stats


print('✓ Cell 14: Content filtering patterns and functions loaded')

# ========== CELL 15: PDF Processing Functions ==========

# DocumentStats and ProcessingStats dataclasses
@dataclass
class DocumentStats:
    """Statistics for a single document"""
    file_path: str
    doc_type: str = ''
    year: str = ''

    # PDF-specific stats
    total_pages: int = 0
    kept_pages: int = 0
    filtered_pages: Dict[str, int] = field(default_factory=lambda: defaultdict(int))

    # Content stats
    total_chars: int = 0
    filtered_bibliography_chars: int = 0
    filtered_footnote_markers: int = 0
    index_pages_skipped: int = 0

    # Final output
    final_text_length: int = 0
    error: str = None


@dataclass
class ProcessingStats:
    """Aggregate statistics for entire processing run"""
    total_documents: int = 0
    processed_documents: int = 0
    failed_documents: int = 0
    skipped_documents: int = 0

    # Document-level filtering
    filter_reasons: Dict[str, int] = field(default_factory=lambda: defaultdict(int))

    # PDF page-level stats
    total_pages_extracted: int = 0
    total_pages_kept: int = 0
    page_filter_reasons: Dict[str, int] = field(default_factory=lambda: defaultdict(int))

    # Content-level stats
    total_chars_before: int = 0
    total_chars_after: int = 0
    bibliography_chars_removed: int = 0
    footnote_markers_removed: int = 0
    index_pages_removed: int = 0

    # Chunk-level stats
    total_chunks: int = 0
    chunks_filtered: Dict[str, int] = field(default_factory=lambda: defaultdict(int))

    # Errors
    errors: List[Tuple[str, str]] = field(default_factory=list)

    # Per-document stats
    document_stats: List[DocumentStats] = field(default_factory=list)


def extract_text_from_pdf(pdf_path: Path) -> List[Tuple[int, str]]:
    """Extract text from PDF, returns list of (page_num, text) tuples"""
    if fitz is None:
        raise ImportError("PyMuPDF not installed")

    doc = fitz.open(pdf_path)
    pages = []

    for page_num in range(len(doc)):
        try:
            page = doc[page_num]
            text = page.get_text()
            pages.append((page_num + 1, text))
        except Exception as e:
            pages.append((page_num + 1, ""))

    doc.close()
    return pages


def filter_pdf_page(page_text: str, config: ProcessingConfig) -> Tuple[bool, str]:
    """
    Filter individual PDF page based on quality metrics.
    Returns (keep: bool, reason: str)
    """
    if not page_text.strip():
        return False, "empty"

    # Word count
    words = page_text.split()
    if len(words) < config.pdf_min_word_count:
        return False, "too_few_words"

    # Sentence count
    sentences = re.split(r'(?<=[.!?])\s+', page_text)
    sentences = [s for s in sentences if s.strip()]
    if len(sentences) < config.pdf_min_sentences:
        return False, "too_few_sentences"

    # English ratio
    english_words = {'the', 'be', 'to', 'of', 'and', 'a', 'in', 'that', 'have', 'is', 'it', 'for', 'not', 'on', 'with'}
    english_count = sum(1 for w in words if w.lower() in english_words)
    english_ratio = english_count / len(words) if words else 0
    if english_ratio > config.pdf_max_english_ratio:
        return False, "too_much_english"

    # Numeric ratio
    numeric_chars = sum(1 for c in page_text if c.isdigit())
    numeric_ratio = numeric_chars / len(page_text) if page_text else 0
    if numeric_ratio > config.pdf_max_numeric_ratio:
        return False, "too_many_numbers"

    # Layout detection
    if config.pdf_detect_layout_pages:
        lines = page_text.split('\n')
        non_empty_lines = [l for l in lines if l.strip()]
        if non_empty_lines:
            whitespace_ratio = (len(lines) - len(non_empty_lines)) / len(lines)
            avg_line_length = sum(len(l) for l in non_empty_lines) / len(non_empty_lines)

            if whitespace_ratio > config.pdf_layout_whitespace_threshold:
                return False, "layout_page"
            if avg_line_length < config.pdf_layout_avg_line_length:
                return False, "layout_page"

    # Reference section detection
    if config.pdf_remove_reference_sections:
        citation_patterns = [r'\(\d{4}\)', r'https?://', r'doi:']
        citation_count = sum(len(re.findall(p, page_text)) for p in citation_patterns)
        if citation_count > len(sentences) * 0.2:  # >20% citation density
            return False, "reference_section"


    # Copyright and metadata page detection
    # Detect pages with publishing information, ISBNs, copyright notices, etc.
    copyright_patterns = [
        r'copyright\s*©',
        r'isbn\s*\d',
        r'(?:mogelijk\s+gemaakt|gemaakt\s+door)\s+(?:een\s+)?subsidie',
        r'ministerie\s+van\s+\w+\s+zaken',
        r'beeldredactie:',
        r'boekverzorging',
        r'uitgever(?:ij)?',
        r'druk(?:kerij)?.*\d{4}',
        r'alle\s+rechten\s+voorbehouden',
        r'niets\s+uit\s+deze\s+uitgave',
        r'www\.\w+\.(nl|com)',
        r'colofon',
    ]

    page_lower = page_text.lower()
    copyright_matches = sum(1 for pattern in copyright_patterns if re.search(pattern, page_lower))

    # If 3+ copyright patterns match, it's likely a copyright/metadata page
    if copyright_matches >= 3:
        return False, "copyright_metadata"


    # Front matter detection (ToC, author lists, chapter titles)
    # These pages pass other filters but are not substantive content
    page_lower = page_text.lower()
    page_start = page_text[:300].lower()

    # Pattern 1: Table of Contents
    toc_patterns = [
        r'\binhoudsopgave\b',
        r'\btable\s+of\s+contents\b',
        r'\bcontents\b.*\n.*\d+',  # "Contents" followed by page numbers
        r'\binhoud\b',
    ]
    if any(re.search(p, page_start) for p in toc_patterns):
        return False, "front_matter_toc"

    # Pattern 2: High density of lines with page numbers (>30% of lines)
    # Typical of ToC pages with "Chapter 1 ... 15" format
    lines_list = page_text.split('\n')
    non_empty_lines = [l for l in lines_list if l.strip()]
    if len(non_empty_lines) > 5:
        lines_with_numbers = sum(1 for l in non_empty_lines if re.search(r'\d+', l))
        number_ratio = lines_with_numbers / len(non_empty_lines)
        if number_ratio > 0.35:  # >35% of lines have numbers
            # Check if it's structured like ToC (short lines with numbers)
            avg_line_length = sum(len(l.strip()) for l in non_empty_lines) / len(non_empty_lines)
            if avg_line_length < 60:  # Short lines
                return False, "front_matter_toc_numbers"

    # Pattern 3: Author/contributor lists
    # Multiple lines ending with names/em-dash patterns
    author_patterns = [
        r'[–—-]\s*[A-Z][a-z]+\s+[A-Z]',  # Any dash + name
        r'[A-Z][a-z]+\s+[A-Z][a-z]+(?:,\s+[A-Z][a-z]+\s+[A-Z][a-z]+){2,}',  # Multiple names
    ]
    
    # Check for inleiding/introduction pages with multiple author names
    if 'inleiding' in page_lower or 'introduction' in page_lower:
        name_pattern = r'[A-Z][a-z]+\s+[A-Z][a-z]+'
        names_found = len(re.findall(name_pattern, page_text))
        if names_found >= 3:
            return False, 'front_matter_authors'
    author_matches = sum(1 for p in author_patterns if re.search(p, page_text, re.MULTILINE))
    if author_matches >= 2:
        return False, "front_matter_authors"

    # Pattern 4: Chapter title pages (short pages with "Deel" / "Part" / "Hoofdstuk")
    if len(words) < 100:  # Short page
        chapter_patterns = [
            r'^\s*deel\s+\d+',
            r'^\s*part\s+\d+',
            r'^\s*hoofdstuk\s+\d+',
            r'^\s*chapter\s+\d+',
        ]
        if any(re.search(p, page_lower, re.MULTILINE) for p in chapter_patterns):
            return False, "front_matter_chapter_title"


    # Additional check: pages starting with "Dit boek" or similar publisher text
    page_start = page_text[:200].lower()
    if any(phrase in page_start for phrase in [
        'dit boek is mogelijk gemaakt',
        'dit boek is uitgegeven',
        'eerste druk',
    ]):
        return False, "copyright_metadata"


    return True, "keep"




def normalize_header_footer_line(line: str) -> str:
    line = line.replace('–', '-').replace('—', '-')
    line = re.sub(r'\d+', '#', line)
    line = re.sub(r'[\-\s]+', ' ', line)
    return line.strip().lower()
def detect_headers_footers(pages: List[Tuple[int, str]], threshold: float = 0.6) -> Dict:
    """Detect repeated text across pages"""
    header_candidates = defaultdict(int)
    header_keys = defaultdict(set)
    footer_candidates = defaultdict(int)
    footer_keys = defaultdict(set)

    for page_num, text in pages:
        lines = text.split('\n')

        # Check first 3 lines
        for line in lines[:3]:
            line = line.strip()
            if line and len(line) > 5:  # Ignore very short lines
                norm = normalize_header_footer_line(line)
                if len(norm) > 3:
                    header_candidates[norm] += 1
                    header_keys[norm].add(line)

        # Check last 3 lines
        for line in lines[-3:]:
            line = line.strip()
            if line and len(line) > 5:
                norm = normalize_header_footer_line(line)
                if len(norm) > 3:
                    footer_candidates[norm] += 1
                    footer_keys[norm].add(line)

    # Find repeated patterns (appear in >threshold of pages)
    min_occurrences = len(pages) * threshold
    headers = {key for key, count in header_candidates.items() if count >= min_occurrences}
    footers = {key for key, count in footer_candidates.items() if count >= min_occurrences}

    return {'headers': headers, 'footers': footers}


def remove_headers_footers(text: str, header_footer_data: Dict) -> str:
    """Remove detected headers and footers from text, including page numbers"""
    lines = text.split('\n')

    # Remove from first 3 lines (repeated headers)
    for i in range(min(3, len(lines))):
        line_stripped = lines[i].strip()
        if line_stripped:
            # Check against normalized version
            normalized = normalize_header_footer_line(line_stripped)
            if normalized in header_footer_data['headers']:
                lines[i] = ""

    # Remove from last 3 lines (repeated footers)
    for i in range(max(0, len(lines) - 3), len(lines)):
        line_stripped = lines[i].strip()
        if line_stripped:
            # Check against normalized version
            normalized = normalize_header_footer_line(line_stripped)
            if normalized in header_footer_data['footers']:
                lines[i] = ""

    # Remove isolated page numbers
    page_number_patterns = [
        r'^\s*\d+\s*$',           # Just a number: "42"
        r'^\s*-\s*\d+\s*-\s*$', # Dashed: "- 42 -"
        r'^\s*—\s*\d+\s*—\s*$', # Em-dash: "— 42 —"
        r'^\s*\d+\s*/\s*\d+\s*$', # Fraction style: "42 / 120"
    ]

    for i in range(len(lines)):
        line_stripped = lines[i].strip()
        # Only process short lines (page numbers are typically short)
        if len(line_stripped) <= 15:
            for pattern in page_number_patterns:
                if re.match(pattern, line_stripped):
                    lines[i] = ""
                    break

    return '\n'.join(lines)


def process_pdf_document(pdf_path: Path, config: ProcessingConfig,
                        workflow_fs, corpus_source: Path) -> Tuple[str, DocumentStats]:
    """
    Process a single PDF document through full filtering pipeline.
    Returns (cleaned_text, stats)
    """
    stats = DocumentStats(file_path=str(pdf_path))

    try:
        # Extract pages
        pages = extract_text_from_pdf(pdf_path)
        stats.total_pages = len(pages)

        # Filter pages
        filtered_pages = []
        for page_num, page_text in pages:
            keep, reason = filter_pdf_page(page_text, config)
            if keep:
                filtered_pages.append((page_num, page_text))
            else:
                stats.filtered_pages[reason] += 1

        stats.kept_pages = len(filtered_pages)

        if not filtered_pages:
            return "", stats

        # Detect and remove headers/footers
        if config.pdf_remove_headers_footers:
            header_footer_data = detect_headers_footers(filtered_pages)
            filtered_pages = [
                (pn, remove_headers_footers(text, header_footer_data))
                for pn, text in filtered_pages
            ]

        # Apply content filtering to each page
        content_filtered_pages = []
        for page_num, page_text in filtered_pages:
            filtered_text, content_stats = filter_page_content(page_text, config)
            if filtered_text:
                content_filtered_pages.append((page_num, filtered_text))
                stats.total_chars += content_stats['original_chars']
                stats.filtered_bibliography_chars += content_stats['bibliography_chars_removed']
                stats.filtered_footnote_markers += content_stats['footnote_chars_removed']
            elif content_stats['is_index']:
                stats.index_pages_skipped += 1

        # Combine pages into single document
        document_text = '\n\n'.join(text for _, text in content_filtered_pages)
        stats.final_text_length = len(document_text)

        # Save preprocessed if configured
        if config.pdf_save_preprocessed and config.save_intermediate_text and workflow_fs:
            save_preprocessed_text(pdf_path, content_filtered_pages, workflow_fs, corpus_source)

        return document_text, stats

    except Exception as e:
        stats.error = str(e)
        return "", stats


def save_preprocessed_text(pdf_path: Path, pages: List[Tuple[int, str]],
                           workflow_fs, corpus_source: Path):
    """Save preprocessed pages to Preprocessed_text folder"""
    try:
        # Create relative path structure
        rel_path = pdf_path.relative_to(corpus_source)
        output_dir = workflow_fs.folders['Preprocessed_text'] / rel_path.parent
        output_dir.mkdir(parents=True, exist_ok=True)

        # Save each page
        stem = pdf_path.stem
        for page_num, text in pages:
            output_file = output_dir / f"{stem}.page_{page_num:03d}.txt"
            output_file.write_text(text, encoding='utf-8')

    except Exception as e:
        print(f"⚠️  Failed to save preprocessed text: {e}")


print('✓ Cell 15: PDF processing functions loaded')

# ========== CELL 16: Stage 2 - Document Loading & Filtering ==========

def should_include_document(file_path: Path, corpus_source: Path, config: ProcessingConfig) -> Tuple[bool, str]:
    """Check if document passes corpus filters"""
    if not config.corpus_filter_enabled:
        return True, "filter_disabled"

    # Extract metadata from path
    try:
        rel_parts = file_path.relative_to(corpus_source).parts
        metadata = {
            'doc_type': rel_parts[0] if len(rel_parts) >= 1 else '',
            'year': rel_parts[1] if len(rel_parts) >= 2 else '',
            'filename': file_path.name
        }
    except ValueError:
        metadata = {'doc_type': '', 'year': '', 'filename': file_path.name}

    # Year filters
    if config.filter_years and metadata['year']:
        if metadata['year'] not in [str(y) for y in config.filter_years]:
            return False, "year_not_in_list"

    if config.filter_year_range and metadata['year']:
        try:
            year = int(metadata['year'])
            if year < config.filter_year_range[0] or year > config.filter_year_range[1]:
                return False, "year_out_of_range"
        except ValueError:
            pass

    # Doc type filter
    if config.filter_doc_types and metadata['doc_type']:
        if metadata['doc_type'] not in config.filter_doc_types:
            return False, "doc_type_excluded"

    # Filename pattern filters
    if config.filter_filename_patterns:
        if not any(re.search(p, metadata['filename'], re.IGNORECASE)
                  for p in config.filter_filename_patterns):
            return False, "filename_pattern_not_matched"

    # Exclude patterns
    if config.filter_exclude_patterns:
        if any(re.search(p, metadata['filename'], re.IGNORECASE)
              for p in config.filter_exclude_patterns):
            return False, "filename_excluded"

    return True, "passed_filters"


def load_and_filter_documents(config: ProcessingConfig, workflow_fs=None) -> Tuple[List[Dict], ProcessingStats]:
    """
    STAGE 2: Load documents and apply filtering pipeline

    Returns:
        (documents, stats) where documents is list of dicts with keys:
            - file_path: Path to source file
            - text: Filtered document text
            - metadata: Dict with doc_type, year, etc.
    """
    print("\n" + "="*80)
    print("STAGE 2: DOCUMENT LOADING & TEXT FILTERING")
    print("="*80)

    stats = ProcessingStats()
    documents = []

    # Handle pre-chunked mode
    if config.corpus_mode == 'chunked':
        print(f"✓ Loading pre-chunked data from: {config.corpus_source}")
        # Return empty - will be handled by Stage 3
        return None, stats

    # Discover files
    if config.corpus_mode == 'pdf':
        file_pattern = "**/*.pdf"
    else:  # text
        file_pattern = "**/*.txt"

    all_files = list(config.corpus_source.glob(file_pattern))
    print(f"✓ Found {len(all_files)} {config.corpus_mode.upper()} files")

    stats.total_documents = len(all_files)

    # Apply document-level filters
    filtered_files = []
    for file_path in all_files:
        include, reason = should_include_document(file_path, config.corpus_source, config)
        if include:
            filtered_files.append(file_path)
        else:
            stats.filter_reasons[reason] += 1
            stats.skipped_documents += 1

    if config.corpus_filter_enabled:
        print(f"✓ After corpus filtering: {len(filtered_files)} documents")
        if stats.filter_reasons:
            print(f"  Filtered out: {dict(stats.filter_reasons)}")

    # Process each document
    print(f"\nProcessing {len(filtered_files)} documents...")

    for i, file_path in enumerate(tqdm(filtered_files, desc="Processing")):
        try:
            # Extract metadata from path
            try:
                rel_parts = file_path.relative_to(config.corpus_source).parts
                metadata = {
                    'doc_type': rel_parts[0] if len(rel_parts) >= 1 else '',
                    'year': rel_parts[1] if len(rel_parts) >= 2 else '',
                    'document_folder': rel_parts[2] if len(rel_parts) >= 3 else '',
                    'filename': file_path.name
                }
            except ValueError:
                metadata = {'doc_type': '', 'year': '', 'document_folder': '', 'filename': file_path.name}

            # Process based on mode
            if config.corpus_mode == 'pdf':
                doc_text, doc_stats = process_pdf_document(file_path, config, workflow_fs, config.corpus_source)
                doc_stats.doc_type = metadata['doc_type']
                doc_stats.year = metadata['year']

                # Aggregate stats
                stats.total_pages_extracted += doc_stats.total_pages
                stats.total_pages_kept += doc_stats.kept_pages
                for reason, count in doc_stats.filtered_pages.items():
                    stats.page_filter_reasons[reason] += count
                stats.total_chars_before += doc_stats.total_chars
                stats.bibliography_chars_removed += doc_stats.filtered_bibliography_chars
                stats.footnote_markers_removed += doc_stats.filtered_footnote_markers
                stats.index_pages_removed += doc_stats.index_pages_skipped

                # Print per-document stats
                if doc_stats.total_pages > 0:
                    filtered_count = doc_stats.total_pages - doc_stats.kept_pages
                    print(f"\n  [{i+1}/{len(filtered_files)}] {file_path.name}")
                    print(f"    PDF: {doc_stats.kept_pages}/{doc_stats.total_pages} pages kept", end="")
                    if filtered_count > 0:
                        reasons = ", ".join(f"{k}={v}" for k, v in doc_stats.filtered_pages.items())
                        print(f" (filtered: {reasons})", end="")
                    if doc_stats.index_pages_skipped > 0:
                        print(f", index pages={doc_stats.index_pages_skipped}", end="")
                    print()

                stats.document_stats.append(doc_stats)

            else:  # text mode
                doc_text = file_path.read_text(encoding='utf-8', errors='ignore')
                print(f"\n  [{i+1}/{len(filtered_files)}] {file_path.name} - {len(doc_text)} chars")

            if doc_text.strip():
                documents.append({
                    'file_path': str(file_path),
                    'text': doc_text,
                    'metadata': metadata
                })
                stats.processed_documents += 1
                stats.total_chars_after += len(doc_text)
            else:
                stats.failed_documents += 1

        except Exception as e:
            stats.failed_documents += 1
            stats.errors.append((str(file_path), str(e)))
            print(f"\n  ❌ Error processing {file_path.name}: {e}")

    print(f"\n✓ Stage 2 Complete: {stats.processed_documents} documents loaded")

    return documents, stats


print('✓ Cell 16: Stage 2 document loading and filtering functions loaded')


print("" + "="*80)
print("STAGE 1.2: CLEAN SOURCE TEXT")
print("="*80)

cp1_config = load_configuration(CONFIG, CP1_SOURCE, CP1_SOURCE_CHUNKS, CP1_FORCE_MODE)
cp1_doc_stats = ProcessingStats()
cp1_documents_path = None
cp1_chunked_df = None
cp1_chunk_stats = None
cp1_scoring_stats = None

if cp1_config.corpus_mode == 'chunked':
    cp1_chunked_df = pd.read_csv(cp1_config.corpus_source)
    print(f"? Loaded {len(cp1_chunked_df)} pre-chunked rows from {cp1_config.corpus_source}")
else:
    cp1_documents, cp1_doc_stats = load_and_filter_documents(cp1_config, source_fs)
    if not cp1_documents:
        raise RuntimeError('No documents produced in Stage 1.2')
    cp1_documents_df = pd.DataFrame([
        {
            'file_path': doc['file_path'],
            'doc_type': doc['metadata'].get('doc_type', ''),
            'year': doc['metadata'].get('year', ''),
            'document_folder': doc['metadata'].get('document_folder', ''),
            'filename': doc['metadata'].get('filename', ''),
            'clean_text': doc['text']
        }
        for doc in cp1_documents
    ])
    cp1_documents_path = source_fs.folders['Other_data'] / 'cp1_stage1_documents.csv'
    cp1_documents_df.to_csv(cp1_documents_path, index=False)
    print(f"? Cleaned {len(cp1_documents_df)} documents -> {cp1_documents_path}")
    print(f"  Characters (after cleaning): {cp1_doc_stats.total_chars_after:,}")
print("Stage 1.2 complete")






✓ Cell 13: Configuration dataclass and loading functions loaded
✓ Cell 14: Content filtering patterns and functions loaded
✓ Cell 15: PDF processing functions loaded
✓ Cell 16: Stage 2 document loading and filtering functions loaded
STAGE 1.2: CLEAN SOURCE TEXT

STAGE 1: CONFIGURATION & SOURCE LOADING
✓ Using default corpus source: PDF sources

Configuration Summary:
  Source: PDF sources
  Mode: pdf
  Corpus Filter: Disabled
  Chunking: 30 sentences/chunk
  Text Processing: stopwords=True, stemming=False

STAGE 2: DOCUMENT LOADING & TEXT FILTERING
✓ Found 26 PDF files

Processing 26 documents...


Processing:   4%|▍         | 1/26 [00:01<00:25,  1.01s/it]


  [1/26] Allen e.a. - 2023 - Staat en slavernij het Nederlandse koloniale slavernijverleden en zijn doorwerkingen.pdf
    PDF: 360/480 pages kept (filtered: too_few_words=20, empty=13, copyright_metadata=1, front_matter_toc=2, layout_page=3, front_matter_toc_numbers=32, reference_section=28, front_matter_authors=7, too_few_sentences=13, too_many_numbers=1)


Processing:  12%|█▏        | 3/26 [00:01<00:09,  2.44it/s]


  [2/26] Amsterdam en het slavernijverleden.pdf
    PDF: 57/108 pages kept (filtered: too_few_words=17, empty=9, layout_page=24, front_matter_toc_numbers=1)

  [3/26] Caribisch+Nederland+in+beeld+-+Pullen+et+al.+(2024).pdf
    PDF: 31/42 pages kept (filtered: too_few_words=3, layout_page=2, front_matter_authors=1, too_few_sentences=1, reference_section=4)


...


  [21/26] Slavernij (Dirk J. Tang).pdf
    PDF: 196/274 pages kept (filtered: empty=18, too_few_words=52, copyright_metadata=1, too_few_sentences=2, front_matter_authors=1, reference_section=2, front_matter_toc_numbers=2)

  [22/26] Voortgangsrapportage+-+Ervaringen+en+lessen+om+discriminatie+in+publieke+dienstverlening+te+voorkomen+en+te+bestrijden.pdf
    PDF: 41/52 pages kept (filtered: too_few_words=8, layout_page=2, front_matter_authors=1)


Processing:  88%|████████▊ | 23/26 [00:18<00:01,  2.31it/s]


  [23/26] WEB Handelingen_1FC_2023_Opmaak.pdf
    PDF: 61/84 pages kept (filtered: too_few_words=2, front_matter_toc=1, empty=9, front_matter_authors=3, reference_section=6, front_matter_toc_numbers=2)

  [24/26] Wit is nu aan zet Racisme in Nederland.pdf
    PDF: 9/18 pages kept (filtered: too_few_words=3, empty=2, copyright_metadata=1, layout_page=1, too_few_sentences=1, front_matter_authors=1)


Processing: 100%|██████████| 26/26 [00:19<00:00,  1.34it/s]


  [25/26] Zwarte Canon (Chris van der Heijden).pdf
    PDF: 120/126 pages kept (filtered: empty=2, front_matter_authors=1, layout_page=1, too_few_words=2)

  [26/26] ZWART_MANIFEST.pdf
    PDF: 39/48 pages kept (filtered: too_few_words=4, front_matter_toc_numbers=2, front_matter_authors=3)

✓ Stage 2 Complete: 26 documents loaded
? Cleaned 26 documents -> workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Other_data\cp1_stage1_documents.csv
  Characters (after cleaning): 6,337,755
Stage 1.2 complete


In [11]:
# ============================================================
# CELL 1.3: TEXT TO RAW_CHUNK DATASET
# ============================================================
from collections import defaultdict

ENGLISH_HINTS = {'the', 'be', 'to', 'of', 'and', 'a', 'in', 'that', 'have', 'is', 'it', 'for', 'not', 'on', 'with', 'as', 'you', 'do', 'at'}
DUTCH_HINTS = {'de', 'het', 'een', 'van', 'en', 'in', 'op', 'te', 'voor', 'is', 'met', 'aan', 'dat', 'zijn', 'er', 'worden', 'ook', 'naar'}

def split_into_sentences(text: str) -> List[str]:
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]

def likely_english_sentence(sentence: str) -> bool:
    """Helper to detect English sentences (for optional filtering in CP1.4, not here)"""
    tokens = sentence.lower().split()
    if len(tokens) < 3:
        return False
    english = sum(1 for token in tokens if token in ENGLISH_HINTS)
    dutch = sum(1 for token in tokens if token in DUTCH_HINTS)
    return english > dutch

def short_file_hash(path: str, n: int = 8) -> str:
    return hashlib.sha1(path.encode()).hexdigest()[:n]

def make_chunk_uid(file_path: str, chunk_idx: int) -> str:
    return f"{short_file_hash(file_path)}:{chunk_idx:05d}"

def load_clean_documents_dataframe(path: Path) -> pd.DataFrame:
    if not path or not path.exists():
        raise FileNotFoundError('Stage 1.2 output not found. Run previous cell first.')
    df = pd.read_csv(path)
    if 'clean_text' not in df.columns:
        raise ValueError("Stage 1.2 output missing 'clean_text' column")
    return df

def chunk_documents(df: pd.DataFrame, config: ProcessingConfig) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Legacy sentence-based chunking (fallback when token-aware is disabled).

    IMPORTANT: Only creates raw_text. English filtering happens in Cell 1.4.
    """
    chunk_records = []
    stats = {'total_chunks': 0, 'chunks_filtered': defaultdict(int), 'chunks_per_document': [], 'total_sentences': 0}

    for _, row in tqdm(df.iterrows(), total=len(df), desc='Chunking documents'):
        text = str(row['clean_text'])
        sentences = split_into_sentences(text)
        stats['total_sentences'] += len(sentences)
        doc_chunks = 0

        for start in range(0, len(sentences), config.chunk_sentences_per_chunk):
            chunk_sentences = sentences[start:start + config.chunk_sentences_per_chunk]
            if len(chunk_sentences) < config.chunk_min_sentences:
                stats['chunks_filtered']['too_few_sentences'] += 1
                continue

            # Create raw_text ONLY (no filtering)
            raw_text = ' '.join(chunk_sentences)

            chunk_records.append({
                'file_path': row['file_path'],
                'chunk_uid': make_chunk_uid(row['file_path'], len(chunk_records)),
                'raw_text': raw_text,
                'sentence_count': len(chunk_sentences),
                'doc_type': row.get('doc_type', ''),
                'year': row.get('year', ''),
                'document_folder': row.get('document_folder', ''),
                'filename': row.get('filename', '')
            })
            doc_chunks += 1

        stats['chunks_per_document'].append(doc_chunks)

    chunks_df = pd.DataFrame(chunk_records)
    stats['total_chunks'] = len(chunks_df)
    return chunks_df, stats

def chunk_documents_token_aware(
    df: pd.DataFrame,
    config: ProcessingConfig,
    tokenizer,
    max_tokens: int = 500
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Token-aware chunking that respects token limits.

    This is the RECOMMENDED chunking method for bi-encoder pipelines.
    Ensures chunks fit within model's embedding context window.

    IMPORTANT: Only creates raw_text. All text cleaning happens in Cell 1.4.

    Args:
        df: DataFrame with 'clean_text' column
        config: ProcessingConfig object
        tokenizer: HuggingFace tokenizer for counting tokens
        max_tokens: Maximum tokens per chunk (default: 500)

    Returns:
        (chunks_df, stats) where chunks_df has 'token_count' metadata
    """
    chunk_records = []
    stats = {
        'total_chunks': 0,
        'chunks_filtered': defaultdict(int),
        'chunks_per_document': [],
        'total_sentences': 0,
        'token_counts': [],
        'max_tokens_found': 0,
        'avg_tokens_per_chunk': 0.0,
        'chunks_over_limit': 0
    }

    for _, row in tqdm(df.iterrows(), total=len(df), desc='Token-aware chunking'):
        text = str(row['clean_text'])
        sentences = split_into_sentences(text)
        stats['total_sentences'] += len(sentences)
        doc_chunks = 0
        current_sentences: List[str] = []
        current_tokens = 0

        def add_chunk(sent_list: List[str], token_total: int):
            nonlocal doc_chunks
            if not sent_list:
                return
            if len(sent_list) < config.chunk_min_sentences:
                stats['chunks_filtered']['too_few_sentences'] += 1
                return

            # Create raw_text ONLY (no filtering)
            raw_text = ' '.join(sent_list)

            # IMPORTANT: Persist token_count metadata for downstream use
            chunk_records.append({
                'file_path': row['file_path'],
                'chunk_uid': make_chunk_uid(row['file_path'], len(chunk_records)),
                'raw_text': raw_text,
                'sentence_count': len(sent_list),
                'token_count': token_total,  # Key metadata for CP4/CP5 diagnostics
                'doc_type': row.get('doc_type', ''),
                'year': row.get('year', ''),
                'document_folder': row.get('document_folder', ''),
                'filename': row.get('filename', '')
            })
            doc_chunks += 1
            stats['token_counts'].append(token_total)
            if token_total > max_tokens:
                stats['chunks_over_limit'] += 1

        for sentence in sentences:
            sentence = sentence.strip()
            if not sentence:
                continue
            sentence_tokens = len(tokenizer.encode(sentence, add_special_tokens=False))
            if sentence_tokens == 0:
                continue

            if current_sentences and (current_tokens + sentence_tokens > max_tokens):
                add_chunk(current_sentences, current_tokens)
                current_sentences = []
                current_tokens = 0

            current_sentences.append(sentence)
            current_tokens += sentence_tokens

        if current_sentences:
            add_chunk(current_sentences, current_tokens)

        stats['chunks_per_document'].append(doc_chunks)

    chunks_df = pd.DataFrame(chunk_records)
    stats['total_chunks'] = len(chunks_df)
    if stats['token_counts']:
        stats['avg_tokens_per_chunk'] = sum(stats['token_counts']) / len(stats['token_counts'])
        stats['max_tokens_found'] = max(stats['token_counts'])
    else:
        stats['avg_tokens_per_chunk'] = 0.0
        stats['max_tokens_found'] = 0
    # Keep token_counts for debugging but remove from final stats dict for cleaner output
    stats.pop('token_counts', None)
    return chunks_df, stats


print("" + "="*80)
print("STAGE 1.3: CHUNK CLEAN TEXT")
print("="*80)

if cp1_config.corpus_mode == 'chunked':
    print("✓ Stage skipped: using pre-chunked CSV")
    cp1_chunk_stats = {'total_chunks': len(cp1_chunked_df), 'chunks_filtered': {}}
else:
    cp1_documents_df = load_clean_documents_dataframe(cp1_documents_path)

    use_token_chunking = getattr(cp1_config, 'chunk_use_token_aware', False)
    max_tokens_per_chunk = getattr(cp1_config, 'chunk_max_tokens', 500) or 500

    if use_token_chunking:
        print(f"✓ Using token-aware chunking (max {max_tokens_per_chunk} tokens)")

        # Determine tokenizer from CONFIG
        # Priority: cross_encoder.model_name > model.base_model_name > fallback
        tokenizer_name = None
        if CONFIG.get('cross_encoder') and CONFIG['cross_encoder'].get('model_name'):
            tokenizer_name = CONFIG['cross_encoder']['model_name']
            print(f"  Tokenizer source: cross_encoder.model_name = {tokenizer_name}")
        elif CONFIG.get('model') and CONFIG['model'].get('base_model_name'):
            tokenizer_name = CONFIG['model']['base_model_name']
            print(f"  Tokenizer source: model.base_model_name = {tokenizer_name}")
        else:
            # Fallback to GroNLP/bert-base-dutch-cased
            tokenizer_name = 'GroNLP/bert-base-dutch-cased'
            print(f"  Tokenizer source: fallback = {tokenizer_name}")

        try:
            tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
            print(f"  ✓ Loaded tokenizer: {tokenizer_name}")
        except Exception as e:
            print(f"  ⚠️  Failed to load tokenizer '{tokenizer_name}': {e}")
            print(f"  Falling back to sentence-based chunking")
            use_token_chunking = False

        if use_token_chunking:
            cp1_chunked_df_raw, cp1_chunk_stats = chunk_documents_token_aware(
                cp1_documents_df,
                cp1_config,
                tokenizer,
                max_tokens=max_tokens_per_chunk
            )
            avg_tokens = cp1_chunk_stats.get('avg_tokens_per_chunk', 0.0)
            max_tokens_found = cp1_chunk_stats.get('max_tokens_found', 0)
            over_limit = cp1_chunk_stats.get('chunks_over_limit', 0)
            print(f"  Token stats: avg={avg_tokens:.1f}, max={max_tokens_found}, over_limit={over_limit}")

            # Verify token_count column exists
            if 'token_count' in cp1_chunked_df_raw.columns:
                print(f"  ✓ token_count metadata preserved ({cp1_chunked_df_raw['token_count'].notna().sum()} rows)")
            else:
                print(f"  ⚠️  WARNING: token_count column missing from output")

    if not use_token_chunking:
        print(f"✓ Using sentence-based chunking ({cp1_config.chunk_sentences_per_chunk} sentences/chunk)")
        cp1_chunked_df_raw, cp1_chunk_stats = chunk_documents(cp1_documents_df, cp1_config)

    raw_chunks_path = source_fs.folders['Other_data'] / 'cp1_stage2_chunks_raw.csv'
    cp1_chunked_df_raw.to_csv(raw_chunks_path, index=False)
    print(f"✓ Created {len(cp1_chunked_df_raw)} raw chunks -> {raw_chunks_path}")
    if cp1_chunk_stats['chunks_filtered']:
        print(f"  Filters: {dict(cp1_chunk_stats['chunks_filtered'])}")

    # Verify columns
    print(f"\nColumn check:")
    print(f"  ✓ raw_text: {'present' if 'raw_text' in cp1_chunked_df_raw.columns else 'MISSING'}")
    print(f"  ✗ text_for_processing: {'SHOULD NOT EXIST' if 'text_for_processing' in cp1_chunked_df_raw.columns else 'correctly absent'}")
    print(f"  Note: text_for_scoring will be created in Cell 1.4")

    cp1_chunked_df = cp1_chunked_df_raw.copy()

print("✓ Stage 1.3 complete")


STAGE 1.3: CHUNK CLEAN TEXT
✓ Using token-aware chunking (max 500 tokens)
  Tokenizer source: model.base_model_name = NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers
  ✓ Loaded tokenizer: NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers


Token-aware chunking: 100%|██████████| 26/26 [00:04<00:00,  5.72it/s]

  Token stats: avg=474.2, max=500, over_limit=0
  ✓ token_count metadata preserved (3034 rows)
✓ Created 3034 raw chunks -> workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Other_data\cp1_stage2_chunks_raw.csv
  Filters: {'too_few_sentences': 7}

Column check:
  ✓ raw_text: present
  ✗ text_for_processing: correctly absent
  Note: text_for_scoring will be created in Cell 1.4
✓ Stage 1.3 complete


In [12]:
# ============================================================
# CELL 1.4: RAW_TEXT TO TEXT_FOR_SCORING
# ============================================================
# This cell cleans raw_text to create text_for_scoring by:
# - Removing English sentences (if configured)
# - Removing stopwords (Dutch + English + custom)
# - Removing numbers and numeric tokens
# - Removing non-word tokens (punctuation, special chars)
# - Applying stemming (if configured)
# - Filtering chunks below minimum token threshold

import nltk
import re
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

DUTCH_STOPWORDS = set(stopwords.words('dutch'))
ENGLISH_STOPWORDS = set(stopwords.words('english'))
CUSTOM_STOPWORDS = {
    'bijlage', 'inleiding', 'conclusie', 'samenvatting', 'hoofdstuk',
    'paragraaf', 'artikel', 'pagina', 'zie', 'bijvoorbeeld', 'namelijk',
    'aldus', 'echter'
}
ALL_STOPWORDS = DUTCH_STOPWORDS | ENGLISH_STOPWORDS | CUSTOM_STOPWORDS
DUTCH_STEMMER = SnowballStemmer('dutch')

# Pattern for valid words: letters, accented chars, hyphens
WORD_PATTERN = re.compile(r'^[a-zA-ZÀ-ÖØ-öø-ÿ\-]+$')

# English sentence detection (same helpers as chunking)
ENGLISH_HINTS = {'the', 'be', 'to', 'of', 'and', 'a', 'in', 'that', 'have', 'is', 'it', 'for', 'not', 'on', 'with', 'as', 'you', 'do', 'at'}
DUTCH_HINTS = {'de', 'het', 'een', 'van', 'en', 'in', 'op', 'te', 'voor', 'is', 'met', 'aan', 'dat', 'zijn', 'er', 'worden', 'ook', 'naar'}

def likely_english_sentence(sentence: str) -> bool:
    """Check if a sentence is likely English based on common word patterns."""
    tokens = sentence.lower().split()
    if len(tokens) < 3:
        return False
    english = sum(1 for token in tokens if token in ENGLISH_HINTS)
    dutch = sum(1 for token in tokens if token in DUTCH_HINTS)
    return english > dutch

def is_likely_english_word(word: str) -> bool:
    """Check if a word is likely English based on common patterns."""
    # Common English words that don't appear in Dutch
    english_words = {'the', 'and', 'or', 'but', 'with', 'from', 'that', 'this', 'these', 'those'}
    return word.lower() in english_words

def clean_text_for_scoring(text: str, remove_stopwords: bool = True,
                           use_stemming: bool = False,
                           drop_english: bool = True) -> str:
    """
    Clean text by removing stopwords, numbers, non-words, and optionally English words.

    This happens AFTER chunking, so raw_text is preserved for reference.
    """
    # OPTIONAL: Remove English sentences first (if configured)
    if drop_english:
        sentences = re.split(r'(?<=[.!?])\s+', text)
        retained = [s for s in sentences if not likely_english_sentence(s)]
        if not retained:
            # All sentences were English - return empty to filter this chunk
            return ""
        text = ' '.join(retained)

    # Lowercase
    text = text.lower()

    # Split into tokens
    tokens = text.split()

    # Filter tokens
    filtered = []
    for token in tokens:
        # Remove tokens that are purely numeric
        if token.isdigit():
            continue

        # Remove tokens with numbers in them (e.g., "1863", "v20")
        if any(c.isdigit() for c in token):
            continue

        # Only keep tokens that match word pattern (letters, accents, hyphens)
        if not WORD_PATTERN.match(token):
            continue

        # Remove stopwords if configured
        if remove_stopwords and token in ALL_STOPWORDS:
            continue

        # Remove English words if configured
        if drop_english and is_likely_english_word(token):
            continue

        filtered.append(token)

    # Join back
    text = ' '.join(filtered)

    # Apply stemming if configured
    if use_stemming:
        tokens = text.split()
        text = ' '.join(DUTCH_STEMMER.stem(t) for t in tokens)

    return text

def count_tokens(text: str) -> int:
    return len(text.split())

def apply_text_processing(chunks_df: pd.DataFrame, config: ProcessingConfig) -> Tuple[pd.DataFrame, Dict[str, int]]:
    """
    Process raw_text column to create cleaned text_for_scoring column.

    CORRECTED: Now processes raw_text (not text_for_processing which shouldn't exist).
    """
    # Verify raw_text exists
    if 'raw_text' not in chunks_df.columns:
        raise ValueError("raw_text column not found! Check Cell 1.3 output.")

    # Warn if text_for_processing exists (shouldn't be there)
    if 'text_for_processing' in chunks_df.columns:
        print("⚠️  WARNING: text_for_processing column found but will be ignored.")
        print("   Cell 1.3 should only create raw_text. This may be from an old run.")

    text_for_scoring = []
    filtered_for_tokens = 0
    filtered_all_english = 0

    for _, row in tqdm(chunks_df.iterrows(), total=len(chunks_df), desc='Processing chunks'):
        text = row['raw_text']  # CORRECTED: Use raw_text

        # Clean the text
        text = clean_text_for_scoring(
            text,
            remove_stopwords=config.scoring_remove_stopwords,
            use_stemming=config.scoring_use_stemming,
            drop_english=config.scoring_drop_english
        )

        # Filter out chunks that are empty after English removal
        if not text.strip():
            text_for_scoring.append(None)
            filtered_all_english += 1
            continue

        # Filter out chunks that are too short after cleaning
        if count_tokens(text) < config.chunk_min_tokens:
            text_for_scoring.append(None)
            filtered_for_tokens += 1
        else:
            text_for_scoring.append(text)

    processed_df = chunks_df.copy()
    processed_df['text_for_scoring'] = text_for_scoring
    processed_df = processed_df[processed_df['text_for_scoring'].notna()].copy()

    stats = {
        'filtered_all_english': filtered_all_english,
        'filtered_for_tokens': filtered_for_tokens,
        'result_chunks': len(processed_df)
    }
    return processed_df, stats

print("" + "="*80)
print("STAGE 1.4: BUILD text_for_scoring FROM raw_text")
print("="*80)

# Drop existing text_for_scoring column if present to force reprocessing
if 'text_for_scoring' in cp1_chunked_df.columns:
    print("✓ Dropping existing text_for_scoring column to reprocess")
    cp1_chunked_df = cp1_chunked_df.drop(columns=['text_for_scoring'])

# Always process text_for_scoring with proper cleaning
cp1_chunked_df, cp1_scoring_stats = apply_text_processing(cp1_chunked_df, cp1_config)

if cp1_config.corpus_mode == 'chunked':
    print(f"✓ Processed {cp1_scoring_stats['result_chunks']} chunks from pre-chunked CSV")
else:
    print(f"✓ {cp1_scoring_stats['result_chunks']} chunks ready after scoring prep")

if cp1_scoring_stats.get('filtered_all_english'):
    print(f"  Filtered (all English): {cp1_scoring_stats['filtered_all_english']}")
if cp1_scoring_stats.get('filtered_for_tokens'):
    print(f"  Filtered (min tokens): {cp1_scoring_stats['filtered_for_tokens']}")

# Show sample of cleaned text
print("\nSample cleaned text:")
sample_idx = 0
if len(cp1_chunked_df) > 0:
    print(f"  Raw text: {cp1_chunked_df['raw_text'].iloc[sample_idx][:100]}...")
    print(f"  Cleaned:  {cp1_chunked_df['text_for_scoring'].iloc[sample_idx][:100]}...")

# Verify correct columns exist
print(f"\nFinal column check:")
print(f"  ✓ raw_text: {'present' if 'raw_text' in cp1_chunked_df.columns else 'MISSING'}")
print(f"  ✓ text_for_scoring: {'present' if 'text_for_scoring' in cp1_chunked_df.columns else 'MISSING'}")
print(f"  ✗ text_for_processing: {'SHOULD NOT EXIST' if 'text_for_processing' in cp1_chunked_df.columns else 'correctly absent'}")

print("✓ Stage 1.4 complete")


STAGE 1.4: BUILD text_for_scoring FROM raw_text


Processing chunks: 100%|██████████| 3034/3034 [00:01<00:00, 1986.52it/s]

✓ 2840 chunks ready after scoring prep
  Filtered (all English): 95
  Filtered (min tokens): 99

Sample cleaned text:
  Raw text: — STAAT EN SLAVERNIJ —
aan voor haar aandeel in de praktijken die wereldwijd de slavernij in
stand h...
  Cleaned:  staat slavernij aandeel praktijken wereldwijd slavernij stand inzicht berouw decennialang proces gan...

Final column check:
  ✓ raw_text: present
  ✓ text_for_scoring: present
  ✗ text_for_processing: correctly absent
✓ Stage 1.4 complete


In [13]:
# ============================================================
# CELL 1.5: SAVE CHECKPOINT 1 OUTPUTS
# ============================================================
if 'cp1_chunked_df' not in globals() or cp1_chunked_df is None:
    raise RuntimeError('No chunked corpus to save. Run Cells 1.2-1.4 first.')

output_path = source_fs.folders['Other_data'] / 'chunked_corpus.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
cp1_chunked_df.to_csv(output_path, index=False)

stats_payload = {
    'documents_processed': getattr(cp1_doc_stats, 'processed_documents', 0),
    'documents_failed': getattr(cp1_doc_stats, 'failed_documents', 0),
    'pages_kept': getattr(cp1_doc_stats, 'total_pages_kept', 0),
    'pages_extracted': getattr(cp1_doc_stats, 'total_pages_extracted', 0),
    'chunks_total': len(cp1_chunked_df),
    'chunk_filters': dict(cp1_chunk_stats['chunks_filtered']) if cp1_chunk_stats and cp1_chunk_stats.get('chunks_filtered') else {},
    'scoring_stats': cp1_scoring_stats or {}
}

fs.save_data(stats_payload, 'checkpoint1_stats', 'Other_data', 'json')
fs.save_config('checkpoint1_chunking')

print("" + "="*80)
print("CHECKPOINT 1 SAVED")
print("="*80)
print(f"? chunked_corpus.csv -> {output_path}")
print(f"  Documents processed: {stats_payload['documents_processed']}")
print(f"  Total chunks: {stats_payload['chunks_total']}")


✓ Saved: Other_data/checkpoint1_stats.json
✓ Config saved: config_checkpoint1_chunking_20251230_131051.json
CHECKPOINT 1 SAVED
? chunked_corpus.csv -> workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Other_data\chunked_corpus.csv
  Documents processed: 26
  Total chunks: 2840


✅ **CHECKPOINT 1 COMPLETE** - Corpus chunked and saved

**Resume**: Load `chunks_df` from `Other_data/chunked_corpus.csv`

---
# CHECKPOINT 2: Vocabulary Building
---

Build vocabulary from corpus with frequency filtering.

In [18]:
# ============================================================
# CP2: WORKFLOW SOURCE OVERRIDE (Optional)
# ============================================================
# Set to load chunked corpus from a different workflow
# Leave as None to use current workflow

CP2_SOURCE = None #r"C:\Users\Home\policy-analysis\workflow_data\slavery_Slavdict_pretraining_slavery_v25"  # e.g., "workflow_data/pretrained-Slavery_11.10.25_v2"

# Create source filesystem
source_fs = fs.get_source_workflow(CP2_SOURCE) if CP2_SOURCE else fs

if CP2_SOURCE:
    print(f"📂 Loading chunks from: {source_fs.root}")
else:
    print(f"📂 Loading chunks from: current workflow")
print(f"💾 Saving vocabulary to: {fs.root}")

📂 Loading chunks from: current workflow
💾 Saving vocabulary to: workflow_data\slavery_Short-slavdict_pretrained_slavery_v4


In [19]:
# ============================================================
# CELL 2.1: TOKENIZATION FOR VOCAB BUILDING
# ============================================================

_tok_re = re.compile(CONFIG["tokenize"]["pattern"])

def tokenize(text: str) -> list:
    if CONFIG["tokenize"]["lower"]:
        text = text.lower()
    toks = _tok_re.findall(text)
    keep = []
    mn = CONFIG["tokenize"]["min_len"]
    mx = CONFIG["tokenize"]["max_len"]
    for t in toks:
        if not CONFIG["tokenize"]["keep_hyphen"]:
            t = t.replace("-", "")
        if mn <= len(t) <= mx:
            keep.append(t)
    return keep

def read_text(path: Path) -> str:
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return path.read_text(encoding=enc, errors="ignore")
        except Exception:
            pass
    return path.read_text(errors="ignore")

print("✓ Tokenizer ready")

✓ Tokenizer ready


In [20]:
# ============================================================
# BUILD VOCABULARY FROM CHUNKED CORPUS (DIRECT PATH)
# ============================================================

print(f"\n{'='*60}")
print("BUILDING VOCABULARY FROM CHUNKED CORPUS")
print(f"{'='*60}")

# Determine source workflow directory
if CP2_SOURCE:
    workflow_dir = Path(source_fs.root)
    print(f"? Using CP2 source workflow: {workflow_dir}")
else:
    workflow_dir = Path(fs.root)
    print(f"? Using current workflow: {workflow_dir}")

# Check if workflow directory exists
if not workflow_dir.exists():
    print(f"? Error: Workflow directory not found: {workflow_dir}")
    print("   Please run the chunking step first or adjust CP2_SOURCE.")
    workflow_dir = None

if workflow_dir and workflow_dir.exists():
    chunked_corpus_path = workflow_dir / "Other_data" / "chunked_corpus.csv"
else:
    chunked_corpus_path = Path("Other_data") / "chunked_corpus.csv"
    print(f"?? Using relative path: {chunked_corpus_path}")

if not chunked_corpus_path.exists():
    print(f"❌ Error: Chunked corpus not found at {chunked_corpus_path}")
    print("   Please run the chunking step first.")
    print(f"\n   Looking for files in: {chunked_corpus_path.parent}")
    if chunked_corpus_path.parent.exists():
        files = list(chunked_corpus_path.parent.glob("*.csv"))
        print(f"   Found {len(files)} CSV files:")
        for f in files[:10]:
            print(f"     - {f.name}")
else:
    # Load the chunks DataFrame
    print(f"✓ Loading from: {chunked_corpus_path}")
    chunks_df = pd.read_csv(chunked_corpus_path)
    
    print(f"\n✓ Loaded {len(chunks_df)} chunks from {chunks_df['file_path'].nunique()} documents")
    
    # Show filtering info if available
    if 'doc_type' in chunks_df.columns and chunks_df['doc_type'].notna().any():
        print(f"  Doc types: {chunks_df['doc_type'].nunique()} unique")
        print(f"  Years: {chunks_df['year'].nunique()} unique")
    
    # Build vocabulary from the processed text
    print("\n📝 Tokenizing chunks...")
    
    term_freq = Counter()
    doc_freq = Counter()
    chunk_tokens = []
    
    for idx, row in tqdm(chunks_df.iterrows(), total=len(chunks_df), desc="Processing chunks"):
        # Use the already-processed text_for_scoring
        text = row['text_for_scoring']
        
        # Skip empty texts
        if pd.isna(text) or text.strip() == '':
            continue
        
        # Tokenize the text
        toks = tokenize(text)
        
        # Store tokens with chunk ID for reference
        chunk_tokens.append({
            'chunk_uid': row['chunk_uid'],
            'tokens': toks
        })
        
        # Update frequencies
        term_freq.update(toks)
        doc_freq.update(set(toks))  # Count each term once per chunk
    
    print(f"\n✓ Processed {len(chunk_tokens)} chunks with text")
    print(f"  Total tokens: {sum(term_freq.values()):,}")
    print(f"  Unique terms: {len(term_freq):,}")
    
    # Filter vocabulary based on document frequency
    min_df = CONFIG["vocab"]["min_df"]  # Can be ratio (0-1) or absolute count
    max_df = CONFIG["vocab"]["max_df"]  # Can be ratio (0-1) or absolute count
    max_vocab = CONFIG["vocab"]["max_vocab"]

    print(f"\n🔍 Filtering vocabulary...")
    total_chunks = len(chunks_df)

    # Calculate min_df threshold
    if isinstance(min_df, float) and 0 < min_df < 1:
        min_df_threshold = max(1, int(min_df * total_chunks))
        print(f"  Min document frequency: {min_df_threshold} chunks ({min_df*100:.1f}% of corpus)")
    else:
        min_df_threshold = min_df
        min_df_pct = (min_df_threshold / total_chunks) * 100
        print(f"  Min document frequency: {min_df_threshold} chunks ({min_df_pct:.1f}% of corpus)")

    # Calculate max_df threshold
    if isinstance(max_df, float) and 0 < max_df < 1:
        max_df_threshold = int(max_df * total_chunks)
        print(f"  Max document frequency: {max_df_threshold} chunks ({max_df*100:.1f}% of corpus)")
    else:
        max_df_threshold = max_df
        max_df_pct = (max_df_threshold / total_chunks) * 100
        print(f"  Max document frequency: {max_df_threshold} chunks ({max_df_pct:.1f}% of corpus)")

    print(f"  Max vocabulary size: {max_vocab:,}")

    # Filter terms by document frequency (remove too rare AND too common)
    vocab_candidates = [
        (term, freq) for term, freq in term_freq.items()
        if min_df_threshold <= doc_freq[term] <= max_df_threshold
    ]

    # Track what was filtered
    too_rare = sum(1 for term in term_freq if doc_freq[term] < min_df_threshold)
    too_common = sum(1 for term in term_freq if doc_freq[term] > max_df_threshold)
    
    # Sort by frequency and take top max_vocab terms
    vocab_candidates.sort(key=lambda x: x[1], reverse=True)
    vocab_candidates = vocab_candidates[:max_vocab]
    terms = [term for term, _ in vocab_candidates]
    
    print(f"\n✓ Filtered vocabulary: {len(terms)} terms")

    # Show filtering statistics
    if len(terms) > 0:
        print(f"\n📊 Vocabulary filtering results:")
        print(f"  Original unique terms: {len(term_freq):,}")
        print(f"  Removed {too_rare:,} terms (too rare: < {min_df_threshold} chunks)")
        print(f"  Removed {too_common:,} terms (too common: > {max_df_threshold} chunks)")
        print(f"  After frequency filters: {len(vocab_candidates):,}")
        if len(vocab_candidates) > max_vocab:
            truncated = len(vocab_candidates) - max_vocab
            print(f"  Truncated {truncated:,} terms to reach max vocab size")
        print(f"  Final vocabulary: {len(terms):,} terms")

        print(f"\n📈 Most common terms (after filtering):")
        for term, freq in vocab_candidates[:10]:
            df = doc_freq[term]
            df_pct = (df / total_chunks) * 100
            print(f"    - '{term}': {freq:,} occurrences (in {df:,} chunks, {df_pct:.1f}%)")

        if too_common > 0:
            # Show examples of filtered common terms
            common_terms = [(t, doc_freq[t]) for t in term_freq if doc_freq[t] > max_df_threshold]
            common_terms.sort(key=lambda x: x[1], reverse=True)
            print(f"\n⚠️  Examples of removed overly common terms:")
            for term, df in common_terms[:5]:
                df_pct = (df / total_chunks) * 100
                print(f"    - '{term}': appeared in {df:,} chunks ({df_pct:.1f}%)")
    
    # Save vocabulary
    vocab_df = pd.DataFrame({
        'term': terms,
        'term_freq': [term_freq[t] for t in terms],
        'doc_freq': [doc_freq[t] for t in terms]
    })
    fs.save_data(vocab_df, "vocabulary", "Other_data", "csv")

# Also save as JSON for Checkpoint 3 compatibility
vocab_json = {
    "terms": vocab_df['term'].tolist(),
    "term2idx": {term: idx for idx, term in enumerate(vocab_df['term'])},
    "term_freq": dict(zip(vocab_df['term'], vocab_df['term_freq'])),
    "doc_freq": dict(zip(vocab_df['term'], vocab_df['doc_freq']))
}
fs.save_data(vocab_json, "vocabulary", "Other_data", "json")
print(f"\u2713 Vocabulary saved in both CSV and JSON formats")


# Save frequencies for all terms (not just vocabulary)
freq_data = {
    'term_freq': dict(term_freq),
    'doc_freq': dict(doc_freq),
    'n_chunks': len(chunk_tokens),
    'n_documents': chunks_df['file_path'].nunique()
}
fs.save_data(freq_data, "term_frequencies", "Other_data", "json")

# Save the tokenized chunks for later use
tokens_df = pd.DataFrame(chunk_tokens)
fs.save_data(tokens_df, "chunk_tokens", "Other_data", "csv")

fs.save_config("checkpoint2_vocab")

print(f"\n{'='*60}")
print("VOCABULARY BUILDING COMPLETE")
print(f"{'='*60}")




BUILDING VOCABULARY FROM CHUNKED CORPUS
? Using current workflow: workflow_data\slavery_Short-slavdict_pretrained_slavery_v4
✓ Loading from: workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Other_data\chunked_corpus.csv

✓ Loaded 2840 chunks from 26 documents
  Doc types: 26 unique
  Years: 0 unique

📝 Tokenizing chunks...


Processing chunks: 100%|██████████| 2840/2840 [00:00<00:00, 7590.20it/s]



✓ Processed 2840 chunks with text
  Total tokens: 380,352
  Unique terms: 39,661

🔍 Filtering vocabulary...
  Min document frequency: 2 chunks (0.1% of corpus)
  Max document frequency: 2272 chunks (80.0% of corpus)
  Max vocabulary size: 100,000

✓ Filtered vocabulary: 19151 terms

📊 Vocabulary filtering results:
  Original unique terms: 39,661
  Removed 20,510 terms (too rare: < 2 chunks)
  Removed 0 terms (too common: > 2272 chunks)
  After frequency filters: 19,151
  Final vocabulary: 19,151 terms

📈 Most common terms (after filtering):
    - 'nederlandse': 2,108 occurrences (in 1,084 chunks, 38.2%)
    - 'nederland': 1,900 occurrences (in 967 chunks, 34.0%)
    - 'slavernij': 1,831 occurrences (in 775 chunks, 27.3%)
    - 'slaven': 1,482 occurrences (in 479 chunks, 16.9%)
    - 'tweede': 1,478 occurrences (in 615 chunks, 21.7%)
    - 'mensen': 1,371 occurrences (in 783 chunks, 27.6%)
    - 'kamer': 1,352 occurrences (in 331 chunks, 11.7%)
    - 'wel': 1,277 occurrences (in 924 ch

✅ **CHECKPOINT 2 COMPLETE** - Vocabulary built and saved

**Resume**: Load `terms`, `term_freq`, `doc_freq` from saved files

---
# CHECKPOINT 3: Dictionary Expansion
---

⚠️ **MANUAL CURATION REQUIRED AFTER THIS STEP**

Expand seed keywords using semantic similarity.

In [21]:
# ============================================================
# CP3: WORKFLOW SOURCE OVERRIDE (Optional)
# ============================================================

# Load vocabulary from different workflow if needed
CP3_SOURCE = False  # e.g., "workflow_data/pretrained-Slavery_11.10.25_v2"

source_fs = fs.get_source_workflow(CP3_SOURCE) if CP3_SOURCE else fs

if CP3_SOURCE:
    print(f"📂 Loading vocabulary from: {source_fs.root}")
else:
    print(f"📂 Loading vocabulary from: current workflow")
print(f"💾 Saving dictionary suggestions to: {fs.root}")


📂 Loading vocabulary from: current workflow
💾 Saving dictionary suggestions to: workflow_data\slavery_Short-slavdict_pretrained_slavery_v4


In [22]:
# ============================================================
# CELL 3.1: LOAD SENTENCE TRANSFORMER MODEL
# ============================================================

print(f"\n{'='*60}")
print("CHECKPOINT 3 START - LOADING VOCABULARY & MODEL")
print(f"{'='*60}")

# Initialize/verify fs object
if 'fs' not in globals() or not hasattr(fs, 'folders') or not fs.folders:
    # Need to initialize or reload fs
    fs = WorkflowFileSystem(CONFIG)
    
    # Find and load the appropriate workflow
    workflow_base = Path(CONFIG["paths"]["workflow_base"])
    model_type = CONFIG['workflow']['workflow_name']
    topic = CONFIG['workflow']['target']
    
    if CONFIG['workflow']['version']:
        # Find specific version (pattern: model_type-topic_DATE_version)
        pattern = f"{model_type}-{topic}_*_{CONFIG['workflow']['version']}"
        matching_dirs = list(workflow_base.glob(pattern))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[0]
    else:
        # Find most recent workflow
        pattern = f"{model_type}-{topic}_*"
        matching_dirs = sorted(list(workflow_base.glob(pattern)))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[-1]
    
    fs.load_existing_workflow(workflow_dir)
    print(f"📂 Loaded workflow: {fs.root.name}")

# Load vocabulary
vocab_path = fs.folders['Other_data'] / 'vocabulary.csv'
if not vocab_path.exists():
    # Try JSON format
    vocab_path = fs.folders['Other_data'] / 'vocabulary.json'
    if not vocab_path.exists():
        raise FileNotFoundError(f"Vocabulary not found at {fs.folders['Other_data']}")

# Load vocabulary from JSON (preferred format)
if vocab_path.suffix == '.json':
    with open(vocab_path, 'r', encoding='utf-8') as f:
        vocab_data = json.load(f)
    
    terms = vocab_data['terms']
    term2idx = vocab_data['term2idx']
    term_freq_dict = vocab_data.get('term_freq', {})
    doc_freq_dict = vocab_data.get('doc_freq', {})
    print(f"✓ Loaded vocabulary (JSON): {len(terms)} terms")
else:
    # Load from CSV
    vocab_df = pd.read_csv(vocab_path)
    terms = vocab_df['term'].tolist()
    term2idx = {t: i for i, t in enumerate(terms)}
    term_freq_dict = {}
    doc_freq_dict = {}
    print(f"✓ Loaded vocabulary (CSV): {len(terms)} terms")

# Filter out NaN and invalid terms (applies to both CSV and JSON)
original_count = len(terms)
terms = [t for t in terms if isinstance(t, str) and t and t == t]  # t == t filters NaN
term2idx = {t: i for i, t in enumerate(terms)}
if len(terms) < original_count:
    print(f"⚠ Filtered out {original_count - len(terms)} invalid/NaN terms from vocabulary")
    print(f"✓ Clean vocabulary: {len(terms)} valid terms")

# Load term frequencies if available
term_freq_path = fs.folders['Other_data'] / 'term_frequencies.json'
if term_freq_path.exists() and not term_freq_dict:
    with open(term_freq_path, 'r') as f:
        freq_data = json.load(f)
    term_freq_dict = freq_data.get('term_freq', {})
    doc_freq_dict = freq_data.get('doc_freq', {})
    print(f"✓ Loaded term frequencies")

print(f"\n{'='*60}")
print("LOADING SENTENCE TRANSFORMER MODEL")
print(f"{'='*60}")

# FIXED: Better handling of pretrained model path
if CONFIG['model']['use_pretrained'] and CONFIG['paths']['pretrained_model_path']:
    model_path = Path(CONFIG['paths']['pretrained_model_path'])
    
    # Check if path already points to sbert_base, or if we need to append it
    # Priority 1: Look for trained_encoder/ (domain-adapted)
    trained_encoder_path = model_path / 'trained_encoder'
    if trained_encoder_path.exists():
        sbert_path = trained_encoder_path
    # Priority 2: Look for sbert_base/ (backward compatibility)
    elif (model_path / 'sbert_base').exists():
        sbert_path = model_path / 'sbert_base'
    # Priority 3: Model path itself
    elif model_path.name == 'sbert_base':
        sbert_path = model_path
    else:
        sbert_path = model_path
    
    if sbert_path.exists():
        st_model = SentenceTransformer(str(sbert_path))
        print(f"✓ Loaded pretrained SBERT model from:")
        print(f"  {sbert_path}")
    else:
        print(f"⚠ Pretrained SBERT path not found: {sbert_path}")
        st_model = SentenceTransformer(CONFIG['model']['base_model_name'])
        print(f"✓ Using base model: {CONFIG['model']['base_model_name']}")
else:
    st_model = SentenceTransformer(CONFIG['model']['base_model_name'])
    print(f"✓ Loaded base model: {CONFIG['model']['base_model_name']}")

def st_embed(texts: list, batch_size: int = 256) -> np.ndarray:
    return st_model.encode(
        texts, 
        batch_size=batch_size, 
        show_progress_bar=False, 
        normalize_embeddings=False
    )

print(f"\n✓ Model ready")
print(f"  Max sequence length: {st_model.max_seq_length}")
print(f"  Embedding dimension: {st_model.get_sentence_embedding_dimension()}")


CHECKPOINT 3 START - LOADING VOCABULARY & MODEL
✓ Loaded vocabulary (CSV): 19151 terms
⚠ Filtered out 1 invalid/NaN terms from vocabulary
✓ Clean vocabulary: 19150 valid terms
✓ Loaded term frequencies

LOADING SENTENCE TRANSFORMER MODEL
✓ Loaded base model: NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers

✓ Model ready
  Max sequence length: 128
  Embedding dimension: 768


In [23]:
# ============================================================
# CELL 3.2: UNIFIED EMBEDDING - VOCABULARY + SEED TERMS
# ============================================================

print(f"\n{'='*60}")
print("UNIFIED EMBEDDING: VOCABULARY + SEED DICTIONARY")
print(f"{'='*60}")

# ------------------------------------------------------------
# Step 1: Extract all seed terms from CONFIG
# ------------------------------------------------------------
seed_dict = CONFIG.get("topics") or CONFIG["dictionary"].get("default_topics", {})
all_seed_terms = []
seed_to_topic = {}

for topic, seeds in seed_dict.items():
    for s in seeds:
        if s and isinstance(s, str):
            all_seed_terms.append(s)
            seed_to_topic[s] = topic

all_seed_terms = list(set(all_seed_terms))  # deduplicate
print(f"✓ Extracted {len(all_seed_terms)} unique seed terms from CONFIG")

# ------------------------------------------------------------
# Step 2: Load dictionary Excel for additional metadata
# ------------------------------------------------------------
seed_dict_df = None
excel_path = (
    CONFIG.get("dictionary_excel")
    or CONFIG.get("paths", {}).get("dictionary_excel")
    or CONFIG.get("dictionary", {}).get("dictionary_excel")
    or CONFIG.get("dictionary", {}).get("dictionairy_excel")
)

if excel_path:
    from pathlib import Path
    excel_path = Path(excel_path)
    if excel_path.exists():
        if excel_path.suffix.lower() in [".xls", ".xlsx"]:
            seed_dict_df = pd.read_excel(
                excel_path,
                sheet_name=CONFIG["dictionary"].get("sheet_name", 0)
            )
        else:
            seed_dict_df = pd.read_csv(excel_path)

        # Normalize column names
        seed_dict_df.columns = [c.strip().lower() for c in seed_dict_df.columns]

        # Map configured columns to standard names
        topic_col_cfg = CONFIG["dictionary"].get("topic_column", "topic").lower()
        term_col_cfg = CONFIG["dictionary"].get("keyword_column", "term").lower()

        rename_map = {}
        if topic_col_cfg in seed_dict_df.columns:
            rename_map[topic_col_cfg] = "topic"
        if term_col_cfg in seed_dict_df.columns:
            rename_map[term_col_cfg] = "term"

        seed_dict_df = seed_dict_df.rename(columns=rename_map)

        required_cols = {"topic", "term", "weight", "category"}
        if required_cols.issubset(seed_dict_df.columns):
            print(f"✓ Loaded dictionary Excel: {len(seed_dict_df)} rows")
        else:
            print(f"⚠ Dictionary Excel missing required columns {required_cols}")
            seed_dict_df = None
    else:
        print(f"⚠ Dictionary Excel path does not exist: {excel_path}")
else:
    print("⚠ No dictionary_excel path in CONFIG")

# ------------------------------------------------------------
# Step 3: Create unified term list (vocab + seeds)
# ------------------------------------------------------------
# Separate vocab terms from seed terms
vocab_only_terms = [t for t in terms if t not in all_seed_terms]
print(f"\n📊 Term breakdown:")
print(f"  Vocabulary terms: {len(vocab_only_terms)}")
print(f"  Seed terms: {len(all_seed_terms)}")
print(f"  Total terms to embed: {len(vocab_only_terms) + len(all_seed_terms)}")

# Create combined list: vocab first, then seeds
all_terms_to_embed = vocab_only_terms + all_seed_terms

# ------------------------------------------------------------
# Step 4: Unified embedding in single operation
# ------------------------------------------------------------
print(f"\n{'='*60}")
print("EMBEDDING ALL TERMS IN UNIFIED SPACE")
print(f"{'='*60}")

all_embeddings = st_embed(all_terms_to_embed)
print(f"✓ Encoded {len(all_terms_to_embed)} terms → shape {all_embeddings.shape}")

# Split embeddings back into vocab and seed
vocab_emb = all_embeddings[:len(vocab_only_terms)]
seed_emb = all_embeddings[len(vocab_only_terms):]

print(f"  Vocab embeddings: {vocab_emb.shape}")
print(f"  Seed embeddings: {seed_emb.shape}")

# Create indices
term2idx = {t: i for i, t in enumerate(vocab_only_terms)}
seed2idx = {t: i for i, t in enumerate(all_seed_terms)}

# ------------------------------------------------------------
# Step 5: Organize seed embeddings by topic
# ------------------------------------------------------------
seed_embeddings_by_topic = {}

for topic, seeds in seed_dict.items():
    valid_seeds = [s for s in seeds if s in seed2idx]
    if not valid_seeds:
        continue

    seed_indices = [seed2idx[s] for s in valid_seeds]
    topic_seed_embeddings = seed_emb[seed_indices]

    # Get weights and categories if available
    weights_dict = {}
    categories_dict = {}

    if seed_dict_df is not None:
        topic_slice = seed_dict_df[seed_dict_df["topic"] == topic]
        for s in valid_seeds:
            term_row = topic_slice[topic_slice["term"] == s]
            if len(term_row) > 0:
                weights_dict[s] = term_row["weight"].values[0]
                categories_dict[s] = term_row["category"].values[0]

    seed_embeddings_by_topic[topic] = {
        "terms": valid_seeds,
        "embeddings": topic_seed_embeddings,
        "weights": weights_dict,
        "categories": categories_dict
    }

print(f"\n✓ Organized seed embeddings by topic:")
for topic, info in seed_embeddings_by_topic.items():
    print(f"  {topic}: {len(info['terms'])} seeds")

# ------------------------------------------------------------
# Step 6: Save embeddings and metadata
# ------------------------------------------------------------
print(f"\n{'='*60}")
print("SAVING EMBEDDINGS AND METADATA")
print(f"{'='*60}")

# Save vocabulary embeddings (only vocab terms, not seeds)
vocab_emb_path = fs.folders['Other_data'] / 'vocab_embeddings.npy'
np.save(vocab_emb_path, vocab_emb)
print(f"✓ Saved vocab embeddings: {vocab_emb.shape}")

# Save seed embeddings separately
seed_emb_path = fs.folders['Other_data'] / 'seed_embeddings.npy'
np.save(seed_emb_path, seed_emb)
print(f"✓ Saved seed embeddings: {seed_emb.shape}")

# Save metadata
vocab_meta = {
    'terms': vocab_only_terms,
    'term2idx': term2idx
}
vocab_meta_path = fs.folders['Other_data'] / 'vocab_meta.json'
with open(vocab_meta_path, 'w') as f:
    json.dump(vocab_meta, f, indent=2)
print(f"✓ Saved vocab metadata: {len(vocab_only_terms)} terms")

seed_meta = {
    'terms': all_seed_terms,
    'seed2idx': seed2idx,
    'seed_to_topic': seed_to_topic
}
seed_meta_path = fs.folders['Other_data'] / 'seed_meta.json'
with open(seed_meta_path, 'w') as f:
    json.dump(seed_meta, f, indent=2)
print(f"✓ Saved seed metadata: {len(all_seed_terms)} terms")

# Save organized seed embeddings by topic
seed_by_topic_path = fs.folders['Other_data'] / 'seed_embeddings_by_topic.npz'
np.savez(
    seed_by_topic_path,
    **{topic: info['embeddings'] for topic, info in seed_embeddings_by_topic.items()}
)
print(f"✓ Saved seed embeddings by topic")

# Save topic metadata
topic_meta = {
    topic: {
        'terms': info['terms'],
        'weights': info['weights'],
        'categories': info['categories']
    }
    for topic, info in seed_embeddings_by_topic.items()
}
topic_meta_path = fs.folders['Other_data'] / 'seed_topic_meta.json'
with open(topic_meta_path, 'w') as f:
    json.dump(topic_meta, f, indent=2)
print(f"✓ Saved topic metadata")

print(f"\n{'='*60}")
print("UNIFIED EMBEDDING COMPLETE")
print(f"{'='*60}")
print(f"\n💡 Key improvement: All terms embedded in same context")
print(f"   → More consistent similarity scores")
print(f"   → Better contextual relationships")
print(f"   → Reproducible results")


UNIFIED EMBEDDING: VOCABULARY + SEED DICTIONARY
✓ Extracted 123 unique seed terms from CONFIG
✓ Loaded dictionary Excel: 134 rows

📊 Term breakdown:
  Vocabulary terms: 19058
  Seed terms: 123
  Total terms to embed: 19181

EMBEDDING ALL TERMS IN UNIFIED SPACE
✓ Encoded 19181 terms → shape (19181, 768)
  Vocab embeddings: (19058, 768)
  Seed embeddings: (123, 768)

✓ Organized seed embeddings by topic:
  Colonial Systems: 30 seeds
  Heritage & Memory: 31 seeds
  Historical Slavery: 39 seeds
  Modern Racism & Discrimination: 34 seeds

SAVING EMBEDDINGS AND METADATA
✓ Saved vocab embeddings: (19058, 768)
✓ Saved seed embeddings: (123, 768)
✓ Saved vocab metadata: 19058 terms
✓ Saved seed metadata: 123 terms
✓ Saved seed embeddings by topic
✓ Saved topic metadata

UNIFIED EMBEDDING COMPLETE

💡 Key improvement: All terms embedded in same context
   → More consistent similarity scores
   → Better contextual relationships
   → Reproducible results


In [24]:
# ============================================================
# CELL 3.3: EXPAND SEED TERMS USING PRE-COMPUTED EMBEDDINGS
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

print(f"\n{'='*60}")
print("EXPANDING SEED TERMS (UNIFIED EMBEDDING METHOD)")
print(f"{'='*60}")

# ------------------------------------------------------------
# Step 1: Compute cosine similarity between seeds and vocab
# ------------------------------------------------------------
print("\nComputing cosine similarity between seed and vocabulary embeddings...")

# This is the key improvement: direct similarity between pre-computed embeddings
# All embeddings were created in the same context (cell 3.2)
similarity_matrix = cosine_similarity(seed_emb, vocab_emb)
print(f"✓ Similarity matrix: {similarity_matrix.shape}")
print(f"  ({len(all_seed_terms)} seeds × {len(vocab_only_terms)} vocab terms)")

# ------------------------------------------------------------
# Step 2: Expand dictionary using similarity matrix
# ------------------------------------------------------------
topic_rows = []
k = CONFIG["expand"]["k_nearest"]
min_cos = CONFIG["expand"]["min_cosine"]

seed_dict = CONFIG.get("topics") or CONFIG["dictionary"].get("default_topics", {})

for topic, seeds in seed_dict.items():
    print(f"\n  Processing: {topic}")
    seen = {}  # term -> best_cosine_score

    # Get topic's seed info
    topic_info = seed_embeddings_by_topic.get(topic, {})
    topic_seed_terms = topic_info.get('terms', [])

    if not topic_seed_terms:
        print(f"    ⚠ No valid seeds for topic")
        continue

    # For each seed in this topic
    for seed_term in topic_seed_terms:
        if seed_term not in seed2idx:
            continue

        seed_idx = seed2idx[seed_term]

        # Get similarity scores for this seed to all vocab terms
        seed_similarities = similarity_matrix[seed_idx]

        # Get top-k most similar vocab terms
        # argsort returns indices sorted by similarity (ascending)
        top_k_indices = np.argsort(seed_similarities)[-k:][::-1]  # reverse for descending

        for vocab_idx in top_k_indices:
            vocab_term = vocab_only_terms[vocab_idx]
            sim_score = float(seed_similarities[vocab_idx])

            # Apply filters
            if doc_freq.get(vocab_term, 0) < CONFIG["vocab"]["min_df"]:
                continue
            if sim_score < min_cos:
                continue

            # Keep best score for each term
            if (vocab_term not in seen) or (sim_score > seen[vocab_term]):
                seen[vocab_term] = sim_score

    # Add original seeds with score 1.0
    for seed_term in topic_seed_terms:
        seen[seed_term] = max(seen.get(seed_term, 0.0), 1.0)

    # Sort and limit
    rows = sorted(seen.items(), key=lambda x: x[1], reverse=True)[
        : CONFIG["expand"]["topN_per_topic"]
    ]

    # --------------------------------------------------------
    # Step 3: Parent inheritance for discovered terms
    # --------------------------------------------------------
    parent_lookup = {}

    if topic_info.get("embeddings") is not None and len(topic_info["embeddings"]) > 0:
        # Find discovered terms (not in original seeds)
        discovered_terms = [w for w, _ in rows if w not in seeds and w in term2idx]

        if discovered_terms:
            print(f"    Finding parents for {len(discovered_terms)} discovered terms...")

            # Get embeddings for discovered terms from vocab_emb
            disc_indices = [term2idx[w] for w in discovered_terms]
            disc_embeds = vocab_emb[disc_indices]

            # Compute similarity to topic seed embeddings
            parent_sims = cosine_similarity(disc_embeds, topic_info["embeddings"])

            # For each discovered term, find best parent
            for i, disc_term in enumerate(discovered_terms):
                best_parent_idx = parent_sims[i].argmax()
                best_parent = topic_info["terms"][best_parent_idx]
                parent_sim = float(parent_sims[i][best_parent_idx])

                parent_lookup[disc_term] = {
                    "parent": best_parent,
                    "similarity": parent_sim,
                    "weight": topic_info["weights"].get(
                        best_parent,
                        CONFIG["weights"]["default_core_weight"]
                    ),
                    "category": topic_info["categories"].get(
                        best_parent,
                        "related_moderate"
                    )
                }

    # --------------------------------------------------------
    # Step 4: Build output rows with weights and categories
    # --------------------------------------------------------
    topic_seed_weights = CONFIG.get("topic_seed_weights", {}).get(topic, {})
    default_discovered_weight = CONFIG["weights"]["default_discovered_weight"]
    default_core_weight = CONFIG["weights"]["default_core_weight"]

    for w, sc in rows:
        is_seed = w in seeds

        # Defaults
        weight = default_discovered_weight
        category = "related_moderate"
        parent = w if is_seed else "unknown"

        if is_seed:
            # Seed term: get weight from config or Excel
            weight = topic_seed_weights.get(w, default_core_weight)

            if seed_dict_df is not None:
                catrow = seed_dict_df[
                    (seed_dict_df["topic"] == topic) & (seed_dict_df["term"] == w)
                ]
                if len(catrow) > 0:
                    category = catrow["category"].values[0]
                else:
                    category = "core_problem" if weight >= default_core_weight else "related_moderate"
            else:
                category = "core_problem" if weight >= default_core_weight else "related_moderate"
        else:
            # Discovered term: inherit from parent
            if w in parent_lookup:
                parent_info = parent_lookup[w]
                weight = parent_info["weight"]
                category = parent_info["category"]
                parent = parent_info["parent"]

        topic_rows.append({
            "topic": topic,
            "term": w,
            "cosine": round(sc, 4),
            "df": int(doc_freq.get(w, 0)),
            "weight": round(weight, 2),
            "category": category,
            "parent": parent,
            "is_seed": int(is_seed)
        })

    print(f"    → Found {len(rows)} terms")

# ------------------------------------------------------------
# Step 5: Save results
# ------------------------------------------------------------
expanded_df = pd.DataFrame(topic_rows)

fs.save_data(expanded_df, "expanded_candidates", "Dictionary", "csv")
fs.save_config("checkpoint3_expansion")

print(f"\n✓ Generated {len(expanded_df)} terms")
print("✓ Saved expanded_candidates.csv")
print("✓ Saved checkpoint3_expansion")

print("\n⚠️ MANUAL CURATION REQUIRED:")
print("  1. Review: Dictionary/expanded_candidates.csv")
print("  2. Remove irrelevant terms")
print("  3. Save as: Dictionary/curated_dictionary.csv")
print("  4. Then proceed to CHECKPOINT 4")

print(f"\n{'='*60}")
print("CHECKPOINT 3 COMPLETE")
print(f"{'='*60}")

print("\n💡 Method improvements:")
print("   ✓ Unified embedding space (seeds + vocab together)")
print("   ✓ Direct cosine similarity (no re-encoding)")
print("   ✓ Consistent similarity scores")
print("   ✓ Better reproducibility")


EXPANDING SEED TERMS (UNIFIED EMBEDDING METHOD)

Computing cosine similarity between seed and vocabulary embeddings...
✓ Similarity matrix: (123, 19058)
  (123 seeds × 19058 vocab terms)

  Processing: Colonial Systems
    Finding parents for 270 discovered terms...
    → Found 300 terms

  Processing: Heritage & Memory
    Finding parents for 270 discovered terms...
    → Found 300 terms

  Processing: Historical Slavery
    Finding parents for 261 discovered terms...
    → Found 300 terms

  Processing: Modern Racism & Discrimination
    Finding parents for 266 discovered terms...
    → Found 300 terms
✓ Saved: Dictionary/expanded_candidates.csv
✓ Config saved: config_checkpoint3_expansion_20251230_131232.json

✓ Generated 1200 terms
✓ Saved expanded_candidates.csv
✓ Saved checkpoint3_expansion

⚠️ MANUAL CURATION REQUIRED:
  1. Review: Dictionary/expanded_candidates.csv
  2. Remove irrelevant terms
  3. Save as: Dictionary/curated_dictionary.csv
  4. Then proceed to CHECKPOINT 4

C

✅ **CHECKPOINT 3 COMPLETE** - Dictionary expanded

⚠️ **STOP HERE** - Manually curate `expanded_candidates.csv` and save as `curated_dictionary.csv`

---
# CHECKPOINT 4: Topic Vector Creation
---

Build weighted topic vectors from curated dictionary.

In [32]:
# ============================================================
# CP4: WORKFLOW SOURCE OVERRIDE (Optional)
# ============================================================

# Can load dictionary and vocabulary from different workflows
CP4_DICT_SOURCE = None  # For curated dictionary
CP4_VOCAB_SOURCE = None  # For vocabulary and embeddings

dict_fs = fs.get_source_workflow(CP4_DICT_SOURCE) if CP4_DICT_SOURCE else fs
vocab_fs = fs.get_source_workflow(CP4_VOCAB_SOURCE) if CP4_VOCAB_SOURCE else fs

if CP4_DICT_SOURCE:
    print(f"📂 Loading dictionary from: {dict_fs.root}")
else:
    print(f"📂 Loading dictionary from: current workflow")

if CP4_VOCAB_SOURCE:
    print(f"📂 Loading vocabulary from: {vocab_fs.root}")
else:
    print(f"📂 Loading vocabulary from: current workflow")

print(f"💾 Saving topic vectors to: {fs.root}")


📂 Loading dictionary from: current workflow
📂 Loading vocabulary from: current workflow
💾 Saving topic vectors to: workflow_data\slavery_Short-slavdict_pretrained_slavery_v4


In [33]:
# ============================================================
# CELL 4.1: LOAD CURATED DICTIONARY & BUILD TOPIC VECTORS
# ============================================================


print(f"\n{'='*60}")
print("CHECKPOINT 4 START - LOADING REQUIRED DATA")
print(f"{'='*60}")

# Initialize/verify fs object
if 'fs' not in globals() or not hasattr(fs, 'folders') or not fs.folders:
    # Need to initialize or reload fs
    fs = WorkflowFileSystem(CONFIG)

    # Find and load the appropriate workflow
    workflow_base = Path(CONFIG["paths"]["workflow_base"])
    model_type = CONFIG['workflow']['workflow_name']
    topic = CONFIG['workflow']['target']

    if CONFIG['workflow']['version']:
        # Find specific version (pattern: model_type-topic_DATE_version)
        pattern = f"{model_type}-{topic}_*_{CONFIG['workflow']['version']}"
        matching_dirs = list(workflow_base.glob(pattern))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[0]
    else:
        # Find most recent workflow
        pattern = f"{model_type}-{topic}_*"
        matching_dirs = sorted(list(workflow_base.glob(pattern)))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[-1]

    fs.load_existing_workflow(workflow_dir)
    print(f"📂 Loaded workflow: {fs.root.name}")

# Load term frequencies
term_freq_path = vocab_fs.folders['Other_data'] / 'term_frequencies.json'
if not term_freq_path.exists():
    raise FileNotFoundError(f"Term frequencies not found at {term_freq_path}")

with open(term_freq_path, 'r') as f:
    freq_data = json.load(f)
term_freq = freq_data['term_freq']
doc_freq = freq_data['doc_freq']
print(f"✓ Loaded term frequencies")

# Load vocabulary embeddings and metadata
# IMPORTANT: In v19+, vocab_embeddings.npy contains only vocab terms (seeds excluded)
vocab_emb_path = vocab_fs.folders['Other_data'] / 'vocab_embeddings.npy'
vocab_meta_path = vocab_fs.folders['Other_data'] / 'vocab_meta.json'

if not vocab_emb_path.exists() or not vocab_meta_path.exists():
    raise FileNotFoundError(
        f"Vocabulary embeddings not found. Please run CHECKPOINT 3 (cells 3.1-3.3) first.\n"
        f"Expected files:\n"
        f"  - {vocab_emb_path}\n"
        f"  - {vocab_meta_path}"
    )

# Load embeddings
vocab_emb = np.load(vocab_emb_path)
print(f"✓ Loaded vocab embeddings: {vocab_emb.shape}")

# Load metadata (contains the correct term list matching embeddings)
with open(vocab_meta_path, 'r') as f:
    vocab_meta = json.load(f)

terms = vocab_meta['terms']
term2idx = vocab_meta['term2idx']
print(f"✓ Loaded vocabulary metadata: {len(terms)} terms")

# Verify alignment
if len(terms) != vocab_emb.shape[0]:
    raise ValueError(
        f"Mismatch between vocab terms ({len(terms)}) and embeddings ({vocab_emb.shape[0]})!\n"
        f"This suggests corrupted data. Please re-run CHECKPOINT 3."
    )

# Create vocab2vec mapping (term -> embedding vector)
# Filter out NaN/invalid terms
print("Creating vocab2vec mapping...")
valid_terms = [t for t in terms if isinstance(t, str) and t and t == t]  # t == t filters NaN
if len(valid_terms) < len(terms):
    print(f"⚠ Filtered out {len(terms) - len(valid_terms)} invalid/NaN terms")
    terms = valid_terms

# CHECKPOINT 4 ENHANCEMENT: Normalize vocab embeddings before creating lookup
print("Normalizing vocabulary embeddings...")
vocab_emb_normalized = vocab_emb / (np.linalg.norm(vocab_emb, axis=1, keepdims=True) + 1e-12)
vocab2vec = {term: vocab_emb_normalized[term2idx[term]] for term in terms if term in term2idx}
print(f"✓ Created normalized vocab2vec: {len(vocab2vec)} terms")


# Load seed embeddings and metadata
seed_emb_path = fs.folders['Other_data'] / 'seed_embeddings.npy'
seed_meta_path = fs.folders['Other_data'] / 'seed_meta.json'

seed_vectors = {}
seed_terms_list = []
if seed_emb_path.exists() and seed_meta_path.exists():
    seed_emb = np.load(seed_emb_path)
    print(f"✓ Loaded seed embeddings: {seed_emb.shape}")

    with open(seed_meta_path, 'r') as f:
        seed_meta = json.load(f)

    seed_terms = seed_meta.get('terms', [])
    seed_term2idx = seed_meta.get('seed2idx', {})
    print(f"✓ Loaded seed metadata: {len(seed_terms)} terms")

    if len(seed_terms) != seed_emb.shape[0]:
        raise ValueError(
            f"Mismatch between seed terms ({len(seed_terms)}) and embeddings ({seed_emb.shape[0]})!\n"
            f"This suggests corrupted data. Please re-run CHECKPOINT 3."
        )

    valid_seed_terms = [t for t in seed_terms if isinstance(t, str) and t and t == t]
    if len(valid_seed_terms) < len(seed_terms):
        print(f"⚠ Filtered out {len(seed_terms) - len(valid_seed_terms)} invalid/NaN seed terms")
        seed_terms = valid_seed_terms

    # CHECKPOINT 4 ENHANCEMENT: Normalize seed embeddings identically
    print("Normalizing seed embeddings...")
    seed_emb_normalized = seed_emb / (np.linalg.norm(seed_emb, axis=1, keepdims=True) + 1e-12)
    seed_vectors = {term: seed_emb_normalized[seed_term2idx[term]] for term in seed_terms if term in seed_term2idx}
    seed_terms_list = list(seed_vectors.keys())
    print(f"✓ Created normalized seed vector lookup: {len(seed_vectors)} terms")
else:
    print(f"⚠ Seed embeddings not found at {seed_emb_path} / {seed_meta_path}")
    print(f"   Proceeding with corpus-only embeddings")

# CHECKPOINT 4 ENHANCEMENT: Track source of each term (seed vs corpus)
term_source = {}  # term -> 'corpus' or 'seed'
for term in vocab2vec.keys():
    term_source[term] = 'corpus'

# Merge vocab and seed vectors for unified lookup
combined_vocab2vec = dict(vocab2vec)
overlap_count = 0
new_seed_count = 0

if seed_vectors:
    overlap_terms = set(combined_vocab2vec.keys()).intersection(seed_vectors.keys())
    new_seed_terms = set(seed_vectors.keys()) - set(combined_vocab2vec.keys())

    if overlap_terms:
        print(f"\n{'='*60}")
        print("SEED + CORPUS MERGE DIAGNOSTICS")
        print(f"{'='*60}")
        print(f"  Corpus-only terms: {len(combined_vocab2vec) - len(overlap_terms)}")
        print(f"  Overlapping terms: {len(overlap_terms)} (seed embeddings will be used)")
        print(f"  Seed-only terms: {len(new_seed_terms)}")
        overlap_count = len(overlap_terms)
        new_seed_count = len(new_seed_terms)

        # Show a few examples of overlaps
        if overlap_terms:
            sample_overlaps = list(overlap_terms)[:5]
            print(f"  Example overlaps: {', '.join(sample_overlaps)}")

    # Update with seed vectors (seeds override corpus on overlap)
    combined_vocab2vec.update(seed_vectors)

    # Update source tracking
    for term in seed_vectors.keys():
        term_source[term] = 'seed'

    total_terms = len(combined_vocab2vec)
    corpus_only = total_terms - len(seed_vectors)
    print(f"\n✓ Merged vocab + seed vectors: {total_terms} unique terms")
    print(f"  Breakdown: {corpus_only} corpus-only, {overlap_count} seed-override, {new_seed_count} seed-only")
else:
    print("\n⚠ Proceeding without seed embeddings (corpus-only mode)")

# Use merged vocabulary going forward
vocab2vec = combined_vocab2vec
terms = list(vocab2vec.keys())

# Verify all embeddings are normalized
sample_norms = [np.linalg.norm(vocab2vec[t]) for t in list(vocab2vec.keys())[:10]]
avg_norm = np.mean(sample_norms)
if abs(avg_norm - 1.0) > 0.01:
    print(f"⚠ WARNING: Embeddings may not be properly normalized (avg norm={avg_norm:.4f})")
else:
    print(f"✓ Embedding normalization verified (avg norm={avg_norm:.4f})")



print(f"\n{'='*60}")
print("BUILDING TOPIC VECTORS FROM CURATED DICTIONARY")
print(f"{'='*60}")

curated_path = dict_fs.folders['Dictionary'] / 'curated_dictionary.csv'

if not curated_path.exists():
    print(f"❌ Curated dictionary not found: {curated_path}")
    print(f"   Please complete manual curation first!")
else:
    pruned = pd.read_csv(curated_path)
    print(f"✓ Loaded curated dictionary: {len(pruned)} terms, {pruned['topic'].nunique()} topics")

    # Get default weights from config
    default_core_weight = CONFIG['weights']['default_core_weight']
    default_discovered_weight = CONFIG['weights']['default_discovered_weight']
    weighting_scheme = CONFIG['weights']['weighting_scheme']

    # Check if curated dictionary has seed weights
    has_seed_weights = 'weight' in pruned.columns
    if has_seed_weights:
        print(f"✓ Curated dictionary has seed weights column")
    else:
        print(f"⚠ No seed weights in curated dictionary - adding default ({default_core_weight})")
        pruned['weight'] = default_core_weight

    # Calculate SIF weights from corpus
    total_tf = max(1, sum(term_freq.values()))
    a = CONFIG['scoring']['sif_a']

    def sif_weight(t: str) -> float:
        """Calculate corpus-based SIF weight."""
        if not CONFIG['scoring']['use_sif']:
            return 1.0
        tf = term_freq.get(t, 1)
        return 1.0 / (a + tf / total_tf)

    def hybrid_weight(term: str, seed_weight: float) -> float:
        """Combine seed dictionary weight with corpus-based SIF weight."""
        sif_w = sif_weight(term)

        if weighting_scheme == "multiplicative":
            combined = seed_weight * sif_w
        elif weighting_scheme == "additive":
            seed_ratio = CONFIG['weights']['additive_seed_ratio']
            combined = (seed_ratio * seed_weight) + ((1.0 - seed_ratio) * sif_w)
        elif weighting_scheme == "seed_dominant":
            seed_ratio = CONFIG['weights']['additive_seed_ratio']
            combined = (seed_ratio * seed_weight) + ((1.0 - seed_ratio) * sif_w)
        elif weighting_scheme == "geometric":
            combined = np.sqrt(seed_weight * sif_w)
        else:
            combined = seed_weight * sif_w

        return combined

    # Build topic vectors with hybrid weights
    topic2vec = {}
    topic2terms = defaultdict(list)
    topic2sources = defaultdict(lambda: {'corpus': 0, 'seed': 0})  # Track term sources per topic

    print(f"\n{'='*60}")
    print("BUILDING TOPIC VECTORS WITH HYBRID WEIGHTS")
    print(f"{'='*60}")
    print(f"Weighting scheme: {weighting_scheme}")
    print(f"Default core weight: {default_core_weight}")
    print(f"Default discovered weight: {default_discovered_weight}")
    print(f"SIF parameter a = {a}")
    if weighting_scheme in ["additive", "seed_dominant"]:
        print(f"Seed ratio: {CONFIG['weights']['additive_seed_ratio']}")
    print()

    for topic in pruned['topic'].unique():
        topic_df = pruned[pruned['topic'] == topic]
        vecs = []
        ws = []

        for _, row in topic_df.iterrows():
            t = row['term']
            seed_w = row.get('weight', default_core_weight)
            if pd.isna(seed_w):
                seed_w = default_core_weight

            if t not in vocab2vec:
                continue

            v = vocab2vec[t]
            w = hybrid_weight(t, seed_w)

            vecs.append(v)
            ws.append(w)
            topic2terms[topic].append(t)

            # Track source
            source = term_source.get(t, 'corpus')
            topic2sources[topic][source] += 1

        if not vecs:
            continue

        # Weighted average of term vectors (already normalized, so result will be normalized)
        V = np.vstack(vecs)
        W = np.array(ws).reshape(-1, 1)
        tv = (V * W).sum(axis=0) / (W.sum() + 1e-12)
        tv = tv / (np.linalg.norm(tv) + 1e-12)  # Re-normalize after averaging
        topic2vec[topic] = tv

        # Show weight statistics with source breakdown
        avg_seed_w = topic_df['weight'].mean() if 'weight' in topic_df.columns else default_core_weight
        avg_combined_w = np.mean(ws)
        sources = topic2sources[topic]
        print(f"  {topic}:")
        print(f"    Terms: {len(topic2terms[topic])} ({sources['seed']} seed, {sources['corpus']} corpus)")
        print(f"    Avg seed weight: {avg_seed_w:.3f}")
        print(f"    Avg combined weight: {avg_combined_w:.3f}")

    print(f"\n✓ Created {len(topic2vec)} topic vectors with hybrid weighting")

    # Save using fs
    topic_vec_path = fs.folders['Other_data'] / 'topic_vectors.npy'
    np.save(topic_vec_path, topic2vec)
    print(f"✓ Saved topic vectors")

    meta = {
        "topics": list(topic2vec.keys()),
        "terms": dict(topic2terms),
        "term_sources": {topic: dict(sources) for topic, sources in topic2sources.items()}
    }
    topic_meta_path = fs.folders['Other_data'] / 'topic_vectors_meta.json'
    with open(topic_meta_path, 'w') as f:
        json.dump(meta, f, indent=2)
    print(f"✓ Saved topic metadata (including source tracking)")

    # Generate suggestions for each topic
    print(f"\n{'='*60}")
    print("GENERATING TOPIC-SPECIFIC SUGGESTIONS")
    print(f"{'='*60}")

    for topic in topic2vec.keys():
        print(f"  Generating suggestions for: {topic}")
        tv = topic2vec[topic]

        # Get similarity scores for all terms
        # Since both tv and term embeddings are normalized, cosine = dot product
        sims = []
        for term, te in vocab2vec.items():
            sim = float(np.dot(tv, te))  # Simplified: both are unit vectors
            source = term_source.get(term, 'corpus')
            sims.append((term, sim, source))

        # Sort by similarity
        sims.sort(key=lambda x: x[1], reverse=True)

        # Take top N suggestions
        n_suggestions = CONFIG.get('expand', {}).get('k_nearest', 100)
        top_sims = sims[:n_suggestions]

        # Create dataframe with source information
        out = pd.DataFrame(top_sims, columns=['term', 'similarity', 'source'])
        out.insert(0, 'topic', topic)
        out['keep'] = True
        topic_filename = f"{topic.replace(' ', '_').replace('&', 'and')}_suggestions.csv"
        topic_path = fs.folders['Dictionary_suggestions'] / topic_filename
        out.to_csv(topic_path, index=False, encoding='utf-8')

    print(f"✓ Saved per-topic suggestions to Dictionary_suggestions/")

    fs.save_config("checkpoint4_vectors")

print(f"\n{'='*60}")
print("CHECKPOINT 4 COMPLETE")
print(f"{'='*60}")



CHECKPOINT 4 START - LOADING REQUIRED DATA
✓ Loaded term frequencies
✓ Loaded vocab embeddings: (19058, 768)
✓ Loaded vocabulary metadata: 19058 terms
Creating vocab2vec mapping...
Normalizing vocabulary embeddings...
✓ Created normalized vocab2vec: 19058 terms
✓ Loaded seed embeddings: (123, 768)
✓ Loaded seed metadata: 123 terms
Normalizing seed embeddings...
✓ Created normalized seed vector lookup: 123 terms

✓ Merged vocab + seed vectors: 19181 unique terms
  Breakdown: 19058 corpus-only, 0 seed-override, 0 seed-only
✓ Embedding normalization verified (avg norm=1.0000)

BUILDING TOPIC VECTORS FROM CURATED DICTIONARY
✓ Loaded curated dictionary: 1040 terms, 4 topics
✓ Curated dictionary has seed weights column

BUILDING TOPIC VECTORS WITH HYBRID WEIGHTS
Weighting scheme: multiplicative
Default core weight: 1.0
Default discovered weight: 0.8
SIF parameter a = 0.001

  Colonial Systems:
    Terms: 296 (30 seed, 266 corpus)
    Avg seed weight: 0.784
    Avg combined weight: 745.592
 

✅ **CHECKPOINT 4 COMPLETE** - Topic vectors created

**Resume**: Load `topic2vec` from `Other_data/topic_vectors.npy`

---
# CHECKPOINT 5: Chunk Scoring & Confidence Classification
---

Score all chunks and classify by confidence level (High/Low/None).

In [34]:
# ============================================================
# CP5: WORKFLOW SOURCE OVERRIDE (Optional)
# ============================================================

# Load corpus and topic vectors from different workflows
CP5_CORPUS_SOURCE = None  # For chunked corpus
CP5_VECTORS_SOURCE = None  # For topic vectors (workflow root folder)
CP5_MODEL_SOURCE = None  # For SBERT model (workflow root folder or model path)

corpus_fs = fs.get_source_workflow(CP5_CORPUS_SOURCE) if CP5_CORPUS_SOURCE else fs
vectors_fs = fs.get_source_workflow(CP5_VECTORS_SOURCE) if CP5_VECTORS_SOURCE else fs

# ============================================================
# LOAD SBERT MODEL (needed for CP5 to run independently)
# ============================================================

# Determine model source (priority order)
model_source = None
source_description = ""

# Priority 1: Override (CP5_MODEL_SOURCE)
if CP5_MODEL_SOURCE is not None:
    model_source = CP5_MODEL_SOURCE
    source_description = "override (CP5_MODEL_SOURCE)"
# Priority 2: CONFIG pretrained path
elif CONFIG.get('model', {}).get('use_pretrained') and CONFIG.get('paths', {}).get('pretrained_model_path'):
    model_source = CONFIG['paths']['pretrained_model_path']
    source_description = "CONFIG (pretrained_model_path)"
# Priority 3: No custom model (use default or existing)
else:
    model_source = None
    source_description = "default base model"

# Load the model based on determined source
if model_source is not None:
    # Custom model path specified (from override OR config)
    model_path = Path(model_source)

    # Look for SBERT model in order of priority
    # Priority 1: trained_encoder/ (domain-adapted SBERT)
    trained_encoder_path = model_path / 'trained_encoder'
    if trained_encoder_path.exists():
        sbert_path = trained_encoder_path
    # Priority 2: sbert_base/ (backward compatibility)
    elif (model_path / 'sbert_base').exists():
        sbert_path = model_path / 'sbert_base'
    # Priority 3: Model path itself
    elif model_path.name == 'sbert_base' or model_path.name == 'trained_encoder':
        sbert_path = model_path
    else:
        sbert_path = model_path

    # Load the model
    if sbert_path.exists():
        st_model = SentenceTransformer(str(sbert_path))
        print(f"📂 Loading SBERT from {source_description}: {sbert_path}")
    else:
        # Path doesn't exist - fall back to default
        st_model = SentenceTransformer(CONFIG['model']['base_model_name'])
        print(f"⚠️  Path not found: {sbert_path}")
        print(f"📂 Loading SBERT from default: {CONFIG['model']['base_model_name']}")
else:
    # No custom model specified - use existing model in memory or load default
    if 'st_model' not in globals():
        st_model = SentenceTransformer(CONFIG['model']['base_model_name'])
        print(f"📂 Loading SBERT from default: {CONFIG['model']['base_model_name']}")
    else:
        print(f"📂 Using existing SBERT model in memory")

if CP5_CORPUS_SOURCE:
    print(f"📂 Loading corpus from: {corpus_fs.root}")
else:
    print(f"📂 Loading corpus from: current workflow")

if CP5_VECTORS_SOURCE:
    print(f"📂 Loading topic vectors from: {vectors_fs.root}")
else:
    print(f"📂 Loading topic vectors from: current workflow")

print(f"💾 Saving scores to: {fs.root}")


📂 Using existing SBERT model in memory
📂 Loading corpus from: current workflow
📂 Loading topic vectors from: current workflow
💾 Saving scores to: workflow_data\slavery_Short-slavdict_pretrained_slavery_v4


In [35]:
# ============================================================
# CELL 5.1: SCORE CHUNKS
# ============================================================

print(f"\n{'='*60}")
print("CHECKPOINT 5 START - LOADING REQUIRED DATA")
print(f"{'='*60}")

# Initialize/verify fs object
if 'fs' not in globals() or not hasattr(fs, 'folders') or not fs.folders:
    # Need to initialize or reload fs
    fs = WorkflowFileSystem(CONFIG)

    # Find and load the appropriate workflow
    workflow_base = Path(CONFIG["paths"]["workflow_base"])
    model_type = CONFIG['workflow']['workflow_name']
    topic = CONFIG['workflow']['target']

    if CONFIG['workflow']['version']:
        # Find specific version (pattern: model_type-topic_DATE_version)
        pattern = f"{model_type}-{topic}_*_{CONFIG['workflow']['version']}"
        matching_dirs = list(workflow_base.glob(pattern))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[0]
    else:
        # Find most recent workflow
        pattern = f"{model_type}-{topic}_*"
        matching_dirs = sorted(list(workflow_base.glob(pattern)))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[-1]

    fs.load_existing_workflow(workflow_dir)
    print(f"📂 Loaded workflow: {fs.root.name}")

# Load chunks DataFrame
chunks_path = corpus_fs.folders['Other_data'] / 'chunked_corpus.csv'
if not chunks_path.exists():
    raise FileNotFoundError(f"Chunked corpus not found at {chunks_path}")

chunks_df = pd.read_csv(chunks_path)
print(f"✓ Loaded chunks: {len(chunks_df)} chunks")

# Load topic vectors
topic_vec_path = vectors_fs.folders['Other_data'] / 'topic_vectors.npy'
topic_meta_path = vectors_fs.folders['Other_data'] / 'topic_vectors_meta.json'

if not topic_vec_path.exists() or not topic_meta_path.exists():
    raise FileNotFoundError(f"Topic vectors not found. Need both files in Other_data/")

topic2vec = np.load(topic_vec_path, allow_pickle=True).item()
with open(topic_meta_path, 'r') as f:
    topic_meta = json.load(f)

print(f"✓ Loaded topic vectors: {len(topic2vec)} topics")
print(f"  Topics: {', '.join(topic2vec.keys())}")

print(f"\n{'='*60}")
print("SCORING CHUNKS WITH DOT PRODUCT (UNNORMALIZED)")
print(f"{'='*60}")

def similarity_score(a: np.ndarray, b: np.ndarray) -> float:
    """
    Dot product similarity - magnitude matters when embeddings are unnormalized.

    Unlike cosine similarity, dot product preserves the magnitude information:
    - Higher magnitude embeddings = stronger semantic content
    - Weighted topic vectors naturally produce proportional scores
    - No artificial rescaling needed

    Returns:
        Dot product score (typically ranges from 0-200+ depending on embedding norms)
    """
    return float(np.dot(a, b))


# Score all chunks
records = []
for idx, chunk in tqdm(chunks_df.iterrows(), total=len(chunks_df), desc="Scoring"):
    text = chunk['text_for_scoring']

    # Base row fields common to both branchesF
    base_row = {
        'filename': chunk['file_path'],
        'chunk_id': chunk['chunk_uid'],
        'sentence_count': chunk['sentence_count'],
        'raw_text': chunk['raw_text'],
    }

    # Check if text is actually a valid string (not NaN, not empty, not a float)
    if isinstance(text, str) and text.strip():
        dv = st_embed([text])[0]
        row = {**base_row, 'text_for_scoring': text}
        # Store dot product scores
        for topic, tv in topic2vec.items():
            row[f'score_{topic}'] = similarity_score(dv, tv)
    else:
        # Handle invalid text (NaN, empty, or float values)
        row = {**base_row, 'text_for_scoring': ''}
        for topic in topic2vec.keys():
            row[f'score_{topic}'] = 0.0

    records.append(row)

all_scores_df = pd.DataFrame(records)
print(f"\n✓ Scored {len(all_scores_df)} chunks across {len(topic2vec)} topics")

# Calculate metrics
topic_cols = [col for col in all_scores_df.columns if col.startswith('score_')]
all_scores_df['max_score'] = all_scores_df[topic_cols].max(axis=1)
all_scores_df['primary_topic'] = all_scores_df[topic_cols].idxmax(axis=1).str.replace('score_', '')

topic_scores = all_scores_df[topic_cols].values
sorted_scores = np.sort(topic_scores, axis=1)[:, ::-1]
all_scores_df['margin_score'] = sorted_scores[:, 0] - sorted_scores[:, 1]

# Show score distribution
print(f"\n{'='*60}")
print("DOT PRODUCT SCORE DISTRIBUTION")
print(f"{'='*60}")
print(f"  Min:  {all_scores_df['max_score'].min():.4f}")
print(f"  Q25:  {all_scores_df['max_score'].quantile(0.25):.4f}")
print(f"  Med:  {all_scores_df['max_score'].median():.4f}")
print(f"  Q75:  {all_scores_df['max_score'].quantile(0.75):.4f}")
print(f"  Max:  {all_scores_df['max_score'].max():.4f}")
print(f"  Std:  {all_scores_df['max_score'].std():.4f}")
print(f"  Range: {all_scores_df['max_score'].max() - all_scores_df['max_score'].min():.4f}")

# Show margin distribution
print(f"\n{'='*60}")
print("SCORE MARGIN DISTRIBUTION (confidence metric)")
print(f"{'='*60}")
print(f"  Min:  {all_scores_df['margin_score'].min():.4f}")
print(f"  Q25:  {all_scores_df['margin_score'].quantile(0.25):.4f}")
print(f"  Med:  {all_scores_df['margin_score'].median():.4f}")
print(f"  Q75:  {all_scores_df['margin_score'].quantile(0.75):.4f}")
print(f"  Max:  {all_scores_df['margin_score'].max():.4f}")

# Show topic-wise score statistics
print(f"\n{'='*60}")
print("SCORE DISTRIBUTION BY TOPIC")
print(f"{'='*60}")
for topic_col in topic_cols:
    topic_name = topic_col.replace('score_', '')
    scores = all_scores_df[topic_col]
    print(f"\n{topic_name}:")
    print(f"  Mean: {scores.mean():.2f}  Std: {scores.std():.2f}")
    print(f"  Min: {scores.min():.2f}   Max: {scores.max():.2f}")
    print(f"  Q25: {scores.quantile(0.25):.2f}   Q75: {scores.quantile(0.75):.2f}")

print(f"\n{'='*60}")
print("✓ Scoring complete - using natural dot product scores")
print(f"✓ No artificial rescaling applied")
print(f"✓ Scores reflect weighted topic vector magnitudes")
print(f"{'='*60}")



CHECKPOINT 5 START - LOADING REQUIRED DATA
✓ Loaded chunks: 2840 chunks
✓ Loaded topic vectors: 4 topics
  Topics: Colonial Systems, Heritage & Memory, Historical Slavery, Modern Racism & Discrimination

SCORING CHUNKS WITH DOT PRODUCT (UNNORMALIZED)


Scoring: 100%|██████████| 2840/2840 [00:34<00:00, 81.19it/s]


✓ Scored 2840 chunks across 4 topics

DOT PRODUCT SCORE DISTRIBUTION
  Min:  1.8379
  Q25:  4.5429
  Med:  5.2793
  Q75:  6.0429
  Max:  9.0057
  Std:  1.1038
  Range: 7.1678

SCORE MARGIN DISTRIBUTION (confidence metric)
  Min:  0.0001
  Q25:  0.2256
  Med:  0.5020
  Q75:  0.9765
  Max:  3.5908

SCORE DISTRIBUTION BY TOPIC

Colonial Systems:
  Mean: 4.38  Std: 1.10
  Min: 1.05   Max: 8.49
  Q25: 3.62   Q75: 5.10

Heritage & Memory:
  Mean: 4.39  Std: 1.01
  Min: 0.99   Max: 9.01
  Q25: 3.71   Q75: 5.01

Historical Slavery:
  Mean: 4.65  Std: 1.31
  Min: 0.66   Max: 8.80
  Q25: 3.69   Q75: 5.59

Modern Racism & Discrimination:
  Mean: 4.52  Std: 1.08
  Min: 0.88   Max: 8.78
  Q25: 3.78   Q75: 5.22

✓ Scoring complete - using natural dot product scores
✓ No artificial rescaling applied
✓ Scores reflect weighted topic vector magnitudes


In [36]:
# ============================================================
# CELL 5.2: CORPUS-ADAPTIVE SIGNIFICANCE CLASSIFICATION
# ============================================================
# V22 Enhancement: Adapts thresholds based on corpus characteristics
# Policy documents have lower score ranges than historical texts

print(f"\n{'='*80}")
print("CORPUS-ADAPTIVE SIGNIFICANCE SCORING")
print(f"{'='*80}")

import numpy as np

# Get topic score columns
topic_cols = [col for col in all_scores_df.columns if col.startswith('score_')]

print(f"\nTopic columns: {topic_cols}")
print(f"Total chunks: {len(all_scores_df)}")

# ============================================================
# DETECT CORPUS TYPE VIA SCORE RANGE ANALYSIS
# ============================================================

print(f"\n{'='*80}")
print("CORPUS TYPE DETECTION")
print(f"{'='*80}")

# Analyze score distribution to determine corpus type
all_scores_flat = []
for col in topic_cols:
    all_scores_flat.extend(all_scores_df[col].values)

corpus_score_min = np.min(all_scores_flat)
corpus_score_max = np.max(all_scores_flat)
corpus_score_range = corpus_score_max - corpus_score_min
corpus_score_median = np.median(all_scores_flat)
corpus_score_mean = np.mean(all_scores_flat)

print(f"\nCorpus Score Statistics:")
print(f"  Min:    {corpus_score_min:.3f}")
print(f"  Max:    {corpus_score_max:.3f}")
print(f"  Range:  {corpus_score_range:.3f}")
print(f"  Median: {corpus_score_median:.3f}")
print(f"  Mean:   {corpus_score_mean:.3f}")

# Corpus type heuristic: Policy docs typically have scores < 5.0
# Historical/narrative texts have higher semantic density (scores > 10)
is_policy_corpus = corpus_score_max < 5.0
corpus_type = "POLICY" if is_policy_corpus else "HISTORICAL/NARRATIVE"

print(f"\n→ Detected corpus type: {corpus_type}")
print(f"  (Based on max_score {'<' if is_policy_corpus else '>='} 5.0)")

# Set adaptive parameters based on corpus type
if is_policy_corpus:
    # Policy documents: Lower semantic density, need relaxed thresholds
    cv_upper_bound = 0.55      # More lenient CV normalization (vs 0.20 for historical)
    weak_signal_threshold = 0.20  # 20% for policy (vs 10% for historical)
    z_score_min = 0.4          # Lower z-score expectations
    z_score_max = 1.3
    magnitude_min = 1.5        # Lower magnitude floor
    magnitude_max = 4.5        # Lower magnitude ceiling
    cv_noise_threshold = 0.08  # Slightly lower CV noise cutoff

    param_context = "policy documents (lower semantic density)"
else:
    # Historical/narrative: Higher semantic density, stricter thresholds
    cv_upper_bound = 0.20      # Stricter CV normalization
    weak_signal_threshold = 0.10  # 10% for historical
    z_score_min = 0.6
    z_score_max = 1.7
    magnitude_min = 2.0
    magnitude_max = 9.0
    cv_noise_threshold = 0.10

    param_context = "historical/narrative texts (higher semantic density)"

print(f"\nAdaptive Parameters ({param_context}):")
print(f"  CV upper bound:        {cv_upper_bound}")
print(f"  Weak signal threshold: {weak_signal_threshold}")
print(f"  Z-score range:         {z_score_min} - {z_score_max}")
print(f"  Magnitude range:       {magnitude_min} - {magnitude_max}")
print(f"  CV noise threshold:    {cv_noise_threshold}")

# ============================================================
# CALCULATE CORPUS-ADAPTIVE SIGNIFICANCE METRICS
# ============================================================

def calculate_significance_adaptive(row, topic_cols, params):
    """
    V22 Enhancement: Corpus-adaptive significance scoring.

    Adjusts normalization ranges and thresholds based on detected corpus type:
    - Policy docs: Lower scores, need relaxed CV bounds and thresholds
    - Historical: Higher scores, can use stricter thresholds

    Key changes from v21:
    1. CV upper bound: 0.55 (policy) vs 0.20 (historical)
    2. Weak signal threshold: 20% (policy) vs 10% (historical)
    3. Z-score normalization adapted to corpus range
    4. Component weights: 0.60 CV / 0.25 magnitude / 0.15 contrast (from 0.50/0.30/0.20)

    Returns value 0-1 where:
    - 1.0 = Highly significant (clear topic, good differentiation)
    - 0.0 = Noise (uniform scores, poor differentiation)
    """
    scores = [row[col] for col in topic_cols]

    max_score = max(scores)
    min_score = min(scores)
    mean_score = np.mean(scores)
    std_score = np.std(scores)

    # Component 1: Magnitude (normalized to corpus-specific range)
    magnitude = (max_score - params['magnitude_min']) / (params['magnitude_max'] - params['magnitude_min'])
    magnitude = np.clip(magnitude, 0, 1)

    # Component 2: Differentiation (CV) - PRIMARY FILTER
    # CV = coefficient of variation = std/mean
    cv = std_score / mean_score if mean_score > 0 else 0

    # Normalize CV using corpus-adaptive upper bound
    differentiation = cv / params['cv_upper_bound']
    differentiation = np.clip(differentiation, 0, 1)

    # Component 3: Contrast (Z-score) - How much max stands out
    z_max = (max_score - mean_score) / std_score if std_score > 0 else 0

    # Normalize z-score using corpus-adaptive range
    contrast = (z_max - params['z_score_min']) / (params['z_score_max'] - params['z_score_min'])
    contrast = np.clip(contrast, 0, 1)

    # V22 ENHANCEMENT: Weighted combination with CV-dominant weighting
    # Increased CV weight from 0.50 → 0.60
    # Reduced magnitude from 0.30 → 0.25
    # Reduced contrast from 0.20 → 0.15
    # Rationale: CV (differentiation) is the most reliable noise filter
    significance = (
        0.60 * differentiation +   # PRIMARY: How well chunk differentiates topics
        0.25 * magnitude +         # SECONDARY: Raw strength of signal
        0.15 * contrast            # TERTIARY: How much winner stands out
    )

    # Categorize using corpus-adaptive CV threshold
    if cv < params['cv_noise_threshold']:
        category = 'noise_uniform_scores'
        priority = 'exclude'
    elif max_score / (mean_score + 1e-12) < (1.0 + params['weak_signal_threshold']):
        # Weak signal: max score barely above mean
        category = 'noise_weak_signal'
        priority = 'exclude'
    elif significance >= params.get('high_threshold', 0.80):
        category = 'high_significance'
        priority = 'primary_training'
    elif significance >= params.get('medium_threshold', 0.50):
        category = 'medium_significance'
        priority = 'secondary_training'
    elif significance >= params.get('low_threshold', 0.30):
        category = 'low_significance'
        priority = 'manual_review'
    else:
        category = 'noise_weak_signal'
        priority = 'exclude'

    return {
        'significance_score': significance,
        'significance_category': category,
        'priority': priority,
        'cv': cv,
        'z_max': z_max,
        'magnitude_norm': magnitude,
        'differentiation_norm': differentiation,
        'contrast_norm': contrast,
        'max_score': max_score,
        'mean_score': mean_score,
        'std_score': std_score
    }

# Package parameters for function
adaptive_params = {
    'cv_upper_bound': cv_upper_bound,
    'weak_signal_threshold': weak_signal_threshold,
    'z_score_min': z_score_min,
    'z_score_max': z_score_max,
    'magnitude_min': magnitude_min,
    'magnitude_max': magnitude_max,
    'cv_noise_threshold': cv_noise_threshold,
    'high_threshold': 0.65 if is_policy_corpus else 0.82,
    'medium_threshold': 0.55 if is_policy_corpus else 0.75,
    'low_threshold': 0.45 if is_policy_corpus else 0.68,
}

print("\nCalculating corpus-adaptive significance scores...")

# Apply to all chunks
from tqdm import tqdm

significance_results = []
for idx, row in tqdm(all_scores_df.iterrows(), total=len(all_scores_df), desc="Calculating significance"):
    sig = calculate_significance_adaptive(row, topic_cols, adaptive_params)
    significance_results.append(sig)

# Add to dataframe
for key in significance_results[0].keys():
    all_scores_df[key] = [r[key] for r in significance_results]

# ============================================================
# SIGNIFICANCE SCORE STATISTICS
# ============================================================

print(f"\n{'='*80}")
print("SIGNIFICANCE SCORE DISTRIBUTION")
print(f"{'='*80}")
print(f"  Min:    {all_scores_df['significance_score'].min():.3f}")
print(f"  Q25:    {all_scores_df['significance_score'].quantile(0.25):.3f}")
print(f"  Median: {all_scores_df['significance_score'].median():.3f}")
print(f"  Q75:    {all_scores_df['significance_score'].quantile(0.75):.3f}")
print(f"  Max:    {all_scores_df['significance_score'].max():.3f}")

print(f"\n{'='*80}")
print("COEFFICIENT OF VARIATION (CV) DISTRIBUTION")
print(f"{'='*80}")
print(f"  Min:    {all_scores_df['cv'].min():.4f}")
print(f"  Q10:    {all_scores_df['cv'].quantile(0.10):.4f}")
print(f"  Q25:    {all_scores_df['cv'].quantile(0.25):.4f}")
print(f"  Median: {all_scores_df['cv'].median():.4f}")
print(f"  Q75:    {all_scores_df['cv'].quantile(0.75):.4f}")
print(f"  Q90:    {all_scores_df['cv'].quantile(0.90):.4f}")
print(f"  Max:    {all_scores_df['cv'].max():.4f}")

print(f"\n{'='*80}")
print("SIGNIFICANCE CATEGORY DISTRIBUTION")
print(f"{'='*80}")
category_counts = all_scores_df['significance_category'].value_counts()
total_chunks = len(all_scores_df)
for category in ['high_significance', 'medium_significance', 'low_significance', 'noise_uniform_scores', 'noise_weak_signal']:
    count = category_counts.get(category, 0)
    pct = count / total_chunks * 100
    print(f"  {category:25s}: {count:5d} ({pct:5.1f}%)")

print(f"\n{'='*80}")
print("PRIORITY FOR TRAINING")
print(f"{'='*80}")
priority_counts = all_scores_df['priority'].value_counts()
for priority in ['primary_training', 'secondary_training', 'manual_review', 'exclude']:
    count = priority_counts.get(priority, 0)
    pct = count / total_chunks * 100
    print(f"  {priority:20s}: {count:5d} ({pct:5.1f}%)")

# ============================================================
# EXAMPLES BY SIGNIFICANCE LEVEL
# ============================================================

print(f"\n{'='*80}")
print("EXAMPLES: HIGH SIGNIFICANCE (Good for training)")
print(f"{'='*80}")

high_sig = all_scores_df[all_scores_df['significance_score'] >= 0.55].head(3)
for idx, row in high_sig.iterrows():
    scores = [row[col] for col in topic_cols]
    print(f"\nChunk {idx}:")
    print(f"  Significance: {row['significance_score']:.3f}")
    print(f"  CV: {row['cv']:.3f}, Z-max: {row['z_max']:.3f}")
    print(f"  Scores: {[f'{s:.2f}' for s in scores]}")
    print(f"  Primary topic: {row['primary_topic']}")
    print(f"  Text: {row['raw_text'][:450]}...")

print(f"\n{'='*80}")
print("EXAMPLES: NOISE (Uniform scores - filtered out)")
print(f"{'='*80}")

noise = all_scores_df[all_scores_df['cv'] < cv_noise_threshold].head(3)
for idx, row in noise.iterrows():
    scores = [row[col] for col in topic_cols]
    print(f"\nChunk {idx}:")
    print(f"  Significance: {row['significance_score']:.3f}")
    print(f"  CV: {row['cv']:.3f} (very low - all scores similar!)")
    print(f"  Scores: {[f'{s:.2f}' for s in scores]} (uniform pattern)")
    print(f"  Text: {row['raw_text'][:450]}...")

print(f"\n{'='*80}")
print("EXAMPLES: MEDIUM SIGNIFICANCE (Review recommended)")
print(f"{'='*80}")

medium_sig = all_scores_df[
    (all_scores_df['significance_score'] >= 0.35) &
    (all_scores_df['significance_score'] < 0.55)
].head(3)
for idx, row in medium_sig.iterrows():
    scores = [row[col] for col in topic_cols]
    print(f"\nChunk {idx}:")
    print(f"  Significance: {row['significance_score']:.3f}")
    print(f"  CV: {row['cv']:.3f}, Z-max: {row['z_max']:.3f}")
    print(f"  Scores: {[f'{s:.2f}' for s in scores]}")
    print(f"  Primary topic: {row['primary_topic']}")
    print(f"  Text: {row['raw_text'][:450]}...")

# ============================================================
# SAVE SIGNIFICANCE-CLASSIFIED FILES
# ============================================================

print(f"\n{'='*80}")
print("SAVING SIGNIFICANCE-CLASSIFIED FILES")
print(f"{'='*80}")

# Split by significance
high_sig_df = all_scores_df[all_scores_df['priority'] == 'primary_training'].copy()
medium_sig_df = all_scores_df[all_scores_df['priority'] == 'secondary_training'].copy()
review_df = all_scores_df[all_scores_df['priority'] == 'manual_review'].copy()
exclude_df = all_scores_df[all_scores_df['priority'] == 'exclude'].copy()

# Save 4-tier files
high_sig_path = fs.folders['Cosine_labeling'] / 'scores_high_significance.csv'
medium_sig_path = fs.folders['Cosine_labeling'] / 'scores_medium_significance.csv'
review_path = fs.folders['Cosine_labeling'] / 'scores_needs_review.csv'
exclude_path = fs.folders['Cosine_labeling'] / 'scores_exclude_noise.csv'

high_sig_df.to_csv(high_sig_path, index=False)
medium_sig_df.to_csv(medium_sig_path, index=False)
review_df.to_csv(review_path, index=False)
exclude_df.to_csv(exclude_path, index=False)

print(f"\nSaved 4-tier significance files:")
print(f"  High (primary training):   {high_sig_path} ({len(high_sig_df)} chunks)")
print(f"  Medium (secondary):        {medium_sig_path} ({len(medium_sig_df)} chunks)")
print(f"  Low (needs review):        {review_path} ({len(review_df)} chunks)")
print(f"  Noise (exclude):           {exclude_path} ({len(exclude_df)} chunks)")

# Save all scores with significance metrics
all_scores_path = fs.folders['Cosine_labeling'] / 'scores_all_labeled.csv'
all_scores_df.to_csv(all_scores_path, index=False)
print(f"  All (with significance):   {all_scores_path} ({len(all_scores_df)} chunks)")

# ============================================================
# BACKWARD COMPATIBILITY: 3-TIER CONFIDENCE FILES
# ============================================================

print(f"\n{'='*80}")
print("CREATING BACKWARD-COMPATIBLE CONFIDENCE FILES")
print(f"{'='*80}")

# Map significance to old confidence system for backward compatibility
# High confidence = High significance
# Low confidence = Medium significance + Low significance
# No confidence = Noise (both types)

all_scores_df['confidence'] = all_scores_df['significance_category'].map({
    'high_significance': 'high',
    'medium_significance': 'low',
    'low_significance': 'low',
    'noise_uniform_scores': 'none',
    'noise_weak_signal': 'none'
})

high_conf_df = all_scores_df[all_scores_df['confidence'] == 'high'].copy()
low_conf_df = all_scores_df[all_scores_df['confidence'] == 'low'].copy()
no_conf_df = all_scores_df[all_scores_df['confidence'] == 'none'].copy()

print(f"\n3-tier mapping (for backward compatibility):")
print(f"  High confidence:   {len(high_conf_df):5d} ({len(high_conf_df)/total_chunks*100:5.1f}%)")
print(f"  Low confidence:    {len(low_conf_df):5d} ({len(low_conf_df)/total_chunks*100:5.1f}%)")
print(f"  No confidence:     {len(no_conf_df):5d} ({len(no_conf_df)/total_chunks*100:5.1f}%)")

high_conf_path = fs.folders['Cosine_labeling'] / 'scores_high_confidence.csv'
low_conf_path = fs.folders['Cosine_labeling'] / 'scores_low_confidence.csv'
no_conf_path = fs.folders['Cosine_labeling'] / 'scores_no_confidence.csv'

high_conf_df.to_csv(high_conf_path, index=False)
low_conf_df.to_csv(low_conf_path, index=False)
no_conf_df.to_csv(no_conf_path, index=False)

print(f"\nSaved 3-tier compatibility files:")
print(f"  High: {high_conf_path}")
print(f"  Low:  {low_conf_path}")
print(f"  None: {no_conf_path}")

# ============================================================
# CORPUS-ADAPTIVE PARAMETERS SUMMARY
# ============================================================

print(f"\n{'='*80}")
print("CORPUS-ADAPTIVE SCORING SUMMARY")
print(f"{'='*80}")

print(f"\nDetected Corpus Type: {corpus_type}")
print(f"  Score range: {corpus_score_min:.2f} - {corpus_score_max:.2f}")

print(f"\nParameters Used:")
print(f"  CV noise threshold:    {cv_noise_threshold:.2f}")
print(f"  CV normalization max:  {cv_upper_bound:.2f}")
print(f"  Weak signal threshold: {weak_signal_threshold:.0%}")
print(f"  Z-score range:         {z_score_min:.1f} - {z_score_max:.1f}")
print(f"  Magnitude range:       {magnitude_min:.1f} - {magnitude_max:.1f}")

print(f"\nComponent Weights (v22 enhanced):")
print(f"  Differentiation (CV):  0.60  (increased from 0.50)")
print(f"  Magnitude:             0.25  (decreased from 0.30)")
print(f"  Contrast (Z-score):    0.15  (decreased from 0.20)")
print(f"  → CV is now dominant factor in noise filtering")

print(f"\nSignificance Thresholds (adapted for {corpus_type.lower()}):")
print(f"  High (primary training):   >= 0.55  (was 0.70 in v21)")
print(f"  Medium (secondary):        >= 0.35  (was 0.50 in v21)")
print(f"  Low (manual review):       >= 0.15  (was 0.30 in v21)")
print(f"  Noise (exclude):           <  0.15")
print(f"  → Lowered thresholds capture more policy-relevant chunks")

high_count = len(high_sig_df)
medium_count = len(medium_sig_df)
review_count = len(review_df)
exclude_count = len(exclude_df)

print(f"\n{'='*80}")
print("RECOMMENDED USE BY SIGNIFICANCE LEVEL")
print(f"{'='*80}")

print(f"\n  HIGH SIGNIFICANCE ({high_count} chunks, {high_count/total_chunks*100:.1f}%):")
print("    → Use for: Primary training data, model fine-tuning")
print(f"    → Characteristics: Clear topic signal, CV > {cv_noise_threshold}")
print("    → Significance: >= 0.55")

print(f"\n  MEDIUM SIGNIFICANCE ({medium_count} chunks, {medium_count/total_chunks*100:.1f}%):")
print("    → Use for: Secondary training data, augmentation")
print("    → Characteristics: Moderate signal or partial differentiation")
print("    → Significance: 0.35 - 0.55")
print("    → Recommend: Batch review before use")

print(f"\n  LOW SIGNIFICANCE ({review_count} chunks, {review_count/total_chunks*100:.1f}%):")
print("    → Use for: Edge cases, manual curation")
print("    → Characteristics: Weak signal or poor differentiation")
print("    → Significance: 0.15 - 0.35")
print("    → Recommend: Individual review needed")

print(f"\n  NOISE ({exclude_count} chunks, {exclude_count/total_chunks*100:.1f}%):")
print("    → Use for: Negative examples (optional)")
print(f"    → Characteristics: CV < {cv_noise_threshold} or weak signal")
print("    → Significance: < 0.15")
print("    → Recommend: Exclude from training")

high_quality = high_count + medium_count
should_filter = exclude_count

print(f"\n✓ High-quality data (High + Medium): {high_quality} chunks ({high_quality/total_chunks*100:.1f}%)")
print(f"✓ Should filter (Noise): {should_filter} chunks ({should_filter/total_chunks*100:.1f}%)")
print(f"✓ Needs review: {review_count} chunks ({review_count/total_chunks*100:.1f}%)")

fs.save_config("checkpoint5_scoring")

print(f"\n{'='*80}")
print("✓ CHECKPOINT 5 COMPLETE - Corpus-adaptive significance scoring done")
print(f"{'='*80}")



CORPUS-ADAPTIVE SIGNIFICANCE SCORING

Topic columns: ['score_Colonial Systems', 'score_Heritage & Memory', 'score_Historical Slavery', 'score_Modern Racism & Discrimination']
Total chunks: 2840

CORPUS TYPE DETECTION

Corpus Score Statistics:
  Min:    0.664
  Max:    9.006
  Range:  8.341
  Median: 4.428
  Mean:   4.488

→ Detected corpus type: HISTORICAL/NARRATIVE
  (Based on max_score >= 5.0)

Adaptive Parameters (historical/narrative texts (higher semantic density)):
  CV upper bound:        0.2
  Weak signal threshold: 0.1
  Z-score range:         0.6 - 1.7
  Magnitude range:       2.0 - 9.0
  CV noise threshold:    0.1

Calculating corpus-adaptive significance scores...


Calculating significance: 100%|██████████| 2840/2840 [00:00<00:00, 9030.64it/s]



SIGNIFICANCE SCORE DISTRIBUTION
  Min:    0.132
  Q25:    0.474
  Median: 0.597
  Q75:    0.751
  Max:    0.986

COEFFICIENT OF VARIATION (CV) DISTRIBUTION
  Min:    0.0054
  Q10:    0.0602
  Q25:    0.0879
  Median: 0.1270
  Q75:    0.1720
  Q90:    0.2223
  Max:    0.5179

SIGNIFICANCE CATEGORY DISTRIBUTION
  high_significance        :   411 ( 14.5%)
  medium_significance      :   305 ( 10.7%)
  low_significance         :   325 ( 11.4%)
  noise_uniform_scores     :   922 ( 32.5%)
  noise_weak_signal        :   877 ( 30.9%)

PRIORITY FOR TRAINING
  primary_training    :   411 ( 14.5%)
  secondary_training  :   305 ( 10.7%)
  manual_review       :   325 ( 11.4%)
  exclude             :  1799 ( 63.3%)

EXAMPLES: HIGH SIGNIFICANCE (Good for training)

Chunk 2:
  Significance: 0.557
  CV: 0.077, Z-max: 1.653
  Scores: ['6.26', '6.14', '7.16', '5.86']
  Primary topic: Historical Slavery
  Text: Staat en slavernij kan zo hopelijk bijdragen aan inzicht en bewustwording,
en misschien ook aan

✅ **CHECKPOINT 5 COMPLETE** - Chunks scored and classified

**Resume**: Load confidence CSVs from `Cosine_labeling/`

---
# CHECKPOINT 6: Training Data Preparation
---

Create train/val splits from confidence tiers.

In [37]:
# ============================================================
# CP6: WORKFLOW SOURCE OVERRIDE (Optional)
# ============================================================

# Load cosine scores from different workflow if needed
CP6_SOURCE = None  # e.g., "workflow_data/pretrained-Slavery_11.10.25_v2"

source_fs = fs.get_source_workflow(CP6_SOURCE) if CP6_SOURCE else fs

if CP6_SOURCE:
    print(f"📂 Loading cosine scores from: {source_fs.root}")
else:
    print(f"📂 Loading cosine scores from: current workflow")
print(f"💾 Saving training data to: {fs.root}")


📂 Loading cosine scores from: current workflow
💾 Saving training data to: workflow_data\slavery_Short-slavdict_pretrained_slavery_v4


In [38]:
# ============================================================
# CELL 6: LOAD LABELED SCORES
# ============================================================

print(f"\n{'='*60}")
print("CHECKPOINT 6 START - LOADING LABELED SCORE FILES")
print(f"{'='*60}")

# Initialize/verify fs object
if 'fs' not in globals() or not hasattr(fs, 'folders') or not fs.folders:
    # Need to initialize or reload fs
    fs = WorkflowFileSystem(CONFIG)
    
    # Find and load the appropriate workflow
    workflow_base = Path(CONFIG["paths"]["workflow_base"])
    model_type = CONFIG['workflow']['workflow_name']
    topic = CONFIG['workflow']['target']
    
    if CONFIG['workflow']['version']:
        # Find specific version (pattern: model_type-topic_DATE_version)
        pattern = f"{model_type}-{topic}_*_{CONFIG['workflow']['version']}"
        matching_dirs = list(workflow_base.glob(pattern))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[0]
    else:
        # Find most recent workflow
        pattern = f"{model_type}-{topic}_*"
        matching_dirs = sorted(list(workflow_base.glob(pattern)))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[-1]
    
    fs.load_existing_workflow(workflow_dir)
    print(f"📂 Loaded workflow: {fs.root.name}")


Scores_all_labeled_path = source_fs.folders['Cosine_labeling'] / 'scores_all_labeled.csv'
if not Scores_all_labeled_path.exists():
    raise FileNotFoundError(f"Labeled scores file not found at {Scores_all_labeled_path}")

Scores_all_labeled = pd.read_csv(Scores_all_labeled_path)
Scores_all_labeled['text'] = Scores_all_labeled['text_for_scoring']
Scores_all_labeled['label'] = Scores_all_labeled['primary_topic']

# Create label mapping
label2id = {label: idx for idx, label in enumerate(sorted(Scores_all_labeled['label'].unique()))}
id2label = {idx: label for label, idx in label2id.items()}
Scores_all_labeled['label_id'] = Scores_all_labeled['label'].map(label2id)
Scores_all_labeled['is_pseudo'] = False

print(f"\nLabel mapping:")
for label, idx in label2id.items():
    count = (Scores_all_labeled['label'] == label).sum()
    print(f"  {idx}: {label} ({count} examples)")

print(f"\nTotal labeled examples: {len(Scores_all_labeled)}")

relevance_threshold = 5

# Create table for Option 4 (All data - RECOMMENDED)
print(f"\n")
dist_data = []
for topic in topics:
    score_col = f'score_{topic}'

    # Train stats
    all_count = (Scores_all_labeled[score_col] >= relevance_threshold).sum()
    all_mean = Scores_all_labeled[score_col].mean(),
    all_std = Scores_all_labeled[score_col].std(),


    dist_data.append({
        'Topic': topic,
        'Total': all_count,
        'Mean': f"{all_mean[0]:.2f}",
        'Std': f"{all_std[0]:.2f}"
    })

dist_table = pd.DataFrame(dist_data)
print("\n" + dist_table.to_string(index=False))
print("="*80)
print(f"Note: 'High' count shows chunks with score >= {relevance_threshold:.2f} (P75 threshold)")
print(f"      Mean/Std show distribution of all scores for that topic")
print("="*80)





CHECKPOINT 6 START - LOADING LABELED SCORE FILES

Label mapping:
  0: Colonial Systems (564 examples)
  1: Heritage & Memory (526 examples)
  2: Historical Slavery (879 examples)
  3: Modern Racism & Discrimination (871 examples)

Total labeled examples: 2840



                         Topic  Total Mean  Std
              Colonial Systems    791 4.38 1.10
             Heritage & Memory    722 4.39 1.01
            Historical Slavery   1089 4.65 1.31
Modern Racism & Discrimination    897 4.52 1.08
Note: 'High' count shows chunks with score >= 5.00 (P75 threshold)
      Mean/Std show distribution of all scores for that topic


In [39]:
# ============================================================
# CELL 6: LOAD LABELED SCORES
# ============================================================

print(f"\n{'='*60}")
print("CHECKPOINT 6 START - LOADING LABELED SCORE FILES")
print(f"{'='*60}")

# Initialize/verify fs object
if 'fs' not in globals() or not hasattr(fs, 'folders') or not fs.folders:
    # Need to initialize or reload fs
    fs = WorkflowFileSystem(CONFIG)
    
    # Find and load the appropriate workflow
    workflow_base = Path(CONFIG["paths"]["workflow_base"])
    model_type = CONFIG['workflow']['workflow_name']
    topic = CONFIG['workflow']['target']
    
    if CONFIG['workflow']['version']:
        # Find specific version (pattern: model_type-topic_DATE_version)
        pattern = f"{model_type}-{topic}_*_{CONFIG['workflow']['version']}"
        matching_dirs = list(workflow_base.glob(pattern))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[0]
    else:
        # Find most recent workflow
        pattern = f"{model_type}-{topic}_*"
        matching_dirs = sorted(list(workflow_base.glob(pattern)))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[-1]
    
    fs.load_existing_workflow(workflow_dir)
    print(f"📂 Loaded workflow: {fs.root.name}")





    

# Load the three CSV files using fs.folders
high_path = source_fs.folders['Cosine_labeling'] / 'scores_high_confidence.csv'
low_path = source_fs.folders['Cosine_labeling'] / 'scores_low_confidence.csv'
no_path = source_fs.folders['Cosine_labeling'] / 'scores_no_confidence.csv'

if not high_path.exists() or not low_path.exists() or not no_path.exists():
    print(f"❌ Error: One or more score files not found")
    print(f"   Looking in: {fs.folders['Cosine_labeling']}")
    print(f"   - {high_path.name} {'✓' if high_path.exists() else '✗'}")
    print(f"   - {low_path.name} {'✓' if low_path.exists() else '✗'}")
    print(f"   - {no_path.name} {'✓' if no_path.exists() else '✗'}")
    raise FileNotFoundError("Required score files not found. Please run CHECKPOINT 5 first.")
else:
    high_df = pd.read_csv(high_path)
    low_df = pd.read_csv(low_path)
    no_df = pd.read_csv(no_path)
    
    print(f"✓ Loaded score files from: {fs.folders['Cosine_labeling']}")
    print(f"  High confidence: {len(high_df)} chunks")
    print(f"  Low confidence:  {len(low_df)} chunks")
    print(f"  No confidence:   {len(no_df)} chunks")
    print(f"  Total:           {len(high_df) + len(low_df) + len(no_df)} chunks")


CHECKPOINT 6 START - LOADING LABELED SCORE FILES
✓ Loaded score files from: workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Cosine_labeling
  High confidence: 411 chunks
  Low confidence:  630 chunks
  No confidence:   1799 chunks
  Total:           2840 chunks


In [40]:
# =====================
# STEP 2: SPLIT EACH OPTION INTO TRAIN/VAL
# Combine confidence levels into single dataset with labels
data_opt4 = pd.concat([
    high_df.assign(label='HIGH'),
    low_df.assign(label='LOW'),
    no_df.assign(label='UNLABELED')  # No confidence chunks treated as unlabeled
], ignore_index=True)

print(f"\nOption 4 (All confidence levels combined):")
print(f"  HIGH confidence: {len(high_df)} chunks")
print(f"  LOW confidence:  {len(low_df)} chunks")
print(f"  NO confidence:   {len(no_df)} chunks (treated as UNLABELED)")
print(f"  Total: {len(data_opt4)} chunks")

# CP6 ENHANCEMENT: Show label distribution per confidence tier
print(f"\n{'='*60}")
print("LABEL DISTRIBUTION BY CONFIDENCE TIER")
print(f"{'='*60}")
print("\nAfter CP5.2 corpus-adaptive thresholds:")

# Get all topic columns
topic_cols = [col for col in high_df.columns if col.startswith('score_')]
topic_names = [col.replace('score_', '') for col in topic_cols]

print(f"\nHIGH Confidence ({len(high_df)} chunks):")
if 'primary_topic' in high_df.columns:
    high_topic_dist = high_df['primary_topic'].value_counts()
    for topic in topic_names:
        count = high_topic_dist.get(topic, 0)
        pct = count / len(high_df) * 100 if len(high_df) > 0 else 0
        print(f"  {topic:45s}: {count:5d} ({pct:5.1f}%)")
else:
    print("  (primary_topic column not found)")

print(f"\nLOW Confidence ({len(low_df)} chunks):")
if 'primary_topic' in low_df.columns:
    low_topic_dist = low_df['primary_topic'].value_counts()
    for topic in topic_names:
        count = low_topic_dist.get(topic, 0)
        pct = count / len(low_df) * 100 if len(low_df) > 0 else 0
        print(f"  {topic:45s}: {count:5d} ({pct:5.1f}%)")
else:
    print("  (primary_topic column not found)")

print(f"\nNO Confidence ({len(no_df)} chunks):")
if 'primary_topic' in no_df.columns:
    no_topic_dist = no_df['primary_topic'].value_counts()
    for topic in topic_names:
        count = no_topic_dist.get(topic, 0)
        pct = count / len(no_df) * 100 if len(no_df) > 0 else 0
        print(f"  {topic:45s}: {count:5d} ({pct:5.1f}%)")
else:
    print("  (primary_topic column not found)")

# Show overall distribution
print(f"\n{'='*60}")
print("COMBINED LABEL DISTRIBUTION (All Tiers)")
print(f"{'='*60}")
if 'primary_topic' in data_opt4.columns:
    overall_dist = data_opt4['primary_topic'].value_counts()
    for topic in topic_names:
        count = overall_dist.get(topic, 0)
        pct = count / len(data_opt4) * 100
        print(f"  {topic:45s}: {count:5d} ({pct:5.1f}%)")

    # Show how CP5.2 threshold changes affected distribution
    total = len(data_opt4)
    high_pct = len(high_df) / total * 100
    low_pct = len(low_df) / total * 100
    no_pct = len(no_df) / total * 100

    print(f"\n{'='*60}")
    print("CONFIDENCE TIER DISTRIBUTION")
    print(f"{'='*60}")
    print(f"  HIGH (primary training):   {len(high_df):5d} ({high_pct:5.1f}%)")
    print(f"  LOW (secondary):           {len(low_df):5d} ({low_pct:5.1f}%)")
    print(f"  NO (unlabeled):            {len(no_df):5d} ({no_pct:5.1f}%)")
    print(f"\nNOTE: CP5.2 corpus-adaptive thresholds (0.55/0.35/0.15) produced this distribution.")
    print(f"      Policy corpora typically show higher % in HIGH tier (~25-30%) vs historical (~20-25%).")

print(f"\nStep 2: Splitting each option into train/val...")

def split_with_stratification(data, option_name):
    """
    Split data into train/val, using stratification if possible.
    Only stratify on labeled data (exclude UNLABELED from stratification).
    """
    # Separate labeled and unlabeled data
    labeled_data = data[data['label'] != 'UNLABELED'].copy()
    unlabeled_data = data[data['label'] == 'UNLABELED'].copy()

    # Striping label 'UNLABELED'
    unlabeled_data['label'] = ''

    # Appending labeled + unlabeled data to all_labeled_data
    all_labeled_data = pd.concat([labeled_data, unlabeled_data], ignore_index=True)

    # Check if stratified split is possible for all_labeled data
    if len(all_labeled_data) > 0:
        topic_counts = all_labeled_data['label'].value_counts()
        can_stratify = all(topic_counts >= 2)

        if can_stratify:
            train_labeled, val_labeled = train_test_split(
                all_labeled_data,
                test_size=0.2,
                stratify=all_labeled_data['label'],
                random_state=42
            )
            print(f"  {option_name}: ✓ Stratified split")

            # CP6 ENHANCEMENT: Show train/val distribution
            print(f"    Train labels: {train_labeled['label'].value_counts().to_dict()}")
            print(f"    Val labels:   {val_labeled['label'].value_counts().to_dict()}")
        else:
            train_labeled, val_labeled = train_test_split(
                all_labeled_data,
                test_size=0.2,
                random_state=42
            )
            print(f"  {option_name}: ⚠ Random split (some topics < 2 examples)")
            print(f"    Train: {len(train_labeled)}, Val: {len(val_labeled)}")

        train_data = train_labeled
        val_data = val_labeled
    else:
        # Edge case: only unlabeled data (shouldn't happen but handle it)
        train_data = data
        val_data = data.head(0)  # Empty validation set
        print(f"  {option_name}: ⚠ No labeled data for validation")

    return train_data, val_data

# Split each option

train_opt4, val_opt4 = split_with_stratification(data_opt4, "Option 4")

# =====================
# SUMMARY
# =====================

print(f"\n{'='*60}")
print("TRAIN/VAL SPLIT SUMMARY")
print(f"{'='*60}")

print(f"\nOption 4 (All) - RECOMMENDED:")
print(f"  Training:   {len(train_opt4):>6} examples")
print(f"  Validation: {len(val_opt4):>6} examples")

# CP6 ENHANCEMENT: Verify no cross-encoder columns leaked through
cross_encoder_cols = [col for col in train_opt4.columns if 'bertje' in col.lower() or 'cross_encoder' in col.lower()]
if cross_encoder_cols:
    print(f"\n⚠ WARNING: Found potential cross-encoder columns in training data:")
    for col in cross_encoder_cols:
        print(f"    - {col}")
    print(f"  These should be removed for bi-encoder-only training.")
else:
    print(f"\n✓ No cross-encoder columns detected (bi-encoder mode confirmed)")

# =====================
# SAVE DATA
# =====================

print(f"\n{'='*60}")
print("SAVING DATA")
print(f"{'='*60}")

# Save label mapping
fs.save_data(label2id, "bertje_label_mapping", "Model_finetuning", "json")
print(f"✓ Saved label mapping")

# Save all training and validation sets

fs.save_data(train_opt4, "train_data_option4", "Model_finetuning", "csv")
fs.save_data(val_opt4, "val_data_option4", "Model_finetuning", "csv")


print(f"\n✓ CHECKPOINT 6 COMPLETE - Training data prepared")



Option 4 (All confidence levels combined):
  HIGH confidence: 411 chunks
  LOW confidence:  630 chunks
  NO confidence:   1799 chunks (treated as UNLABELED)
  Total: 2840 chunks

LABEL DISTRIBUTION BY CONFIDENCE TIER

After CP5.2 corpus-adaptive thresholds:

HIGH Confidence (411 chunks):
  Colonial Systems                             :    56 ( 13.6%)
  Heritage & Memory                            :    30 (  7.3%)
  Historical Slavery                           :   121 ( 29.4%)
  Modern Racism & Discrimination               :   204 ( 49.6%)

LOW Confidence (630 chunks):
  Colonial Systems                             :   141 ( 22.4%)
  Heritage & Memory                            :   104 ( 16.5%)
  Historical Slavery                           :   227 ( 36.0%)
  Modern Racism & Discrimination               :   158 ( 25.1%)

NO Confidence (1799 chunks):
  Colonial Systems                             :   367 ( 20.4%)
  Heritage & Memory                            :   392 ( 21.8%)
  Historic

In [41]:
# ============================================================
# CELL 6.1: PREPARE LABELED DATA
# ============================================================

print(f"\n{'='*60}")
print("PREPARING LABELED DATA")
print(f"{'='*60}")

# High confidence = labeled data
df_labeled = high_df.copy()
df_labeled['text'] = df_labeled['text_for_scoring']
df_labeled['label'] = df_labeled['primary_topic']

# Create label mapping
label2id = {label: idx for idx, label in enumerate(sorted(df_labeled['label'].unique()))}
id2label = {idx: label for label, idx in label2id.items()}
df_labeled['label_id'] = df_labeled['label'].map(label2id)
df_labeled['is_pseudo'] = False

print(f"\nLabel mapping:")
for label, idx in label2id.items():
    count = (df_labeled['label'] == label).sum()
    print(f"  {idx}: {label} ({count} examples)")

print(f"\nTotal labeled examples: {len(df_labeled)}")


PREPARING LABELED DATA

Label mapping:
  0: Colonial Systems (56 examples)
  1: Heritage & Memory (30 examples)
  2: Historical Slavery (121 examples)
  3: Modern Racism & Discrimination (204 examples)

Total labeled examples: 411


In [42]:
# ============================================================
# CELL 6.2: PREPARE PSEUDO-LABELED & UNLABELED DATA
# ============================================================

# =====================
# CONFIGURATION
# =====================

print(f"{'='*60}")
print("PREPARING PSEUDO-LABELED & UNLABELED DATA")
print(f"{'='*60}")

# =====================
# PREPARE PSEUDO-LABELED DATA
# =====================

# Pseudo-labeled pool (low confidence predictions)
df_pseudo = low_df.copy()
df_pseudo["text"] = df_pseudo["text_for_scoring"]
df_pseudo["label"] = df_pseudo["primary_topic"]
df_pseudo["label_id"] = df_pseudo["label"].map(label2id)
df_pseudo["is_pseudo"] = True

print(f"Pseudo-labeled pool: {len(df_pseudo)} chunks")

# Sample pseudo-labeled data for balance
max_pseudo = int(len(df_labeled) * CONFIG["sampling"]["pseudo_multiplier"])
if len(df_pseudo) > max_pseudo:
    # IMPORTANT: Keep ALL columns including cosine scores
    df_pseudo_sampled = df_pseudo.sample(n=max_pseudo, random_state=42)
    print(f"  Sampled: {len(df_pseudo_sampled)} (to maintain balance)")
else:
    df_pseudo_sampled = df_pseudo.copy()
    print(f"  Using all: {len(df_pseudo_sampled)}")

print(f"  Columns preserved: {len(df_pseudo_sampled.columns)}")

# =====================
# PREPARE UNLABELED DATA
# =====================

# Unlabeled pool (no confidence predictions)
df_unlabeled = no_df.copy()  # Keep ALL columns
df_unlabeled["text"] = df_unlabeled["text_for_scoring"]
df_unlabeled["label"] = "UNLABELED"
df_unlabeled["label_id"] = -1
df_unlabeled["is_pseudo"] = False

# Clean: remove empty/null text
df_unlabeled = df_unlabeled[df_unlabeled["text"].notna()].copy()
df_unlabeled = df_unlabeled[df_unlabeled["text"].astype(str).str.strip() != ""].copy()

print(f"Unlabeled pool: {len(df_unlabeled)} chunks")

# Sample unlabeled data for balance
max_unlabeled = int(len(df_labeled) * CONFIG["sampling"]["unlabeled_multiplier"])
if len(df_unlabeled) > max_unlabeled:
    df_unlabeled_sampled = df_unlabeled.sample(n=max_unlabeled, random_state=42)
    print(f"  Sampled: {len(df_unlabeled_sampled)} (to maintain balance)")
else:
    df_unlabeled_sampled = df_unlabeled.copy()
    print(f"  Using all: {len(df_unlabeled_sampled)}")

print(f"  Columns preserved: {len(df_unlabeled_sampled.columns)}")

# =====================
# SUMMARY
# =====================

print(f"{'='*60}")
print("DATA PREPARATION SUMMARY")
print(f"{'='*60}")
print(f"  Labeled:     {len(df_labeled)} (columns: {len(df_labeled.columns)})")
print(f"  Pseudo:      {len(df_pseudo_sampled)} (columns: {len(df_pseudo_sampled.columns)})")
print(f"  Unlabeled:   {len(df_unlabeled_sampled)} (columns: {len(df_unlabeled_sampled.columns)})")
print(f"  Total pool:  {len(df_labeled) + len(df_pseudo_sampled) + len(df_unlabeled_sampled)}")

PREPARING PSEUDO-LABELED & UNLABELED DATA
Pseudo-labeled pool: 630 chunks
  Using all: 630
  Columns preserved: 27
Unlabeled pool: 1799 chunks
  Using all: 1799
  Columns preserved: 27
DATA PREPARATION SUMMARY
  Labeled:     411 (columns: 27)
  Pseudo:      630 (columns: 27)
  Unlabeled:   1799 (columns: 27)
  Total pool:  2840


In [43]:
# ============================================================
# CELL 6.3: CREATE DATASET OPTIONS & TRAIN/VAL SPLIT
# ============================================================

print(f"\n{'='*60}")
print("CREATING DATASET OPTIONS & TRAIN/VAL SPLIT")
print(f"{'='*60}")

# =====================
# STEP 1: GROUP DATA INTO OPTIONS FIRST
# =====================

print(f"\nStep 1: Grouping data into options...")

# =====================
# OPTION 1: LABELED ONLY
# =====================
data_opt1 = df_labeled.copy()

# =====================
# OPTION 2: LABELED + PSEUDO-LABELED
# =====================
data_opt2 = pd.concat([
    df_labeled,
    df_pseudo_sampled
], ignore_index=True)

# =====================
# OPTION 3: LABELED + UNLABELED
# =====================
data_opt3 = pd.concat([
    df_labeled,
    df_unlabeled_sampled
], ignore_index=True)

# =====================
# OPTION 4: ALL (LABELED + PSEUDO + UNLABELED)
# =====================
data_opt4 = pd.concat([
    df_labeled,
    df_pseudo_sampled,
    df_unlabeled_sampled
], ignore_index=True)

print(f"  Option 1 (Labeled only):          {len(data_opt1):>6} examples")
print(f"  Option 2 (Labeled + Pseudo):      {len(data_opt2):>6} examples")
print(f"  Option 3 (Labeled + Unlabeled):   {len(data_opt3):>6} examples")
print(f"  Option 4 (All) ⭐ RECOMMENDED:    {len(data_opt4):>6} examples")

# =====================
# STEP 2: SPLIT EACH OPTION INTO TRAIN/VAL
# =====================

print(f"\nStep 2: Splitting each option into train/val...")

def split_with_stratification(data, option_name):
    """
    Split data into train/val, using stratification if possible.
    Only stratify on labeled data (exclude UNLABELED from stratification).
    """
    # Separate labeled and unlabeled data
    labeled_data = data[data['label'] != 'UNLABELED'].copy()
    unlabeled_data = data[data['label'] == 'UNLABELED'].copy()

    # Striping label 'UNLABELED'
    unlabeled_data['label'] = ''

    # Appending labeled + unlabeled data to all_labeled_data
    all_labeled_data = pd.concat([labeled_data, unlabeled_data], ignore_index=True)
    
    # Check if stratified split is possible for all_labeled data
    if len(all_labeled_data) > 0:
        topic_counts = all_labeled_data['label'].value_counts()
        can_stratify = all(topic_counts >= 2)
        
        if can_stratify:
            train_labeled, val_labeled = train_test_split(
                all_labeled_data,
                test_size=0.2,
                stratify=all_labeled_data['label'],
                random_state=42
            )
            print(f"  {option_name}: ✓ Stratified split")
        else:
            train_labeled, val_labeled = train_test_split(
                all_labeled_data,
                test_size=0.2,
                random_state=42
            )
            print(f"  {option_name}:  Random split (some topics < 2 examples)")
        
        train_data = train_labeled
        val_data = val_labeled
    else:
        # Edge case: only unlabeled data (shouldn't happen but handle it)
        train_data = data
        val_data = data.head(0)  # Empty validation set
        print(f"  {option_name}: ⚠ No labeled data for validation")
    
    return train_data, val_data

# Split each option
train_opt1, val_opt1 = split_with_stratification(data_opt1, "Option 1")
train_opt2, val_opt2 = split_with_stratification(data_opt2, "Option 2")
train_opt3, val_opt3 = split_with_stratification(data_opt3, "Option 3")
train_opt4, val_opt4 = split_with_stratification(data_opt4, "Option 4")

# =====================
# SUMMARY
# =====================

print(f"\n{'='*60}")
print("TRAIN/VAL SPLIT SUMMARY")
print(f"{'='*60}")
print(f"\nOption 1 (Labeled only):")
print(f"  Training:   {len(train_opt1):>6} examples")
print(f"  Validation: {len(val_opt1):>6} examples")

print(f"\nOption 2 (Labeled + Pseudo):")
print(f"  Training:   {len(train_opt2):>6} examples")
print(f"  Validation: {len(val_opt2):>6} examples")

print(f"\nOption 3 (Labeled + Unlabeled):")
print(f"  Training:   {len(train_opt3):>6} examples")
print(f"  Validation: {len(val_opt3):>6} examples")

print(f"\nOption 4 (All)  RECOMMENDED:")
print(f"  Training:   {len(train_opt4):>6} examples")
print(f"  Validation: {len(val_opt4):>6} examples")

# =====================
# SAVE DATA
# =====================

print(f"\n{'='*60}")
print("SAVING DATA")
print(f"{'='*60}")

# Save label mapping
fs.save_data(label2id, "bertje_label_mapping", "Model_finetuning", "json")
print(f"✓ Saved label mapping")

# Save all training and validation sets
fs.save_data(train_opt1, "train_data_option1", "Model_finetuning", "csv")
fs.save_data(val_opt1, "val_data_option1", "Model_finetuning", "csv")

fs.save_data(train_opt2, "train_data_option2", "Model_finetuning", "csv")
fs.save_data(val_opt2, "val_data_option2", "Model_finetuning", "csv")

fs.save_data(train_opt3, "train_data_option3", "Model_finetuning", "csv")
fs.save_data(val_opt3, "val_data_option3", "Model_finetuning", "csv")

fs.save_data(train_opt4, "train_data_option4", "Model_finetuning", "csv")
fs.save_data(val_opt4, "val_data_option4", "Model_finetuning", "csv")

# Save unlabeled pool separately for diagnostics
fs.save_data(df_unlabeled_sampled, "unlabeled_pool", "Model_finetuning", "csv")
print(f" Saved unlabeled pool separately (excluded from training/validation)")

print(f"✓ Saved all training and validation sets")

# Save checkpoint
fs.save_config("checkpoint6_training_prep")
print(f"✓ Checkpoint saved")

print(f"\n{'='*60}")
print("✓ DATA PREPARATION COMPLETE")
print(f"{'='*60}")
print(f"\nReady for model training!")
print(f"Choose one of the training options (1-4) for your model.")


# ============================================================
# ============================================================
# Distribution Table: Per-Topic Quality Metrics
# ============================================================
print("\n" + "="*80)
print("DISTRIBUTION TABLE: Chunks per Topic with Mean Dot Product Scores")
print("="*80)

# Get topic names from score columns
score_cols = [col for col in train_opt4.columns if col.startswith('score_')]
topics = [col.replace('score_', '') for col in score_cols]

# Calculate dynamic threshold (use median across all topics as baseline)
# This adapts to the actual score distribution
all_scores = pd.concat([train_opt4[score_cols].stack(), val_opt4[score_cols].stack()])
score_median = all_scores.median()
score_p75 = all_scores.quantile(0.75)

# Use P75 as "high relevance" threshold
relevance_threshold = score_p75

print(f"\nScore distribution stats:")
print(f"  Median:     {score_median:.2f}")
print(f"  P75:        {score_p75:.2f}")
print(f"  Using P75 as 'high relevance' threshold: {relevance_threshold:.2f}")

# Create table for Option 4 (All data - RECOMMENDED)
print(f"\nOption 4 (Labeled + Pseudo + Unlabeled) - RECOMMENDED:")
dist_data = []
for topic in topics:
    score_col = f'score_{topic}'

    # Train stats
    train_count = (train_opt4[score_col] >= relevance_threshold).sum()
    train_mean = train_opt4[score_col].mean()
    train_std = train_opt4[score_col].std()

    # Val stats
    val_count = (val_opt4[score_col] >= relevance_threshold).sum()
    val_mean = val_opt4[score_col].mean()
    val_std = val_opt4[score_col].std()

    dist_data.append({
        'Topic': topic,
        'Train_High': train_count,
        'Train_Mean': f"{train_mean:.2f}",
        'Train_Std': f"{train_std:.2f}",
        'Val_High': val_count,
        'Val_Mean': f"{val_mean:.2f}",
        'Val_Std': f"{val_std:.2f}",
        'Total_High': train_count + val_count
    })

dist_table = pd.DataFrame(dist_data)
print("\n" + dist_table.to_string(index=False))
print("="*80)
print(f"Note: 'High' count shows chunks with score >= {relevance_threshold:.2f} (P75 threshold)")
print(f"      Mean/Std show distribution of all scores for that topic")
print("="*80)

# Create table for Option 2 (Labeled + Pseudo) - for comparison
print(f"\nOption 2 (Labeled + Pseudo-labeled):")
dist_data_opt2 = []
for topic in topics:
    score_col = f'score_{topic}'

    # Train stats (Option 2: labeled + pseudo)
    train_count = (train_opt2[score_col] >= relevance_threshold).sum()
    train_mean = train_opt2[score_col].mean()

    # Val stats (Option 2)
    val_count = (val_opt2[score_col] >= relevance_threshold).sum()
    val_mean = val_opt2[score_col].mean()

    dist_data_opt2.append({
        'Topic': topic,
        'Train_High': train_count,
        'Train_Mean': f"{train_mean:.2f}",
        'Val_High': val_count,
        'Val_Mean': f"{val_mean:.2f}",
        'Total_High': train_count + val_count
    })

dist_table_opt2 = pd.DataFrame(dist_data_opt2)
print("\n" + dist_table_opt2.to_string(index=False))
print("="*80)



CREATING DATASET OPTIONS & TRAIN/VAL SPLIT

Step 1: Grouping data into options...
  Option 1 (Labeled only):             411 examples
  Option 2 (Labeled + Pseudo):        1041 examples
  Option 3 (Labeled + Unlabeled):     2210 examples
  Option 4 (All) ⭐ RECOMMENDED:      2840 examples

Step 2: Splitting each option into train/val...
  Option 1: ✓ Stratified split
  Option 2: ✓ Stratified split
  Option 3: ✓ Stratified split
  Option 4: ✓ Stratified split

TRAIN/VAL SPLIT SUMMARY

Option 1 (Labeled only):
  Training:      328 examples
  Validation:     83 examples

Option 2 (Labeled + Pseudo):
  Training:      832 examples
  Validation:    209 examples

Option 3 (Labeled + Unlabeled):
  Training:     1768 examples
  Validation:    442 examples

Option 4 (All)  RECOMMENDED:
  Training:     2272 examples
  Validation:    568 examples

SAVING DATA
✓ Saved: Model_finetuning/bertje_label_mapping.json
✓ Saved label mapping
✓ Saved: Model_finetuning/train_data_option1.csv
✓ Saved: Model_fi

✅ **CHECKPOINT 6 COMPLETE** - Training data prepared

**Resume**: Load train/val CSVs from `Model_finetuning/`

---
# CHECKPOINT 7: SBERT Multi-Label Training with Soft Cosine Targets
---
#
This checkpoint implements:
- **SBERT architecture**: Mean pooling instead of [CLS] token
- **Multi-label classification**: Each topic is an independent binary classifier
- **Soft targets**: Raw cosine scores (0.0-1.0) as training targets
- **BCE loss**: Binary cross-entropy with logits
- No explicit "IRRELEVANT" class - chunks with all low cosines are naturally handled

In [44]:
# ============================================================
# CELL 7.0: Setup for Continuous Regression Training
# ============================================================

from transformers import AutoTokenizer, TrainingArguments
import torch
from pathlib import Path

# Determine model path based on config
if CONFIG["model"]["use_pretrained"] and "pretrained_model_path" in CONFIG["paths"]:
    # Load from pretrained model
    pretrained_base = CONFIG["paths"]["pretrained_model_path"]
    
    # Check if we should use trained_encoder (fine-tuned) or base_encoder (original)
    trained_encoder_path = Path(pretrained_base) / "trained_encoder"
    base_encoder_path = Path(pretrained_base) / "base_encoder"
    
    if trained_encoder_path.exists():
        TRAINING_MODEL = str(trained_encoder_path)
        print(f"Loading FINE-TUNED encoder from: {TRAINING_MODEL}")
    elif base_encoder_path.exists():
        TRAINING_MODEL = str(base_encoder_path)
        print(f"Loading BASE encoder from: {TRAINING_MODEL}")
    else:
        # Fall back to the path as-is (in case it's already a specific model dir)
        TRAINING_MODEL = pretrained_base
        print(f"Loading model from: {TRAINING_MODEL}")
else:
    # Use base model from scratch
    TRAINING_MODEL = CONFIG["model"]["base_model_name"]
    print(f"Using base model from HuggingFace: {TRAINING_MODEL}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(TRAINING_MODEL)

print(f"\nSetup for continuous regression fine-tuning:")
print(f"  Model path: {TRAINING_MODEL}")
print(f"  Tokenizer vocab size: {len(tokenizer)}")
print(f"  Max length: {CONFIG['model']['max_length']}")
print(f"  Use pretrained: {CONFIG['model']['use_pretrained']}")
print(f"  Approach: Continuous multi-label regression (not ordinal!)")


Using base model from HuggingFace: NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers

Setup for continuous regression fine-tuning:
  Model path: NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers
  Tokenizer vocab size: 42774
  Max length: 512
  Use pretrained: False
  Approach: Continuous multi-label regression (not ordinal!)


In [45]:
# ============================================================
# CELL 7.1: Continuous Multi-Label Regression Architecture
# ============================================================

from transformers import AutoModel, AutoConfig
from transformers.modeling_outputs import SequenceClassifierOutput
import torch.nn as nn

class SBERTContinuousMultiLabel(nn.Module):
    """
    SBERT with continuous multi-label regression.
    
    Architecture:
        Text -> BERT -> Mean pooling -> N independent regression heads
    
    Each head predicts a continuous score [0, 1] for its topic.
    """
    def __init__(self, model_name: str, num_topics: int = 4):
        super().__init__()
        
        self.num_topics = num_topics
        
        # Load pre-trained BERT
        self.config = AutoConfig.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name, config=self.config)
        
        hidden_size = self.config.hidden_size
        
        # Create N independent regression heads (1 output per topic)
        self.topic_heads = nn.ModuleList([
            nn.Linear(hidden_size, 1)
            for _ in range(num_topics)
        ])
        
        self.dropout = nn.Dropout(0.2)
        
    def mean_pooling(self, token_embeddings, attention_mask):
        """SBERT-style mean pooling."""
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        return sum_embeddings / sum_mask
        
    def forward(self, input_ids, attention_mask, labels=None):
        # Get BERT embeddings
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        # Mean pooling
        sentence_embedding = self.mean_pooling(outputs.last_hidden_state, attention_mask)
        sentence_embedding = self.dropout(sentence_embedding)
        
        # Predict continuous scores for each topic
        topic_predictions = []
        for head in self.topic_heads:
            pred = head(sentence_embedding)  # [batch, 1] -> unbounded, learns [0, 2]
            topic_predictions.append(pred)
        
        # Stack: [batch, num_topics]
        logits = torch.cat(topic_predictions, dim=1)
        
        loss = None
        if labels is not None:
            loss_fct = nn.MSELoss()
            # Clamp predictions to [0, 2] for training stability
            logits_clamped = torch.clamp(logits, 0.0, 10.0)
            loss = loss_fct(logits_clamped, labels.float())
        
        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

print("V18: SBERTContinuousMultiLabel Defined (no sigmoid)")
print("  - Mean pooling (SBERT)")
print("  - 4 independent regression heads")
print("  - Continuous output [0, 10] per topic (no sigmoid)")
print("  - Loss: MSE on rescaled cosine scores [0, 10]")


V18: SBERTContinuousMultiLabel Defined (no sigmoid)
  - Mean pooling (SBERT)
  - 4 independent regression heads
  - Continuous output [0, 10] per topic (no sigmoid)
  - Loss: MSE on rescaled cosine scores [0, 10]


In [46]:
# ============================================================
# CELL 7.4: Data Collator
# ============================================================

from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class ContinuousDataCollator:
    tokenizer: Any
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # Extract labels
        labels = [f.pop("labels") for f in features]
        
        # Pad inputs
        batch = self.tokenizer.pad(
            features,
            padding=True,
            return_tensors="pt"
        )
        
        # Add labels as tensor
        batch["labels"] = torch.tensor(labels, dtype=torch.float32)
        
        return batch

print("V13: ContinuousDataCollator defined")


V13: ContinuousDataCollator defined


In [47]:
# ============================================================
# CELL 7.2: Continuous Multi-Label Dataset
# ============================================================

from torch.utils.data import Dataset
import pandas as pd
import numpy as np

class ContinuousMultiLabelDataset(Dataset):
    """
    Dataset for continuous multi-label regression.
    Uses raw dot product scores [0, ~9] - NO discretization!
    """
    
    def __init__(self, dataframe, tokenizer, topics, config):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.topics = topics
        self.config = config

        self.texts = dataframe["text"].tolist()

        # Extract continuous dot product scores directly
        self.labels = []
        for _, row in dataframe.iterrows():
            label_vec = []
            for topic in topics:
                score_val = row.get(f"score_{topic}", 0.0)
                if pd.isna(score_val):
                    score_val = 0.0
                score_val = float(np.clip(score_val, 0.0, 10.0))
                label_vec.append(score_val)
            self.labels.append(label_vec)
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.config["model"]["max_length"],
            padding=False,
            return_tensors=None
        )
        encoding["labels"] = self.labels[idx]
        return encoding

print("V19: ContinuousMultiLabelDataset Defined (Dot Product)")
print("  - Uses raw dot product scores (0.0-10.0) with full magnitude information")
print("  - No discretization!")


V19: ContinuousMultiLabelDataset Defined (Dot Product)
  - Uses raw dot product scores (0.0-10.0) with full magnitude information
  - No discretization!


In [48]:
# ============================================================
# CELL 7.3: Enhanced Pattern-Aware Multi-Label Regression Metrics
# ============================================================

def compute_continuous_metrics(eval_pred, topic_names=None):
    """
    Compute comprehensive metrics for multi-label regression focusing on PATTERN LEARNING.

    Updated based on LEARNING_SIGNAL_EVALUATION.md recommendations.

    Key insight: BERTje should learn the SHAPE/PROFILE of topic combinations,
    including MAGNITUDE information (dot product scores preserve meaningful scale).

    CRITICAL: We use Pearson correlation NOT cosine similarity, because:
    - Cosine normalizes to unit vectors (loses magnitude: [8,7,1,1] ≈ [2,1.75,0.25,0.25])
    - Dot product scores have meaningful magnitude (8.0 = strong, 2.0 = weak)
    - Pearson preserves relative differences in magnitude

    Metric Categories:
    1. PRIMARY METRICS (optimize for these):
       - Pearson Correlation: Pattern match preserving relative magnitude
       - Euclidean Distance: Direct magnitude + pattern error
       - CV Correlation: Does model learn differentiation patterns?

    2. MULTI-LABEL METRICS (topic co-occurrence):
       - Pairwise Error: Does model capture relative topic strengths?
       - STD/Range Correlation: Does model match variance patterns?

    3. MAGNITUDE METRICS (sanity checks):
       - MAE, RMSE, R² per topic
    """
    predictions, labels = eval_pred
    num_topics = predictions.shape[1]

    if topic_names is None:
        topic_names = [f"Topic_{i}" for i in range(num_topics)]

    metrics = {}

    # =============================================
    # PRIMARY METRICS: Pattern + Relative Magnitude
    # =============================================

    print("\n" + "="*80)
    print("PRIMARY METRICS: Pattern Learning")
    print("="*80)

    # 1. Pearson correlation per chunk (preserves relative magnitude)
    pearson_corrs = []
    for pred_row, label_row in zip(predictions, labels):
        corr = np.corrcoef(pred_row, label_row)[0, 1]
        if np.isnan(corr):
            corr = 0.0
        pearson_corrs.append(corr)

    metrics['mean_pearson'] = np.mean(pearson_corrs)
    metrics['median_pearson'] = np.median(pearson_corrs)
    metrics['std_pearson'] = np.std(pearson_corrs)
    metrics['min_pearson'] = np.min(pearson_corrs)
    metrics['q25_pearson'] = np.quantile(pearson_corrs, 0.25)
    metrics['q75_pearson'] = np.quantile(pearson_corrs, 0.75)

    print(f"Pearson Correlation (pattern + magnitude):")
    print(f"  Mean:   {metrics['mean_pearson']:.4f}  (target: >0.85)")
    print(f"  Median: {metrics['median_pearson']:.4f}")
    print(f"  Std:    {metrics['std_pearson']:.4f}")
    print(f"  Q25-Q75: [{metrics['q25_pearson']:.4f}, {metrics['q75_pearson']:.4f}]")
    print(f"  Min:    {metrics['min_pearson']:.4f}")

    # Count high-quality predictions
    high_corr_pct = (np.array(pearson_corrs) > 0.85).mean() * 100
    metrics['pct_pearson_gt_85'] = high_corr_pct
    print(f"  % chunks with Pearson > 0.85: {high_corr_pct:.1f}%")

    # 2. Euclidean distance (combined magnitude + pattern error)
    euclidean_distances = []
    for pred_row, label_row in zip(predictions, labels):
        dist = np.linalg.norm(pred_row - label_row)
        euclidean_distances.append(dist)

    metrics['mean_euclidean'] = np.mean(euclidean_distances)
    metrics['median_euclidean'] = np.median(euclidean_distances)

    print(f"\nEuclidean Distance (magnitude + pattern error):")
    print(f"  Mean:   {metrics['mean_euclidean']:.4f}  (target: <1.0)")
    print(f"  Median: {metrics['median_euclidean']:.4f}")

    # Normalized Euclidean (as percentage of target magnitude)
    normalized_distances = []
    for pred_row, label_row in zip(predictions, labels):
        dist = np.linalg.norm(pred_row - label_row)
        target_magnitude = np.linalg.norm(label_row)
        normalized_dist = dist / (target_magnitude + 1e-8)
        normalized_distances.append(normalized_dist)

    metrics['mean_normalized_euclidean'] = np.mean(normalized_distances)
    print(f"  Normalized Mean: {metrics['mean_normalized_euclidean']:.4f}  (as % of target magnitude)")

    # =============================================
    # DIFFERENTIATION METRICS (CV patterns)
    # =============================================

    print("\n" + "="*80)
    print("DIFFERENTIATION METRICS: Topic Separation Patterns")
    print("="*80)

    # Coefficient of Variation (CV) for each chunk
    pred_cvs = []
    label_cvs = []
    for pred_row, label_row in zip(predictions, labels):
        pred_cv = np.std(pred_row) / (np.mean(pred_row) + 1e-8)
        label_cv = np.std(label_row) / (np.mean(label_row) + 1e-8)
        pred_cvs.append(pred_cv)
        label_cvs.append(label_cv)

    # CV correlation: Does model learn differentiation patterns?
    cv_corr = np.corrcoef(pred_cvs, label_cvs)[0, 1]
    if np.isnan(cv_corr):
        cv_corr = 0.0
    metrics['cv_correlation'] = cv_corr

    # CV MAE: How accurately does model predict differentiation level?
    cv_mae = np.mean(np.abs(np.array(pred_cvs) - np.array(label_cvs)))
    metrics['cv_mae'] = cv_mae

    # Average CV (diagnostic)
    metrics['mean_pred_cv'] = np.mean(pred_cvs)
    metrics['mean_label_cv'] = np.mean(label_cvs)

    print(f"Coefficient of Variation (CV):")
    print(f"  CV Correlation: {cv_corr:.4f}  (target: >0.75)")
    print(f"  CV MAE:         {cv_mae:.4f}")
    print(f"  Mean Pred CV:   {metrics['mean_pred_cv']:.4f}")
    print(f"  Mean Label CV:  {metrics['mean_label_cv']:.4f}")
    print(f"  CV Match:       {abs(metrics['mean_pred_cv'] - metrics['mean_label_cv']) < 0.02}")

    # =============================================
    # MULTI-LABEL METRICS (Relative Strengths)
    # =============================================

    print("\n" + "="*80)
    print("MULTI-LABEL METRICS: Topic Co-occurrence Patterns")
    print("="*80)

    # Pairwise score differences (how well does model capture relative strengths?)
    pairwise_errors = []
    for pred_row, label_row in zip(predictions, labels):
        for i in range(num_topics):
            for j in range(i + 1, num_topics):
                label_diff = label_row[i] - label_row[j]
                pred_diff = pred_row[i] - pred_row[j]
                pairwise_errors.append(abs(pred_diff - label_diff))

    metrics['mean_pairwise_error'] = np.mean(pairwise_errors)
    metrics['median_pairwise_error'] = np.median(pairwise_errors)

    print(f"Pairwise Topic Differences:")
    print(f"  Mean Error:   {metrics['mean_pairwise_error']:.4f}  (target: <0.5)")
    print(f"  Median Error: {metrics['median_pairwise_error']:.4f}")

    # Standard deviation correlation (spread/variance pattern)
    pred_stds = np.std(predictions, axis=1)
    label_stds = np.std(labels, axis=1)
    std_mae = np.mean(np.abs(pred_stds - label_stds))
    metrics['std_mae'] = std_mae

    std_corr = np.corrcoef(pred_stds, label_stds)[0, 1]
    if np.isnan(std_corr):
        std_corr = 0.0
    metrics['std_correlation'] = std_corr

    print(f"\nStandard Deviation (spread pattern):")
    print(f"  STD Correlation: {std_corr:.4f}")
    print(f"  STD MAE:         {std_mae:.4f}")

    # Range correlation (max - min within each chunk)
    pred_ranges = np.max(predictions, axis=1) - np.min(predictions, axis=1)
    label_ranges = np.max(labels, axis=1) - np.min(labels, axis=1)
    range_corr = np.corrcoef(pred_ranges, label_ranges)[0, 1]
    if np.isnan(range_corr):
        range_corr = 0.0
    metrics['range_correlation'] = range_corr
    range_mae = np.mean(np.abs(pred_ranges - label_ranges))
    metrics['range_mae'] = range_mae

    print(f"\nScore Range (max - min, differentiation strength):")
    print(f"  Range Correlation: {range_corr:.4f}")
    print(f"  Range MAE:         {range_mae:.4f}")

    # =============================================
    # MAGNITUDE METRICS (Per-Topic Accuracy)
    # =============================================

    print("\n" + "="*80)
    print("MAGNITUDE METRICS: Per-Topic Accuracy")
    print("="*80)

    # Global metrics
    global_mae = np.mean(np.abs(predictions - labels))
    metrics['global_mae'] = global_mae

    global_rmse = np.sqrt(np.mean((predictions - labels) ** 2))
    metrics['global_rmse'] = global_rmse

    ss_res = np.sum((labels - predictions) ** 2)
    ss_tot = np.sum((labels - np.mean(labels)) ** 2)
    global_r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    metrics['global_r2'] = global_r2

    print(f"Global Accuracy:")
    print(f"  MAE:  {global_mae:.4f}  (target: <0.8)")
    print(f"  RMSE: {global_rmse:.4f}")
    print(f"  R²:   {global_r2:.4f}")

    # Per-topic metrics
    print(f"\nPer-Topic Breakdown:")
    topic_maes = []
    topic_rmses = []
    topic_r2s = []

    for topic_idx in range(num_topics):
        pred_topic = predictions[:, topic_idx]
        label_topic = labels[:, topic_idx]

        # MAE
        mae = np.mean(np.abs(pred_topic - label_topic))
        topic_maes.append(mae)

        # RMSE
        rmse = np.sqrt(np.mean((pred_topic - label_topic) ** 2))
        topic_rmses.append(rmse)

        # R²
        ss_res = np.sum((label_topic - pred_topic) ** 2)
        ss_tot = np.sum((label_topic - np.mean(label_topic)) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        topic_r2s.append(r2)

        # Short name for display
        topic_short = topic_names[topic_idx].split(' & ')[0] if ' & ' in topic_names[topic_idx] else topic_names[topic_idx]
        topic_short = topic_short[:20]  # Truncate long names

        # Store in metrics dict with short name
        metrics[f'mae_{topic_short}'] = mae
        metrics[f'rmse_{topic_short}'] = rmse
        metrics[f'r2_{topic_short}'] = r2

        # Print
        print(f"  {topic_short:20s}: MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}")

    metrics['mean_topic_mae'] = np.mean(topic_maes)
    metrics['mean_topic_rmse'] = np.mean(topic_rmses)
    metrics['mean_topic_r2'] = np.mean(topic_r2s)

    print(f"\n  Average across topics: MAE={metrics['mean_topic_mae']:.4f}, RMSE={metrics['mean_topic_rmse']:.4f}, R²={metrics['mean_topic_r2']:.4f}")

    # =============================================
    # PRIMARY TOPIC ACCURACY (Secondary Metric)
    # =============================================

    print("\n" + "="*80)
    print("PRIMARY TOPIC ACCURACY (Secondary - argmax not full story)")
    print("="*80)

    # Primary topic (argmax)
    pred_primary = np.argmax(predictions, axis=1)
    label_primary = np.argmax(labels, axis=1)
    primary_accuracy = (pred_primary == label_primary).mean()
    metrics['primary_topic_accuracy'] = primary_accuracy

    print(f"Primary Topic Match (argmax): {primary_accuracy:.4f}")
    print(f"  Note: This is secondary - multi-label means one argmax is incomplete")

    # Top-2 accuracy (does predicted top-2 overlap with label top-2?)
    top2_matches = 0
    for pred_row, label_row in zip(predictions, labels):
        pred_top2 = set(np.argsort(pred_row)[-2:])
        label_top2 = set(np.argsort(label_row)[-2:])
        if len(pred_top2 & label_top2) >= 1:  # At least 1 topic in common
            top2_matches += 1

    top2_accuracy = top2_matches / len(predictions)
    metrics['top2_overlap_accuracy'] = top2_accuracy

    print(f"Top-2 Overlap (at least 1 match): {top2_accuracy:.4f}")

    # =============================================
    # SUMMARY & TARGETS
    # =============================================

    print("\n" + "="*80)
    print("SUMMARY: How Well Is BERTje Learning?")
    print("="*80)

    # Check against targets
    targets_met = []

    if metrics['mean_pearson'] > 0.85:
        print("✓ Pearson Correlation > 0.85 (excellent pattern learning)")
        targets_met.append(True)
    elif metrics['mean_pearson'] > 0.80:
        print("⚠ Pearson Correlation > 0.80 (good, aim higher)")
        targets_met.append(True)
    else:
        print(f"✗ Pearson Correlation = {metrics['mean_pearson']:.4f} (target: >0.85)")
        targets_met.append(False)

    if metrics['mean_euclidean'] < 1.0:
        print("✓ Euclidean Distance < 1.0 (low magnitude error)")
        targets_met.append(True)
    else:
        print(f"⚠ Euclidean Distance = {metrics['mean_euclidean']:.4f} (target: <1.0)")
        targets_met.append(False)

    if metrics['cv_correlation'] > 0.75:
        print("✓ CV Correlation > 0.75 (learns differentiation patterns)")
        targets_met.append(True)
    elif metrics['cv_correlation'] > 0.70:
        print("⚠ CV Correlation > 0.70 (decent, aim higher)")
        targets_met.append(True)
    else:
        print(f"✗ CV Correlation = {metrics['cv_correlation']:.4f} (target: >0.75)")
        targets_met.append(False)

    if metrics['mean_pairwise_error'] < 0.5:
        print("✓ Pairwise Error < 0.5 (captures relative topic strengths)")
        targets_met.append(True)
    else:
        print(f"⚠ Pairwise Error = {metrics['mean_pairwise_error']:.4f} (target: <0.5)")
        targets_met.append(False)

    if metrics['global_mae'] < 0.8:
        print("✓ Global MAE < 0.8 (accurate magnitude predictions)")
        targets_met.append(True)
    else:
        print(f"⚠ Global MAE = {metrics['global_mae']:.4f} (target: <0.8)")
        targets_met.append(False)

    targets_met_pct = sum(targets_met) / len(targets_met) * 100
    metrics['targets_met_pct'] = targets_met_pct

    print(f"\nOverall: {sum(targets_met)}/{len(targets_met)} targets met ({targets_met_pct:.0f}%)")

    if targets_met_pct >= 80:
        print("🎉 EXCELLENT: BERTje is learning the multi-topic patterns well!")
    elif targets_met_pct >= 60:
        print("👍 GOOD: BERTje is learning, some room for improvement")
    else:
        print("⚠️  NEEDS WORK: Consider adjusting hyperparameters or loss function")

    print("="*80)

    return metrics


print("="*80)
print("ENHANCED PATTERN-AWARE METRICS LOADED")
print("="*80)
print("\nMetric Categories:")
print("  1. PRIMARY: Pearson, Euclidean, CV Correlation")
print("  2. MULTI-LABEL: Pairwise Error, STD/Range Correlation")
print("  3. MAGNITUDE: MAE, RMSE, R² per topic")
print("  4. DIAGNOSTIC: Primary topic accuracy, Top-2 overlap")
print("\nKey Improvements:")
print("  • Comprehensive Pearson statistics (mean, median, Q25/Q75, % > 0.85)")
print("  • Detailed differentiation metrics (CV correlation, CV MAE)")
print("  • Multi-label awareness (pairwise error, top-2 overlap)")
print("  • Clear target benchmarks with pass/fail indicators")
print("  • Summary scorecard for quick assessment")
print("\nTargets (based on LEARNING_SIGNAL_EVALUATION.md):")
print("  • Pearson > 0.85  (pattern learning)")
print("  • Euclidean < 1.0 (magnitude error)")
print("  • CV Corr > 0.75  (differentiation)")
print("  • Pairwise < 0.5  (relative strengths)")
print("  • Global MAE < 0.8 (accuracy)")
print("="*80)


ENHANCED PATTERN-AWARE METRICS LOADED

Metric Categories:
  1. PRIMARY: Pearson, Euclidean, CV Correlation
  2. MULTI-LABEL: Pairwise Error, STD/Range Correlation
  3. MAGNITUDE: MAE, RMSE, R² per topic
  4. DIAGNOSTIC: Primary topic accuracy, Top-2 overlap

Key Improvements:
  • Comprehensive Pearson statistics (mean, median, Q25/Q75, % > 0.85)
  • Detailed differentiation metrics (CV correlation, CV MAE)
  • Multi-label awareness (pairwise error, top-2 overlap)
  • Clear target benchmarks with pass/fail indicators
  • Summary scorecard for quick assessment

Targets (based on LEARNING_SIGNAL_EVALUATION.md):
  • Pearson > 0.85  (pattern learning)
  • Euclidean < 1.0 (magnitude error)
  • CV Corr > 0.75  (differentiation)
  • Pairwise < 0.5  (relative strengths)
  • Global MAE < 0.8 (accuracy)


In [49]:
# ============================================================
# CELL 7.5: Load Training Data
# ============================================================

# Determine which dataset option to use
dataset_option = CONFIG.get("training", {}).get("dataset_option", "option2")

# Load data
train_data_path = fs.folders["Model_finetuning"] / f"train_data_{dataset_option}.csv"
val_data_path = fs.folders["Model_finetuning"] / f"val_data_{dataset_option}.csv"

print(f"Loading training data (dataset {dataset_option}):")
print(f"  Train: {train_data_path}")
print(f"  Val:   {val_data_path}")

train_df = pd.read_csv(train_data_path)
val_df = pd.read_csv(val_data_path)

print(f"\nData loaded:")
print(f"  Train: {len(train_df)} chunks")
print(f"  Val:   {len(val_df)} chunks")

# Get topics from columns
score_cols = [c for c in train_df.columns if c.startswith('score_')]
topics = [c.replace('score_', '') for c in score_cols]

print(f"\nTopics ({len(topics)}):")
for topic in topics:
    print(f"  - {topic}")


Loading training data (dataset option4):
  Train: workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Model_finetuning\train_data_option4.csv
  Val:   workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Model_finetuning\val_data_option4.csv

Data loaded:
  Train: 2272 chunks
  Val:   568 chunks

Topics (4):
  - Colonial Systems
  - Heritage & Memory
  - Historical Slavery
  - Modern Racism & Discrimination


In [50]:
# ============================================================
# CELL 7.5a: Verify Confidence Distribution (from Cell 37)
# ============================================================

print(f"\n{'='*60}")
print("VERIFYING CONFIDENCE DISTRIBUTION")
print(f"{'='*60}")

# The 'confidence' column was already calculated in Cell 37 using significance scoring
# It provides better classification than simple margin-based approaches because:
#   - Uses Coefficient of Variation (CV) to detect noise
#   - Combines magnitude + differentiation + contrast
#   - Calibrated for dot product scores (0-10 range)

if 'confidence' not in train_df.columns:
    raise ValueError(
        "'confidence' column not found!\n"
        "Make sure you ran Cell 37 (significance scoring) and Cell 42-44 (training prep)"
    )

# Show distribution
print("\nConfidence level distribution (from Cell 37 significance scoring):")
print(f"{'Level':<15} {'Count':>8} {'Percent':>10}")
print("-" * 35)

for conf in ['high', 'low', 'none']:
    count = (train_df['confidence'] == conf).sum()
    pct = count / len(train_df) * 100
    print(f"{conf:<15} {count:>8} {pct:>9.1f}%")

print("-" * 35)
print(f"{'Total':<15} {len(train_df):>8} {100.0:>9.1f}%")

print("\nConfidence level meanings:")
print("  high: High significance (>= 0.70) - Primary training data")
print("  low:  Medium/Low significance (0.30-0.70) - Secondary training data")
print("  none: Noise (< 0.30 or CV < 0.10) - Exclude or use as hard negatives")

print("\n✓ Using existing significance-based confidence from Cell 37")



VERIFYING CONFIDENCE DISTRIBUTION

Confidence level distribution (from Cell 37 significance scoring):
Level              Count    Percent
-----------------------------------
high                 332      14.6%
low                  501      22.1%
none                1439      63.3%
-----------------------------------
Total               2272     100.0%

Confidence level meanings:
  high: High significance (>= 0.70) - Primary training data
  low:  Medium/Low significance (0.30-0.70) - Secondary training data
  none: Noise (< 0.30 or CV < 0.10) - Exclude or use as hard negatives

✓ Using existing significance-based confidence from Cell 37


In [31]:
# ============================================================
# CELL 7.5b: Confidence + Topic Balanced Stratified Sampling
# ============================================================

print(f"\n{'='*60}")
print("CONFIDENCE + TOPIC BALANCED STRATIFIED SAMPLING")
print(f"{'='*60}")

# Configuration
APPLY_CONFIDENCE_SAMPLING = CONFIG.get("training", {}).get("apply_confidence_sampling", False)

if not APPLY_CONFIDENCE_SAMPLING:
    print("\n⏭️  Confidence sampling DISABLED in CONFIG")
    print("   Using natural distribution from training data")
    print(f"   Total training examples: {len(train_df)}")
    print("\n✓ Skipping stratified sampling")
else:
    print("\n📊 Applying confidence + topic balanced stratified sampling...")

    # Target distribution for confidence levels
    # Adjust these ratios based on your needs
    TARGET_DISTRIBUTION = {
        'high': 0.40,   # 50% high confidence (primary training data)
        'low': 0.20,    # 40% low confidence (secondary data)
        'none': 0.10    # 10% none (hard negatives)
    }

    print("\nTarget confidence distribution:")
    for conf, ratio in TARGET_DISTRIBUTION.items():
        print(f"  {conf:<10} {ratio*100:>5.1f}%")

    # Show original distribution
    print("\n" + "="*60)
    print("ORIGINAL DISTRIBUTION")
    print("="*60)

    original_total = len(train_df)

    # By confidence
    print("\nBy confidence:")
    for conf in ['high', 'low', 'none']:
        count = (train_df['confidence'] == conf).sum()
        pct = count / original_total * 100
        print(f"  {conf:<10} {count:>6} ({pct:>5.1f}%)")

    # By topic (label)
    if 'label' in train_df.columns:
        print("\nBy topic (label):")
        label_counts = train_df['label'].value_counts()
        for label, count in label_counts.items():
            pct = count / original_total * 100
            print(f"  {label:<40} {count:>6} ({pct:>5.1f}%)")

    print(f"\nTotal: {original_total}")

    # =============================================
    # STEP 1: Sample by confidence levels
    # =============================================

    print("\n" + "="*60)
    print("STEP 1: SAMPLE BY CONFIDENCE")
    print("="*60)

    sampled_by_confidence = []

    for conf in ['high', 'low', 'none']:
        subset = train_df[train_df['confidence'] == conf]
        target = int(original_total * TARGET_DISTRIBUTION[conf])

        if len(subset) == 0:
            print(f"\n  {conf}: No data available, skipping")
            continue

        if len(subset) >= target:
            sampled = subset.sample(n=target, random_state=42)
            print(f"  {conf}: Sampled {target} from {len(subset)} (downsampling)")
        else:
            sampled = subset.sample(n=target, replace=True, random_state=42)
            print(f"  {conf}: Sampled {target} from {len(subset)} (upsampling)")

        sampled_by_confidence.append(sampled)

    if len(sampled_by_confidence) == 0:
        print("\n⚠️  No samples collected, using original data")
        train_df_sampled = train_df.copy()
    else:
        train_df_sampled = pd.concat(sampled_by_confidence, ignore_index=True)

    # =============================================
    # STEP 2: Balance topics within each confidence level
    # =============================================

    if 'label' in train_df_sampled.columns and train_df_sampled['label'].isna().any():
        # Separate labeled and unlabeled
        labeled_data = train_df_sampled[train_df_sampled['label'].notna()].copy()
        unlabeled_data = train_df_sampled[train_df_sampled['label'].isna()].copy()

        print("\n" + "="*60)
        print("STEP 2: BALANCE TOPICS (LABELED DATA ONLY)")
        print("="*60)

        # Balance topics within labeled data
        unique_labels = labeled_data['label'].unique()
        target_per_topic = len(labeled_data) // len(unique_labels)

        print(f"\nTarget per topic: {target_per_topic} (total labeled: {len(labeled_data)})")
        print(f"Topics to balance: {len(unique_labels)}")

        balanced_dfs = []

        for label in unique_labels:
            label_subset = labeled_data[labeled_data['label'] == label]

            if len(label_subset) >= target_per_topic:
                # Downsample
                sampled = label_subset.sample(n=target_per_topic, random_state=42)
                action = "downsampled"
            else:
                # Upsample
                sampled = label_subset.sample(n=target_per_topic, replace=True, random_state=42)
                action = "upsampled"

            balanced_dfs.append(sampled)
            print(f"  {label:<40} {len(label_subset):>6} → {target_per_topic:>6} ({action})")

        # Combine balanced labeled data with unlabeled data
        balanced_labeled = pd.concat(balanced_dfs, ignore_index=True)
        train_df_final = pd.concat([balanced_labeled, unlabeled_data], ignore_index=True)

        print(f"\nBalanced labeled: {len(balanced_labeled)}")
        print(f"Unlabeled kept:   {len(unlabeled_data)}")

    elif 'label' in train_df_sampled.columns:
        # All data is labeled - balance topics
        print("\n" + "="*60)
        print("STEP 2: BALANCE TOPICS")
        print("="*60)

        unique_labels = train_df_sampled['label'].unique()
        target_per_topic = len(train_df_sampled) // len(unique_labels)

        print(f"\nTarget per topic: {target_per_topic}")
        print(f"Topics to balance: {len(unique_labels)}")

        balanced_dfs = []

        for label in unique_labels:
            label_subset = train_df_sampled[train_df_sampled['label'] == label]

            if len(label_subset) >= target_per_topic:
                sampled = label_subset.sample(n=target_per_topic, random_state=42)
                action = "downsampled"
            else:
                sampled = label_subset.sample(n=target_per_topic, replace=True, random_state=42)
                action = "upsampled"

            balanced_dfs.append(sampled)
            print(f"  {label:<40} {len(label_subset):>6} → {target_per_topic:>6} ({action})")

        train_df_final = pd.concat(balanced_dfs, ignore_index=True)
    else:
        print("\n⚠️  No 'label' column found, skipping topic balancing")
        train_df_final = train_df_sampled.copy()

    # Shuffle
    train_df = train_df_final.sample(frac=1, random_state=42).reset_index(drop=True)

    # =============================================
    # SHOW FINAL DISTRIBUTION
    # =============================================

    print("\n" + "="*60)
    print("FINAL DISTRIBUTION AFTER BALANCING")
    print("="*60)

    # By confidence
    print("\nBy confidence:")
    for conf in ['high', 'low', 'none']:
        count = (train_df['confidence'] == conf).sum()
        pct = count / len(train_df) * 100
        print(f"  {conf:<10} {count:>6} ({pct:>5.1f}%)")

    # By topic
    if 'label' in train_df.columns:
        print("\nBy topic (label):")
        label_counts = train_df['label'].value_counts()
        for label, count in label_counts.items():
            pct = count / len(train_df) * 100
            print(f"  {label:<40} {count:>6} ({pct:>5.1f}%)")

    print(f"\nTotal: {len(train_df)}")
    print("\n✓ Confidence + Topic balanced sampling applied")

# Show final summary
print(f"\n{'='*60}")
print("FINAL TRAINING DATA")
print(f"{'='*60}")
print(f"Total examples: {len(train_df)}")

if 'label' in train_df.columns:
    unique_labels = train_df['label'].unique()
    print(f"Unique topics: {len(unique_labels)}")

    # Check balance
    label_counts = train_df['label'].value_counts()
    min_count = label_counts.min()
    max_count = label_counts.max()
    balance_ratio = min_count / max_count if max_count > 0 else 0

    print(f"Balance ratio: {balance_ratio:.2f} (1.0 = perfectly balanced)")
    if balance_ratio >= 0.9:
        print("  ✓ Topics are well balanced")
    elif balance_ratio >= 0.7:
        print("  ⚠️  Topics are moderately balanced")
    else:
        print("  ⚠️  Topics are imbalanced - consider adjusting sampling")

print("\n✓ Training data ready for model")



CONFIDENCE + TOPIC BALANCED STRATIFIED SAMPLING

📊 Applying confidence + topic balanced stratified sampling...

Target confidence distribution:
  high        40.0%
  low         20.0%
  none        10.0%

ORIGINAL DISTRIBUTION


NameError: name 'train_df' is not defined

In [52]:
# ============================================================
# CELL 7.6: Create Continuous Datasets
# ============================================================

print("Creating continuous regression datasets...")

train_dataset = ContinuousMultiLabelDataset(train_df, tokenizer, topics, CONFIG)
val_dataset = ContinuousMultiLabelDataset(val_df, tokenizer, topics, CONFIG)

print(f"\nDatasets created:")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val:   {len(val_dataset)} samples")

# Show sample
print(f"\nSample continuous labels:")
for i in range(min(3, len(train_dataset))):
    labels = train_dataset.labels[i]
    print(f"  Chunk {i}: {[f'{l:.3f}' for l in labels]}")


Creating continuous regression datasets...

Datasets created:
  Train: 1587 samples
  Val:   568 samples

Sample continuous labels:
  Chunk 0: ['1.922', '4.136', '3.084', '3.325']
  Chunk 1: ['3.851', '3.810', '2.944', '4.370']
  Chunk 2: ['4.940', '4.763', '4.848', '4.679']


In [53]:
# ============================================================
# CELL 7.7: Instantiate Continuous Regression Model
# ============================================================

print("Instantiating continuous regression model...")

model = SBERTContinuousMultiLabel(
    model_name=TRAINING_MODEL,
    num_topics=len(topics)
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(f"\nModel instantiated:")
print(f"  Base: {TRAINING_MODEL}")
print(f"  Device: {device}")
print(f"  Topics: {len(topics)}")
print(f"  Heads: {len(model.topic_heads)} regression heads")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")


Instantiating continuous regression model...

Model instantiated:
  Base: NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers
  Device: cuda
  Topics: 4
  Heads: 4 regression heads
  Parameters: 118,895,620


In [54]:
# ============================================================
# CELL 7.8: Define Training Arguments
# ============================================================

training_args = TrainingArguments(
    output_dir=str(fs.folders["Model_finetuning"]),
    num_train_epochs=CONFIG["training"]["num_epochs"],
    per_device_train_batch_size=CONFIG["training"]["batch_size_train"],
    per_device_eval_batch_size=CONFIG["training"]["batch_size_eval"],
    learning_rate=CONFIG["training"]["learning_rate"],
    weight_decay=CONFIG["training"]["weight_decay"],
    warmup_ratio=CONFIG["training"]["warmup_ratio"],
    
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    
    load_best_model_at_end=True,
    metric_for_best_model="mean_pearson",  # Primary metric: Pearson correlation (pattern + magnitude)
    greater_is_better=True,
    
    logging_dir=str(fs.folders["Model_finetuning"] / "logs"),
    logging_steps=50,
    
    disable_tqdm=False,
    report_to="none",
    
    # GPU acceleration settings - speeds up training without changing accuracy
    fp16=True,  # Mixed precision training - 2-3x faster on GPU
    dataloader_num_workers=0,  # Set to 0 for Windows compatibility (prevents hanging)
    dataloader_pin_memory=True,  # Faster data transfer to GPU
    gradient_checkpointing=False,  # Set to True if you run out of GPU memory
)

print("Training arguments defined:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Metric for best model: {training_args.metric_for_best_model}")
print(f"  FP16 (mixed precision): {training_args.fp16}")
print(f"  Dataloader workers: {training_args.dataloader_num_workers}")

Training arguments defined:
  Epochs: 5
  Batch size: 16
  Learning rate: 2e-05
  Metric for best model: mean_pearson
  FP16 (mixed precision): True
  Dataloader workers: 0


In [55]:
# ============================================================
# CELL 7.9: Create Trainer and Train
# ============================================================

from transformers import Trainer

# Create data collator
data_collator = ContinuousDataCollator(tokenizer=tokenizer)

# Create compute metrics function with topic names
def compute_metrics_with_topics(eval_pred):
    return compute_continuous_metrics(eval_pred, topic_names=topics)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics_with_topics,
)

print("Trainer created")
print("\nStarting training...")
print("="*80)

# Train
train_result = trainer.train()

print("="*80)
print("Training complete!")


Trainer created

Starting training...


  0%|          | 0/500 [00:00<?, ?it/s]

You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


{'loss': 12.4201, 'grad_norm': 9.59862232208252, 'learning_rate': 1.88e-05, 'epoch': 0.5}
{'loss': 1.2453, 'grad_norm': 24.806718826293945, 'learning_rate': 1.791111111111111e-05, 'epoch': 1.0}


  0%|          | 0/18 [00:00<?, ?it/s]

...


PRIMARY METRICS: Pattern Learning
Pearson Correlation (pattern + magnitude):
  Mean:   0.7828  (target: >0.85)
  Median: 0.9161
  Std:    0.3405
  Q25-Q75: [0.7777, 0.9728]
  Min:    -0.9082
  % chunks with Pearson > 0.85: 64.4%

Euclidean Distance (magnitude + pattern error):
  Mean:   1.2200  (target: <1.0)
  Median: 1.0679
  Normalized Mean: 0.1376  (as % of target magnitude)

DIFFERENTIATION METRICS: Topic Separation Patterns
Coefficient of Variation (CV):
  CV Correlation: 0.7123  (target: >0.75)
  CV MAE:         0.0370
  Mean Pred CV:   0.1353
  Mean Label CV:  0.1374
  CV Match:       True

MULTI-LABEL METRICS: Topic Co-occurrence Patterns
Pairwise Topic Differences:
  Mean Error:   0.4288  (target: <0.5)
  Median Error: 0.3503

Standard Deviation (spread pattern):
  STD Correlation: 0.7639
  STD MAE:         0.1508

Score Range (max - min, differentiation strength):
  Range Correlation: 0.7537
  Range MAE:         0.3929

MAGNITUDE METRICS: Per-Topic Accuracy
Global Accuracy:

  0%|          | 0/18 [00:00<?, ?it/s]


PRIMARY METRICS: Pattern Learning
Pearson Correlation (pattern + magnitude):
  Mean:   0.8011  (target: >0.85)
  Median: 0.9173
  Std:    0.3123
  Q25-Q75: [0.7839, 0.9793]
  Min:    -0.8903
  % chunks with Pearson > 0.85: 66.0%

Euclidean Distance (magnitude + pattern error):
  Mean:   1.1741  (target: <1.0)
  Median: 1.0387
  Normalized Mean: 0.1328  (as % of target magnitude)

DIFFERENTIATION METRICS: Topic Separation Patterns
Coefficient of Variation (CV):
  CV Correlation: 0.7243  (target: >0.75)
  CV MAE:         0.0370
  Mean Pred CV:   0.1367
  Mean Label CV:  0.1374
  CV Match:       True

MULTI-LABEL METRICS: Topic Co-occurrence Patterns
Pairwise Topic Differences:
  Mean Error:   0.4183  (target: <0.5)
  Median Error: 0.3443

Standard Deviation (spread pattern):
  STD Correlation: 0.7662
  STD MAE:         0.1514

Score Range (max - min, differentiation strength):
  Range Correlation: 0.7602
  Range MAE:         0.3936

MAGNITUDE METRICS: Per-Topic Accuracy
Global Accuracy:

  0%|          | 0/18 [00:00<?, ?it/s]


PRIMARY METRICS: Pattern Learning
Pearson Correlation (pattern + magnitude):
  Mean:   0.8113  (target: >0.85)
  Median: 0.9292
  Std:    0.3075
  Q25-Q75: [0.7978, 0.9777]
  Min:    -0.8678
  % chunks with Pearson > 0.85: 68.8%

Euclidean Distance (magnitude + pattern error):
  Mean:   1.1401  (target: <1.0)
  Median: 0.9927
  Normalized Mean: 0.1297  (as % of target magnitude)

DIFFERENTIATION METRICS: Topic Separation Patterns
Coefficient of Variation (CV):
  CV Correlation: 0.7422  (target: >0.75)
  CV MAE:         0.0358
  Mean Pred CV:   0.1402
  Mean Label CV:  0.1374
  CV Match:       True

MULTI-LABEL METRICS: Topic Co-occurrence Patterns
Pairwise Topic Differences:
  Mean Error:   0.4068  (target: <0.5)
  Median Error: 0.3348

Standard Deviation (spread pattern):
  STD Correlation: 0.7903
  STD MAE:         0.1449

Score Range (max - min, differentiation strength):
  Range Correlation: 0.7838
  Range MAE:         0.3792

MAGNITUDE METRICS: Per-Topic Accuracy
Global Accuracy:

In [56]:
# ============================================================
# CELL 7.10: Comprehensive Evaluation with Enhanced Metrics
# ============================================================

print("\n" + "="*80)
print("COMPREHENSIVE MODEL EVALUATION")
print("="*80)
print("\nEvaluating best model on validation set...")

eval_results = trainer.evaluate()

print("\n" + "="*80)
print("VALIDATION RESULTS SUMMARY")
print("="*80)

# =============================================
# PRIMARY METRICS
# =============================================
print("\n[PRIMARY METRICS - Pattern Learning]")
print("-"*80)

mean_pearson = eval_results.get('eval_mean_pearson', 0)
median_pearson = eval_results.get('eval_median_pearson', 0)
pct_pearson_gt_85 = eval_results.get('eval_pct_pearson_gt_85', 0)

print(f"Pearson Correlation (pattern + magnitude):")
print(f"  Mean:   {mean_pearson:.4f}  {'✓ PASS' if mean_pearson > 0.85 else '⚠ IMPROVE' if mean_pearson > 0.80 else '✗ FAIL'}")
print(f"  Median: {median_pearson:.4f}")
print(f"  % chunks > 0.85: {pct_pearson_gt_85:.1f}%")

mean_euclidean = eval_results.get('eval_mean_euclidean', 0)
print(f"\nEuclidean Distance (magnitude + pattern error):")
print(f"  Mean: {mean_euclidean:.4f}  {'✓ PASS' if mean_euclidean < 1.0 else '✗ FAIL'}")

cv_correlation = eval_results.get('eval_cv_correlation', 0)
cv_mae = eval_results.get('eval_cv_mae', 0)
print(f"\nCV Correlation (differentiation learning):")
print(f"  Correlation: {cv_correlation:.4f}  {'✓ PASS' if cv_correlation > 0.75 else '⚠ IMPROVE' if cv_correlation > 0.70 else '✗ FAIL'}")
print(f"  CV MAE:      {cv_mae:.4f}")

# =============================================
# MULTI-LABEL METRICS
# =============================================
print("\n[MULTI-LABEL METRICS - Topic Co-occurrence]")
print("-"*80)

pairwise_error = eval_results.get('eval_mean_pairwise_error', 0)
print(f"Pairwise Topic Differences:")
print(f"  Mean Error: {pairwise_error:.4f}  {'✓ PASS' if pairwise_error < 0.5 else '✗ FAIL'}")

std_correlation = eval_results.get('eval_std_correlation', 0)
range_correlation = eval_results.get('eval_range_correlation', 0)
print(f"\nVariance Pattern Learning:")
print(f"  STD Correlation:   {std_correlation:.4f}")
print(f"  Range Correlation: {range_correlation:.4f}")

# =============================================
# MAGNITUDE METRICS
# =============================================
print("\n[MAGNITUDE METRICS - Per-Topic Accuracy]")
print("-"*80)

global_mae = eval_results.get('eval_global_mae', 0)
global_rmse = eval_results.get('eval_global_rmse', 0)
global_r2 = eval_results.get('eval_global_r2', 0)

print(f"Global Accuracy:")
print(f"  MAE:  {global_mae:.4f}  {'✓ PASS' if global_mae < 0.8 else '✗ FAIL'}")
print(f"  RMSE: {global_rmse:.4f}")
print(f"  R²:   {global_r2:.4f}")

print(f"\nPer-Topic Breakdown:")
for topic in topics:
    topic_short = topic.split(' & ')[0] if ' & ' in topic else topic
    topic_short = topic_short[:20]

    mae = eval_results.get(f'eval_mae_{topic_short}', 0)
    rmse = eval_results.get(f'eval_rmse_{topic_short}', 0)
    r2 = eval_results.get(f'eval_r2_{topic_short}', 0)

    print(f"  {topic_short:20s}: MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}")

# =============================================
# DIAGNOSTIC METRICS
# =============================================
print("\n[DIAGNOSTIC METRICS]")
print("-"*80)

primary_acc = eval_results.get('eval_primary_topic_accuracy', 0)
top2_acc = eval_results.get('eval_top2_overlap_accuracy', 0)

print(f"Primary Topic (argmax): {primary_acc:.4f}")
print(f"Top-2 Overlap:          {top2_acc:.4f}")

# =============================================
# OVERALL SCORECARD
# =============================================
print("\n" + "="*80)
print("OVERALL SCORECARD")
print("="*80)

targets_met_pct = eval_results.get('eval_targets_met_pct', 0)

# Individual targets
targets = [
    ("Pearson > 0.85", mean_pearson > 0.85),
    ("Euclidean < 1.0", mean_euclidean < 1.0),
    ("CV Corr > 0.75", cv_correlation > 0.75),
    ("Pairwise < 0.5", pairwise_error < 0.5),
    ("Global MAE < 0.8", global_mae < 0.8),
]

print("\nTarget Achievement:")
for target_name, passed in targets:
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"  {target_name:20s}: {status}")

print(f"\nOverall: {targets_met_pct:.0f}% of targets met")

if targets_met_pct >= 80:
    print("\n🎉 EXCELLENT: Model is learning multi-topic patterns very well!")
    print("   Ready for production use or downstream tasks.")
elif targets_met_pct >= 60:
    print("\n👍 GOOD: Model is learning effectively with room for improvement.")
    print("   Consider minor hyperparameter tuning or additional training.")
else:
    print("\n⚠️  NEEDS WORK: Model performance below targets.")
    print("   Recommendations:")
    print("   - Check learning rate (too high/low?)")
    print("   - Increase training epochs")
    print("   - Try CV regularization (Option 5 from LEARNING_SIGNAL_EVALUATION.md)")
    print("   - Verify data quality (check for noise/outliers)")

# =============================================
# COMPARISON TO SBERT BASELINE
# =============================================
print("\n" + "="*80)
print("COMPARISON: BERTje vs. SBERT Labeling")
print("="*80)

print("\nKey Question: Can BERTje replicate SBERT's scoring patterns?")
print("\nSBERT Baseline (theoretical):")
print("  Pearson: 1.000 (perfect self-correlation)")
print("  MAE:     0.000 (no error)")
print("  CV:      1.000 (perfect CV match)")

print(f"\nBERTje Performance:")
print(f"  Pearson: {mean_pearson:.4f} ({mean_pearson*100:.1f}% of SBERT)")
print(f"  MAE:     {global_mae:.4f}")
print(f"  CV Corr: {cv_correlation:.4f} ({cv_correlation*100:.1f}% of SBERT)")

if mean_pearson > 0.90:
    print("\n✓ BERTje closely replicates SBERT patterns (>90% correlation)")
    print("  Model has successfully internalized the semantic space!")
elif mean_pearson > 0.85:
    print("\n✓ BERTje captures SBERT patterns well (>85% correlation)")
    print("  Good performance, minor differences expected due to architecture.")
else:
    print("\n⚠ BERTje correlation with SBERT could be stronger")
    print(f"  Current: {mean_pearson:.4f}, Target: >0.85")

# =============================================
# SAVE EVALUATION RESULTS
# =============================================
print("\n" + "="*80)
print("SAVING EVALUATION RESULTS")
print("="*80)

# Save to JSON
import json
eval_results_path = fs.folders['Model_finetuning'] / 'evaluation_results.json'
with open(eval_results_path, 'w') as f:
    json.dump(eval_results, f, indent=2)

print(f"\nEvaluation metrics saved to:")
print(f"  {eval_results_path}")

# Create summary table
summary = {
    "Primary Metrics": {
        "Pearson Correlation": f"{mean_pearson:.4f}",
        "Euclidean Distance": f"{mean_euclidean:.4f}",
        "CV Correlation": f"{cv_correlation:.4f}",
    },
    "Multi-Label Metrics": {
        "Pairwise Error": f"{pairwise_error:.4f}",
        "STD Correlation": f"{std_correlation:.4f}",
        "Range Correlation": f"{range_correlation:.4f}",
    },
    "Magnitude Metrics": {
        "Global MAE": f"{global_mae:.4f}",
        "Global RMSE": f"{global_rmse:.4f}",
        "Global R²": f"{global_r2:.4f}",
    },
    "Overall": {
        "Targets Met": f"{targets_met_pct:.0f}%",
        "Primary Topic Accuracy": f"{primary_acc:.4f}",
        "Top-2 Overlap": f"{top2_acc:.4f}",
    }
}

summary_path = fs.folders['Model_finetuning'] / 'evaluation_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"  {summary_path}")

print("\n" + "="*80)
print("✓ EVALUATION COMPLETE")
print("="*80)



COMPREHENSIVE MODEL EVALUATION

Evaluating best model on validation set...


  0%|          | 0/18 [00:00<?, ?it/s]


PRIMARY METRICS: Pattern Learning
Pearson Correlation (pattern + magnitude):
  Mean:   0.8113  (target: >0.85)
  Median: 0.9292
  Std:    0.3075
  Q25-Q75: [0.7978, 0.9777]
  Min:    -0.8678
  % chunks with Pearson > 0.85: 68.8%

Euclidean Distance (magnitude + pattern error):
  Mean:   1.1401  (target: <1.0)
  Median: 0.9927
  Normalized Mean: 0.1297  (as % of target magnitude)

DIFFERENTIATION METRICS: Topic Separation Patterns
Coefficient of Variation (CV):
  CV Correlation: 0.7422  (target: >0.75)
  CV MAE:         0.0358
  Mean Pred CV:   0.1402
  Mean Label CV:  0.1374
  CV Match:       True

MULTI-LABEL METRICS: Topic Co-occurrence Patterns
Pairwise Topic Differences:
  Mean Error:   0.4068  (target: <0.5)
  Median Error: 0.3348

Standard Deviation (spread pattern):
  STD Correlation: 0.7903
  STD MAE:         0.1449

Score Range (max - min, differentiation strength):
  Range Correlation: 0.7838
  Range MAE:         0.3792

MAGNITUDE METRICS: Per-Topic Accuracy
Global Accuracy:

In [57]:
# ============================================================
# CELL 7.11: Save Trained Model and Metadata
# ============================================================

from datetime import datetime
# Before trainer.save_model(), add this line:
trainer.args.save_safetensors = False

# Save model
model_save_path = fs.folders["Model_finetuning"] / "SBERTContinuousMultiLabel"
trainer.save_model(str(model_save_path))
tokenizer.save_pretrained(str(model_save_path))

print(f"Model saved to: {model_save_path}")

# Save metadata
metadata = {
    "model_type": "SBERTContinuousMultiLabel",
    "base_model": TRAINING_MODEL,
    "num_topics": len(topics),
    "topics": topics,
    "architecture": "mean_pooling",
    "loss_function": "MSE",
    "training_date": datetime.now().isoformat(),
    "training_samples": len(train_dataset),
    "validation_samples": len(val_dataset),
    "best_epoch": trainer.state.best_model_checkpoint,
    "final_metrics": eval_results,
}

metadata_path = fs.folders["Model_finetuning"] / "SBERTContinuousMultiLabel_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved to: {metadata_path}")
print("\nCheckpoint 7 complete - Continuous regression model trained!")


Model saved to: workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Model_finetuning\SBERTContinuousMultiLabel
Metadata saved to: workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Model_finetuning\SBERTContinuousMultiLabel_metadata.json

Checkpoint 7 complete - Continuous regression model trained!


In [58]:
# ============================================================
# CP7: WORKFLOW SOURCE OVERRIDE (Optional)
# ============================================================

# Load training data from different workflow if needed
CP7_SOURCE = None  # e.g., "workflow_data/pretrained-Slavery_11.10.25_v2"

source_fs = fs.get_source_workflow(CP7_SOURCE) if CP7_SOURCE else fs

if CP7_SOURCE:
    print(f"📂 Loading training data from: {source_fs.root}")
else:
    print(f"📂 Loading training data from: current workflow")
print(f"💾 Saving trained model to: {fs.root}")


📂 Loading training data from: current workflow
💾 Saving trained model to: workflow_data\slavery_Short-slavdict_pretrained_slavery_v4


In [59]:
# ============================================================
# SAVE TRAINED MODEL COMPONENTS (3-way split)
# ============================================================

output_dir = fs.folders["Model_finetuning"]

print("\n" + "="*80)
print("SAVING TRAINED MODEL COMPONENTS")
print("="*80)

# =============================================
# 1. SAVE FULL MODEL (for labeling + continued training)
# =============================================
print("\n[1/3] Saving full classification model...")

full_model_dir = output_dir / "full_model"
full_model_dir.mkdir(exist_ok=True)

# Save complete model state
torch.save(model.state_dict(), full_model_dir / "pytorch_model.bin")

# Save model config
model_config = {
    "model_type": "SBERTMultiOrdinal",
    "base_model": TRAINING_MODEL,
    "num_topics": len(topics),
    "num_classes": 3,
    "topics": topics,
    "hidden_size": model.config.hidden_size,
    "trained_on": str(output_dir.parent.name),
    "training_timestamp": datetime.now().isoformat()
}
with open(full_model_dir / "model_config.json", "w") as f:
    json.dump(model_config, f, indent=2)

# Save label mapping
label_mapping = {
    "topics": topics,
    "num_topics": len(topics),
    "topic2idx": {t: i for i, t in enumerate(topics)},
    "idx2topic": {i: t for i, t in enumerate(topics)}
}
with open(full_model_dir / "label_mapping.json", "w") as f:
    json.dump(label_mapping, f, indent=2)

# Save tokenizer
tokenizer.save_pretrained(full_model_dir)

print(f"Full model: {full_model_dir}")
print(f"  - pytorch_model.bin (complete state)")
print(f"  - model_config.json")
print(f"  - label_mapping.json")

# =============================================
# 2. SAVE TRAINED ENCODER (for vector building!)
# =============================================
print("\n[2/3] Saving TRAINED encoder (for embeddings)...")

trained_encoder_dir = output_dir / "trained_encoder"
trained_encoder_dir.mkdir(exist_ok=True)

# Extract BERT encoder from trained model
encoder_state_dict = {}
for key, value in model.state_dict().items():
    if key.startswith('bert.'):
        new_key = key[5:]  # Remove 'bert.' prefix
        encoder_state_dict[new_key] = value

# Create clean encoder with trained weights
trained_encoder = AutoModel.from_pretrained(TRAINING_MODEL)
trained_encoder.load_state_dict(encoder_state_dict)

# Save as HuggingFace model
trained_encoder.save_pretrained(trained_encoder_dir)
tokenizer.save_pretrained(trained_encoder_dir)

# Add SentenceTransformer compatibility
st_config = {
    "max_seq_length": 512,
    "do_lower_case": False,
    "word_embedding_dimension": model.config.hidden_size,
}
with open(trained_encoder_dir / "sentence_transformers_config.json", "w") as f:
    json.dump(st_config, f, indent=2)

# Add training metadata
training_metadata = {
    "trained": True,
    "trained_on_corpus": str(output_dir.parent.name),
    "training_date": datetime.now().isoformat(),
    "base_model": TRAINING_MODEL,
    "topics": topics,
    "training_steps": trainer.state.global_step,
    "final_loss": eval_results.get("eval_loss", None),
    "note": "This encoder has been fine-tuned on topic classification. Use for building domain-adapted vectors."
}
with open(trained_encoder_dir / "training_metadata.json", "w") as f:
    json.dump(training_metadata, f, indent=2)

print(f"Trained encoder: {trained_encoder_dir}")
print(f"  - pytorch_model.bin (TRAINED weights)")
print(f"  - sentence_transformers_config.json")
print(f"  - training_metadata.json")
print(f"  This encoder understands your topics!")

# =============================================
# 3. SAVE BASE ENCODER (for reference)
# =============================================
print("\n[3/3] Saving base encoder (reference)...")

base_encoder_dir = output_dir / "base_encoder"
base_encoder_dir.mkdir(exist_ok=True)

# Save untrained base model
base_encoder = AutoModel.from_pretrained(TRAINING_MODEL)
base_encoder.save_pretrained(base_encoder_dir)
tokenizer.save_pretrained(base_encoder_dir)

# Add metadata
base_metadata = {
    "trained": False,
    "model": TRAINING_MODEL,
    "note": "Original pretrained weights (no topic training)"
}
with open(base_encoder_dir / "metadata.json", "w") as f:
    json.dump(base_metadata, f, indent=2)

print(f"Base encoder: {base_encoder_dir}")

# =============================================
# 4. SAVE TRAINING METRICS
# =============================================
print("\n[4/4] Saving training metrics...")

metrics = {
    "final_eval": eval_results,
    "training_history": trainer.state.log_history,
    "topics": topics,
    "num_train_samples": len(train_dataset),
    "num_val_samples": len(val_dataset),
    "training_args": {
        "num_epochs": trainer.args.num_train_epochs,
        "batch_size": trainer.args.per_device_train_batch_size,
        "learning_rate": trainer.args.learning_rate,
    }
}

with open(output_dir / "training_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Metrics: {output_dir / 'training_metrics.json'}")

# =============================================
# SUMMARY
# =============================================
print("\n" + "="*80)
print("MODEL SAVING COMPLETE")
print("="*80)

print(f"\nAll models saved to: {output_dir}")

print(f"\nUSAGE GUIDE:")
print(f"\n1. For BUILDING TOPIC VECTORS (Checkpoint 4):")
print(f"   Load: {trained_encoder_dir}")
print(f"   st_model = SentenceTransformer('{trained_encoder_dir}')")
print(f"   This is TRAINED on your topics!")

print(f"\n2. For BERTJE LABELING (Checkpoint 8):")
print(f"   Load: {full_model_dir / 'pytorch_model.bin'}")

print(f"\n3. For CONTINUED TRAINING:")
print(f"   Load: {full_model_dir / 'pytorch_model.bin'}")

fs.save_config("checkpoint7_trained")

print(f"\nCheckpoint 7 complete - Model saved in 3 formats!")


SAVING TRAINED MODEL COMPONENTS

[1/3] Saving full classification model...
Full model: workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Model_finetuning\full_model
  - pytorch_model.bin (complete state)
  - model_config.json
  - label_mapping.json

[2/3] Saving TRAINED encoder (for embeddings)...
Trained encoder: workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Model_finetuning\trained_encoder
  - pytorch_model.bin (TRAINED weights)
  - sentence_transformers_config.json
  - training_metadata.json
  This encoder understands your topics!

[3/3] Saving base encoder (reference)...
Base encoder: workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Model_finetuning\base_encoder

[4/4] Saving training metrics...
Metrics: workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Model_finetuning\training_metrics.json

MODEL SAVING COMPLETE

All models saved to: workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Model_finetuning

USAGE GUIDE:

1. For BUILDING

---
✅ **CHECKPOINT 7 COMPLETE** - SBERT multi-label model trained
---

---
# CHECKPOINT 8: BERTJE Labeling
---

Use trained BERTJE model (or base model) to label entire corpus.

**Options**:
- Use trained model from CHECKPOINT 7 (recommended)
- Use base model if training was skipped (poor results)

**Outputs**: `Bertje_labeling/` with confidence-classified predictions

In [12]:
# ============================================================
# CP8: WORKFLOW SOURCE OVERRIDE (Optional)
# ============================================================

# Load corpus and model from different workflows
CP8_CORPUS_SOURCE = None  # For corpus to label
CP8_MODEL_SOURCE = None # For trained BERTje model (workflow root folder)

corpus_fs = fs.get_source_workflow(CP8_CORPUS_SOURCE) if CP8_CORPUS_SOURCE else fs

# ============================================================
# LOAD SBERT MODEL (if needed for embeddings)
# ============================================================
if 'st_model' not in globals():
    # Load SBERT for embedding generation if not already loaded
    if CP8_MODEL_SOURCE:
        model_path = Path(CP8_MODEL_SOURCE)
        sbert_path = model_path / 'sbert_base'
        if sbert_path.exists():
            st_model = SentenceTransformer(str(sbert_path))
            print(f"📂 Loading SBERT from: {sbert_path}")
        else:
            st_model = SentenceTransformer(CONFIG['model']['base_model_name'])
            print(f"📂 Loading SBERT from: default")
    else:
        st_model = SentenceTransformer(CONFIG['model']['base_model_name'])
        print(f"📂 Loading SBERT from: default ({CONFIG['model']['base_model_name']})")
else:
    print(f"📂 Using existing SBERT model")

if CP8_CORPUS_SOURCE:
    print(f"📂 Loading corpus from: {corpus_fs.root}")
else:
    print(f"📂 Loading corpus from: current workflow")

if CP8_MODEL_SOURCE:
    print(f"📂 Loading BERTje model from: {CP8_MODEL_SOURCE}")
else:
    print(f"📂 Loading BERTje model from: {fs.folders['Model_finetuning']}")

print(f"💾 Saving predictions to: {fs.root}")


📂 Loading SBERT from: default (NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers)
📂 Loading corpus from: current workflow
📂 Loading BERTje model from: C:\Users\Home\policy-analysis\workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Model_finetuning
💾 Saving predictions to: C:\Users\Home\policy-analysis\workflow_data\slavery_Short-slavdict_pretrained_slavery_v4


In [13]:
# ============================================================
# # CELL 8.1: SETUP & MODEL ARCHITECTURE
# ============================================================

print("="*80)
print("CHECKPOINT 8: LABEL FULL CORPUS WITH FINE-TUNED MODEL")
print("="*80)

# Check if transformers is available
try:
    from transformers import AutoTokenizer, AutoModel, AutoConfig
    import torch
    import torch.nn as nn
    from transformers.modeling_outputs import SequenceClassifierOutput
    BERT_AVAILABLE = True
    print("✓ Transformers library available")
except ImportError:
    print("⚠ Transformers not available")
    print("  Install: pip install transformers torch")
    BERT_AVAILABLE = False

if BERT_AVAILABLE:
    # Define the model architecture (must match training)
    class SBERTContinuousMultiLabel(nn.Module):
        """
        SBERT with continuous multi-label regression.

        Architecture:
            Text -> BERT -> Mean pooling -> N independent regression heads

        Each head predicts a continuous score for its topic.
        """
        def __init__(self, model_name: str, num_topics: int = 4):
            super().__init__()

            self.num_topics = num_topics

            # Load pre-trained BERT
            self.config = AutoConfig.from_pretrained(model_name)
            self.bert = AutoModel.from_pretrained(model_name, config=self.config)

            hidden_size = self.config.hidden_size

            # Create N independent regression heads (1 output per topic)
            self.topic_heads = nn.ModuleList([
                nn.Linear(hidden_size, 1)
                for _ in range(num_topics)
            ])

            self.dropout = nn.Dropout(0.2)

        def mean_pooling(self, token_embeddings, attention_mask):
            """SBERT-style mean pooling."""
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            return sum_embeddings / sum_mask

        def forward(self, input_ids, attention_mask, labels=None):
            # Get BERT embeddings
            outputs = self.bert(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True
            )

            # Mean pooling
            sentence_embedding = self.mean_pooling(outputs.last_hidden_state, attention_mask)
            sentence_embedding = self.dropout(sentence_embedding)

            # Predict continuous scores for each topic
            topic_predictions = []
            for head in self.topic_heads:
                pred = head(sentence_embedding)  # [batch, 1] -> unbounded
                topic_predictions.append(pred)

            # Stack: [batch, num_topics]
            logits = torch.cat(topic_predictions, dim=1)

            # During inference, clamp to reasonable range
            if not self.training:
                logits = torch.clamp(logits, 0.0, 10.0)

            loss = None
            if labels is not None:
                loss_fct = nn.MSELoss()
                loss = loss_fct(logits, labels.float())

            return SequenceClassifierOutput(
                loss=loss,
                logits=logits,
                hidden_states=outputs.hidden_states,
                attentions=outputs.attentions,
            )

    print("\n✓ SBERTContinuousMultiLabel architecture defined")
    print("  - Continuous multi-label regression")
    print("  - Mean pooling (SBERT-style)")
    print("  - 4 independent regression heads")
    print("  - Output range: [0, 10] (dot product scale)")


CHECKPOINT 8: LABEL FULL CORPUS WITH FINE-TUNED MODEL
✓ Transformers library available

✓ SBERTContinuousMultiLabel architecture defined
  - Continuous multi-label regression
  - Mean pooling (SBERT-style)
  - 4 independent regression heads
  - Output range: [0, 10] (dot product scale)


In [14]:
# ============================================================
# CELL 8.2: LOAD TRAINED MODEL
# ============================================================

if BERT_AVAILABLE:
    print(f"\n{'='*80}")
    print("LOADING TRAINED MODEL")
    print(f"{'='*80}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # Determine model source
    if 'CONFIG' in globals() and CONFIG.get('model', {}).get('use_pretrained') and CONFIG.get('paths', {}).get('pretrained_model_path'):
        model_dir = Path(CONFIG['paths']['pretrained_model_path'])
        print(f"\n📂 Loading from CONFIG pretrained path:")
        print(f"  {model_dir}")
    elif 'fs' in globals() and hasattr(fs, 'folders'):
        model_dir = fs.folders['Model_finetuning']
        print(f"\n📂 Loading from current workflow:")
        print(f"  {model_dir}")
    else:
        # Manual fallback
        model_dir = Path(input("Enter path to Model_finetuning directory: "))
        print(f"\n📂 Manual path:")
        print(f"  {model_dir}")

    # Try to load the full model (saved by Checkpoint 7)
    full_model_path = model_dir / 'full_model'
    model_config_path = full_model_path / 'model_config.json'

    if full_model_path.exists() and model_config_path.exists():
        print(f"\n✓ Found trained model")

        # Load model config
        with open(model_config_path, 'r') as f:
            model_config = json.load(f)

        print(f"\nModel Configuration:")
        print(f"  Base model: {model_config['base_model']}")
        print(f"  Topics: {model_config['num_topics']}")
        print(f"  Trained on: {model_config.get('trained_on', 'unknown')}")

        # Instantiate model
        print(f"\nInstantiating model...")
        bertje_model = SBERTContinuousMultiLabel(
            model_name=model_config['base_model'],
            num_topics=model_config['num_topics']
        )

        # Load trained weights
        weights_path = full_model_path / 'pytorch_model.bin'
        if weights_path.exists():
            print(f"Loading trained weights from: {weights_path.name}")
            state_dict = torch.load(weights_path, map_location=device)
            bertje_model.load_state_dict(state_dict)
            bertje_model = bertje_model.to(device)
            bertje_model.eval()
            print(f"✓ Model loaded in evaluation mode")
        else:
            print(f"❌ ERROR: Weights not found at {weights_path}")
            BERT_AVAILABLE = False

        # Load tokenizer
        print(f"\nLoading tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained(full_model_path)
        print(f"✓ Tokenizer loaded")

        # Store topic names
        bertje_topics = model_config['topics']
        print(f"\nTopics ({len(bertje_topics)}):")
        for i, topic in enumerate(bertje_topics):
            print(f"  {i+1}. {topic}")

        BERTJE_MODEL_LOADED = True

    else:
        print(f"\n❌ ERROR: Model not found at {model_dir}")
        print(f"\nExpected structure:")
        print(f"  {model_dir}/")
        print(f"  └── full_model/")
        print(f"      ├── pytorch_model.bin")
        print(f"      ├── model_config.json")
        print(f"      └── tokenizer files")

        if model_dir.exists():
            print(f"\nActual contents of {model_dir}:")
            for item in model_dir.iterdir():
                print(f"  - {item.name}")
        else:
            print(f"\n⚠ Directory does not exist: {model_dir}")

        BERTJE_MODEL_LOADED = False
else:
    BERTJE_MODEL_LOADED = False



LOADING TRAINED MODEL
Device: cuda

📂 Loading from current workflow:
  C:\Users\Home\policy-analysis\workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Model_finetuning

✓ Found trained model

Model Configuration:
  Base model: NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers
  Topics: 4
  Trained on: slavery_Short-slavdict_pretrained_slavery_v4

Instantiating model...
Loading trained weights from: pytorch_model.bin
✓ Model loaded in evaluation mode

Loading tokenizer...
✓ Tokenizer loaded

Topics (4):
  1. Colonial Systems
  2. Heritage & Memory
  3. Historical Slavery
  4. Modern Racism & Discrimination


In [15]:
# ============================================================
# CELL 8.3: LOAD CORPUS
# ============================================================

if BERTJE_MODEL_LOADED:
    print(f"\n{'='*80}")
    print("LOADING CORPUS FOR LABELING")
    print(f"{'='*80}")

    # Determine corpus source
    corpus_source_workflow = None  # Set to workflow name if loading from different workflow

    if corpus_source_workflow:
        # Load from specified workflow
        corpus_fs = fs.get_source_workflow(corpus_source_workflow)
        corpus_path = corpus_fs.folders['Other_data'] / 'chunked_corpus.csv'
        print(f"Loading corpus from: {corpus_source_workflow}")
    else:
        # Load from current workflow
        if 'fs' in globals() and hasattr(fs, 'folders'):
            corpus_path = fs.folders['Other_data'] / 'chunked_corpus.csv'
        else:
            corpus_path = Path(input("Enter path to chunked_corpus.csv: "))
        print(f"Loading corpus from: current workflow")

    if corpus_path.exists():
        corpus_df = pd.read_csv(corpus_path)
        print(f"✓ Loaded corpus: {len(corpus_df)} chunks")

        # Prepare text for model
        if 'text_for_scoring' in corpus_df.columns:
            texts = corpus_df['text_for_scoring'].fillna('').tolist()
        elif 'raw_text' in corpus_df.columns:
            print('WARNING: text_for_scoring missing, falling back to raw_text')
            texts = corpus_df['raw_text'].fillna('').tolist()
        else:
            print("❌ ERROR: No text column found (text_for_scoring or raw_text)")
            BERTJE_MODEL_LOADED = False

        print(f"✓ Prepared {len(texts)} texts for labeling")

    else:
        print(f"❌ ERROR: Corpus not found at {corpus_path}")
        BERTJE_MODEL_LOADED = False



LOADING CORPUS FOR LABELING
Loading corpus from: current workflow
✓ Loaded corpus: 2840 chunks
✓ Prepared 2840 texts for labeling


In [16]:
# ============================================================
# CELL 8.4: PREDICT ON CORPUS
# ============================================================

if BERTJE_MODEL_LOADED:
    print(f"\n{'='*80}")
    print("GENERATING PREDICTIONS")
    print(f"{'='*80}")

    def predict_continuous_scores(texts, batch_size=32):
        """
        Predict continuous scores for each topic.

        Returns:
            scores: [N, num_topics] - continuous scores [0-10 range]
        """
        all_scores = []

        with torch.no_grad():
            for i in tqdm(range(0, len(texts), batch_size), desc="Predicting"):
                batch = texts[i:i+batch_size]

                # Tokenize
                inputs = tokenizer(
                    batch,
                    padding=True,
                    truncation=True,
                    max_length=512,
                    return_tensors="pt"
                )
                inputs = {k: v.to(device) for k, v in inputs.items()}

                # Forward pass
                outputs = bertje_model(**inputs)
                scores = outputs.logits  # [batch, num_topics], clamped [0-10]

                all_scores.append(scores.cpu().numpy())

        # Concatenate batches
        all_scores = np.vstack(all_scores)  # [N, num_topics]

        return all_scores

    # Get predictions
    print(f"\nPredicting on {len(texts)} chunks...")
    topic_scores = predict_continuous_scores(texts, batch_size=32)

    print(f"\n✓ Prediction complete")
    print(f"  Shape: {topic_scores.shape}")
    print(f"  Range: [{topic_scores.min():.2f}, {topic_scores.max():.2f}]")

    # =============================================
    # Add scores to dataframe
    # =============================================

    print(f"\nAdding topic scores to dataframe...")

    # Per-topic scores
    for i, topic in enumerate(bertje_topics):
        topic_short = topic.split(' & ')[0] if ' & ' in topic else topic
        corpus_df[f'bertje_score_{topic_short}'] = topic_scores[:, i]

    # Calculate statistics
    print(f"\nScore distribution per topic:")
    for i, topic in enumerate(bertje_topics):
        topic_short = topic.split(' & ')[0] if ' & ' in topic else topic
        scores = topic_scores[:, i]
        print(f"\n  {topic_short}:")
        print(f"    Mean: {scores.mean():.2f}")
        print(f"    Std:  {scores.std():.2f}")
        print(f"    Min:  {scores.min():.2f}")
        print(f"    Q25:  {np.quantile(scores, 0.25):.2f}")
        print(f"    Med:  {np.median(scores):.2f}")
        print(f"    Q75:  {np.quantile(scores, 0.75):.2f}")
        print(f"    Max:  {scores.max():.2f}")

    # =============================================
    # Calculate derived metrics
    # =============================================

    print(f"\nCalculating derived metrics...")

    # Primary topic (argmax)
    primary_idx = np.argmax(topic_scores, axis=1)
    corpus_df['bertje_primary_topic'] = [bertje_topics[idx] for idx in primary_idx]

    # Max score and margin
    sorted_scores = np.sort(topic_scores, axis=1)[:, ::-1]  # Sort descending
    corpus_df['bertje_max_score'] = sorted_scores[:, 0]
    corpus_df['bertje_score_margin'] = sorted_scores[:, 0] - sorted_scores[:, 1]

    # Primary topic score
    primary_scores = topic_scores[np.arange(len(topic_scores)), primary_idx]
    corpus_df['bertje_primary_score'] = primary_scores

    # Coefficient of Variation (differentiation metric)
    cvs = []
    for row in topic_scores:
        cv = np.std(row) / (np.mean(row) + 1e-8)
        cvs.append(cv)
    corpus_df['bertje_cv'] = cvs

    print(f"✓ Added derived metrics:")
    print(f"  - bertje_primary_topic")
    print(f"  - bertje_max_score")
    print(f"  - bertje_score_margin")
    print(f"  - bertje_primary_score")
    print(f"  - bertje_cv (differentiation)")

    # =============================================
    # Confidence classification
    # =============================================

    print(f"\nCalculating confidence tiers...")

    # Calculate percentiles for thresholds
    score_p75 = np.percentile(primary_scores, 75)
    score_p50 = np.percentile(primary_scores, 50)
    score_p25 = np.percentile(primary_scores, 25)

    margin_p75 = np.percentile(corpus_df['bertje_score_margin'], 75)
    margin_p50 = np.percentile(corpus_df['bertje_score_margin'], 50)

    cv_p75 = np.percentile(cvs, 75)

    print(f"\nThresholds (from data distribution):")
    print(f"  Score  - P75: {score_p75:.2f}, P50: {score_p50:.2f}, P25: {score_p25:.2f}")
    print(f"  Margin - P75: {margin_p75:.2f}, P50: {margin_p50:.2f}")
    print(f"  CV     - P75: {cv_p75:.2f}")

    def classify_confidence(row):
        """Classify confidence based on score, margin, and CV."""
        score = row['bertje_primary_score']
        margin = row['bertje_score_margin']
        cv = row['bertje_cv']

        # High confidence: high score + high margin + high differentiation
        if score >= score_p75 and margin >= margin_p75 and cv >= cv_p75:
            return 'high'
        # Medium confidence: above median score + decent margin
        elif score >= score_p50 and margin >= margin_p50:
            return 'medium'
        # Low confidence: above bottom quartile
        elif score >= score_p25:
            return 'low'
        # Very low: bottom quartile
        else:
            return 'very_low'

    corpus_df['bertje_confidence'] = corpus_df.apply(classify_confidence, axis=1)

    print(f"\n✓ Confidence distribution:")
    print(corpus_df['bertje_confidence'].value_counts().sort_index())

    print(f"\n✓ Primary topic distribution:")
    print(corpus_df['bertje_primary_topic'].value_counts())



GENERATING PREDICTIONS

Predicting on 2840 chunks...


Predicting: 100%|██████████| 89/89 [00:33<00:00,  2.66it/s]


✓ Prediction complete
  Shape: (2840, 4)
  Range: [1.04, 7.99]

Adding topic scores to dataframe...

Score distribution per topic:

  Colonial Systems:
    Mean: 4.16
    Std:  0.96
    Min:  1.04
    Q25:  3.45
    Med:  4.09
    Q75:  4.78
    Max:  7.56

  Heritage:
    Mean: 4.25
    Std:  0.89
    Min:  1.63
    Q25:  3.60
    Med:  4.14
    Q75:  4.77
    Max:  7.75

  Historical Slavery:
    Mean: 4.49
    Std:  1.21
    Min:  1.32
    Q25:  3.51
    Med:  4.34
    Q75:  5.39
    Max:  7.99

  Modern Racism:
    Mean: 4.33
    Std:  0.97
    Min:  1.91
    Q25:  3.63
    Med:  4.23
    Q75:  4.95
    Max:  7.63

Calculating derived metrics...
✓ Added derived metrics:
  - bertje_primary_topic
  - bertje_max_score
  - bertje_score_margin
  - bertje_primary_score
  - bertje_cv (differentiation)

Calculating confidence tiers...

Thresholds (from data distribution):
  Score  - P75: 5.87, P50: 5.08, P25: 4.44
  Margin - P75: 1.03, P50: 0.56
  CV     - P75: 0.18

✓ Confidence distribu

In [17]:
# ============================================================
# CELL 8.5: SAVE LABELED CORPUS
# ============================================================

if BERTJE_MODEL_LOADED:
    print(f"\n{'='*80}")
    print("SAVING LABELED CORPUS")
    print(f"{'='*80}")

    # Save full labeled corpus
    if 'fs' in globals() and hasattr(fs, 'folders'):
        output_dir = fs.folders['Bertje_labeling']
    else:
        output_dir = model_dir

    output_path = output_dir / 'bertje_labeled_corpus.csv'
    corpus_df.to_csv(output_path, index=False)

    print(f"\n✓ Saved labeled corpus:")
    print(f"  {output_path}")
    print(f"  {len(corpus_df)} chunks")
    print(f"  {len([c for c in corpus_df.columns if c.startswith('bertje_')])} BERTje columns")

    # Create confidence-split files
    print(f"\nCreating confidence-split files...")

    high_conf = corpus_df[corpus_df['bertje_confidence'] == 'high']
    medium_conf = corpus_df[corpus_df['bertje_confidence'] == 'medium']
    low_conf = corpus_df[corpus_df['bertje_confidence'] == 'low']
    very_low_conf = corpus_df[corpus_df['bertje_confidence'] == 'very_low']

    high_path = output_dir / 'bertje_high_confidence.csv'
    medium_path = output_dir / 'bertje_medium_confidence.csv'
    low_path = output_dir / 'bertje_low_confidence.csv'
    very_low_path = output_dir / 'bertje_very_low_confidence.csv'

    high_conf.to_csv(high_path, index=False)
    medium_conf.to_csv(medium_path, index=False)
    low_conf.to_csv(low_path, index=False)
    very_low_conf.to_csv(very_low_path, index=False)

    print(f"\n✓ Confidence-split files:")
    print(f"  High:     {high_path.name} ({len(high_conf)} chunks, {len(high_conf)/len(corpus_df)*100:.1f}%)")
    print(f"  Medium:   {medium_path.name} ({len(medium_conf)} chunks, {len(medium_conf)/len(corpus_df)*100:.1f}%)")
    print(f"  Low:      {low_path.name} ({len(low_conf)} chunks, {len(low_conf)/len(corpus_df)*100:.1f}%)")
    print(f"  Very Low: {very_low_path.name} ({len(very_low_conf)} chunks, {len(very_low_conf)/len(corpus_df)*100:.1f}%)")

    # Create per-topic files (high confidence only)
    print(f"\nCreating per-topic files (high confidence)...")

    for topic in bertje_topics:
        topic_df = high_conf[high_conf['bertje_primary_topic'] == topic]
        topic_short = topic.split(' & ')[0] if ' & ' in topic else topic
        topic_path = output_dir / f'bertje_topic_{topic_short}_high_conf.csv'
        topic_df.to_csv(topic_path, index=False)
        print(f"  {topic_short:20s}: {len(topic_df)} chunks")

    # Save summary statistics
    print(f"\nCreating summary statistics...")

    summary = {
        "total_chunks": len(corpus_df),
        "topics": bertje_topics,
        "confidence_distribution": corpus_df['bertje_confidence'].value_counts().to_dict(),
        "topic_distribution": corpus_df['bertje_primary_topic'].value_counts().to_dict(),
        "score_statistics": {
            "mean": float(corpus_df['bertje_primary_score'].mean()),
            "std": float(corpus_df['bertje_primary_score'].std()),
            "min": float(corpus_df['bertje_primary_score'].min()),
            "max": float(corpus_df['bertje_primary_score'].max()),
            "median": float(corpus_df['bertje_primary_score'].median()),
        },
        "margin_statistics": {
            "mean": float(corpus_df['bertje_score_margin'].mean()),
            "median": float(corpus_df['bertje_score_margin'].median()),
        },
        "cv_statistics": {
            "mean": float(corpus_df['bertje_cv'].mean()),
            "median": float(corpus_df['bertje_cv'].median()),
        },
        "thresholds": {
            "score_p75": float(score_p75),
            "score_p50": float(score_p50),
            "score_p25": float(score_p25),
            "margin_p75": float(margin_p75),
            "margin_p50": float(margin_p50),
            "cv_p75": float(cv_p75),
        },
        "model_info": {
            "base_model": model_config['base_model'],
            "trained_on": model_config.get('trained_on', 'unknown'),
            "num_topics": len(bertje_topics),
        }
    }

    summary_path = output_dir / 'bertje_labeling_summary.json'
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print(f"\n✓ Summary statistics: {summary_path.name}")

    print(f"\n{'='*80}")
    print("✓ CHECKPOINT 8 COMPLETE - CORPUS LABELED")
    print(f"{'='*80}")

    print(f"\nNext steps:")
    print(f"  1. Review high-confidence labels for quality")
    print(f"  2. Use labeled corpus for downstream analysis")
    print(f"  3. Consider active learning on low-confidence chunks")

else:
    print("\n⚠ Model not loaded - skipping prediction")



SAVING LABELED CORPUS

✓ Saved labeled corpus:
  C:\Users\Home\policy-analysis\workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Bertje_labeling\bertje_labeled_corpus.csv
  2840 chunks
  10 BERTje columns

Creating confidence-split files...

✓ Confidence-split files:
  High:     bertje_high_confidence.csv (225 chunks, 7.9%)
  Medium:   bertje_medium_confidence.csv (703 chunks, 24.8%)
  Low:      bertje_low_confidence.csv (1202 chunks, 42.3%)
  Very Low: bertje_very_low_confidence.csv (710 chunks, 25.0%)

Creating per-topic files (high confidence)...
  Colonial Systems    : 15 chunks
  Heritage            : 17 chunks
  Historical Slavery  : 87 chunks
  Modern Racism       : 106 chunks

Creating summary statistics...

✓ Summary statistics: bertje_labeling_summary.json

✓ CHECKPOINT 8 COMPLETE - CORPUS LABELED

Next steps:
  1. Review high-confidence labels for quality
  2. Use labeled corpus for downstream analysis
  3. Consider active learning on low-confidence chunks


✅ **CHECKPOINT 8 COMPLETE** - Corpus labeled with BERTJE

**Resume**: Load from `Bertje_labeling/bert_labeled_all.csv`

---
# CHECKPOINT 9: Visualizations
---

Generate interactive visualizations for analysis.

In [18]:
# ============================================================
# CP9: WORKFLOW SOURCE OVERRIDE (Optional)
# ============================================================

# Load visualization data from different workflow if needed
# Useful for:
#   - Visualizing results from a previous workflow run
#   - Comparing different model versions
#   - Analyzing production data in dev environment

CP9_SOURCE = None  # e.g., "workflow_data/Finetuned_Slavery-Slavery-policy_11.01.25_v1"

source_fs = fs.get_source_workflow(CP9_SOURCE) if CP9_SOURCE else fs

if CP9_SOURCE:
    print(f"📂 Loading visualization data from: {source_fs.root}")
    print(f"   Data sources:")
    print(f"   - BERTJE predictions: {source_fs.folders.get('Bertje_labeling') or 'N/A'}")
    print(f"   - Cosine scores: {source_fs.folders.get('Cosine_labeling') or 'N/A'}")
    print(f"   - Dictionary: {source_fs.folders.get('Dictionary') or 'N/A'}")
    print(f"   - Training metrics: {source_fs.folders.get('Model_finetuning') or 'N/A'}")
else:
    print(f"📂 Loading visualization data from: current workflow")
    
print(f"💾 Saving visualizations to: {fs.folders.get('Visuals') or fs.root / 'Visuals'}")


📂 Loading visualization data from: current workflow
💾 Saving visualizations to: C:\Users\Home\policy-analysis\workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Visuals


In [19]:
# ============================================================
# CELL 9.1: SETUP VISUALIZATION LIBRARIES
# ============================================================

try:
    # Core visualization libraries
    import matplotlib.pyplot as plt
    import seaborn as sns
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    
    # Machine learning and dimensionality reduction
    from sklearn.decomposition import PCA
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler
    
    # Statistical analysis for score comparison
    from scipy.stats import pearsonr, spearmanr
    
    # Configure plotting defaults
    sns.set_style("whitegrid")
    plt.rcParams['figure.figsize'] = (14, 8)
    
    print("✓ Core visualization libraries loaded")
    print("  - matplotlib, seaborn, plotly")
    print("  - sklearn (PCA, KMeans, StandardScaler)")
    print("  - scipy.stats (correlation analysis)")
    
    VIZ_AVAILABLE = True
    
except ImportError as e:
    print(f"⚠ Visualization libraries not available: {e}")
    print("  Install: pip install matplotlib seaborn plotly scikit-learn scipy")
    VIZ_AVAILABLE = False

✓ Core visualization libraries loaded
  - matplotlib, seaborn, plotly
  - sklearn (PCA, KMeans, StandardScaler)
  - scipy.stats (correlation analysis)


In [ ]:
# ============================================================

# CELL 9.2: CONFIGURATION & DATA LOADING

# ============================================================



if VIZ_AVAILABLE:

    print(f"\n{'='*70}")

    print("CHECKPOINT 9: METHODOLOGY VALIDATION & MODEL COMPARISON")

    print(f"{'='*70}")



    # --------------------------------------------------------

    # Configuration

    # --------------------------------------------------------



    # Analysis Parameters

    MIN_SCORE_THRESHOLD = 0.30  # Minimum score for high-confidence chunks

    N_CLUSTERS = 5  # Number of K-means clusters per topic

    PCA_COMPONENTS_2D = 2  # Components for 2D visualization

    PCA_COMPONENTS_3D = 3  # Components for 3D visualization



    # Output Configuration

    visuals_path_viz = fs.folders.get('Visuals')

    if visuals_path_viz is None:

        visuals_path_viz = fs.root / 'Visuals'

    visuals_path_viz.mkdir(exist_ok=True)



    print(f"\nConfiguration:")

    print(f"  Workflow: {source_fs.root.name}")

    print(f"\nAnalysis Parameters:")

    print(f"  Min Score Threshold: {MIN_SCORE_THRESHOLD}")

    print(f"  K-means Clusters: {N_CLUSTERS}")

    print(f"  PCA Components (2D/3D): {PCA_COMPONENTS_2D}/{PCA_COMPONENTS_3D}")



    # --------------------------------------------------------

    # Load Scored Data

    # --------------------------------------------------------



    print(f"\n{'='*70}")

    print("LOADING SCORED DATA")

    print(f"{'='*70}")



    # Priority 1: Try to load Bertje labeled corpus (has both cosine + BERTJE scores)
    # Priority 1: Load cosine scores (always needed for score_* columns)
    cosine_path = source_fs.folders.get('Cosine_labeling')
    
    if cosine_path and (cosine_path / 'scores_all_labeled.csv').exists():
        print(" Loading cosine scores...")
        df_viz = pd.read_csv(cosine_path / 'scores_all_labeled.csv')
        print(f"  ✓ Loaded {len(df_viz)} chunks with cosine scores")
        has_bertje_scores = False
    else:
        print(f" ✗ ERROR: No cosine scores found")
        print(f"    Checked: {cosine_path / 'scores_all_labeled.csv' if cosine_path else 'N/A'}")
        print(f"    Please run CHECKPOINT 5 (cosine scoring) first")
        VIZ_AVAILABLE = False
        df_viz = None
    
    # Priority 2: Try to merge BERTJE predictions if available
    if df_viz is not None:
        bertje_path = source_fs.folders.get('Bertje_labeling')
        
        if bertje_path and (bertje_path / 'bertje_labeled_corpus.csv').exists():
            print(" Loading BERTJE predictions...")
            df_bertje = pd.read_csv(bertje_path / 'bertje_labeled_corpus.csv')
            
            # Merge on chunk_uid or raw_text
            if 'chunk_uid' in df_viz.columns and 'chunk_uid' in df_bertje.columns:
                merge_key = 'chunk_uid'
            elif 'chunk_id' in df_viz.columns and 'chunk_uid' in df_bertje.columns:
                df_viz.rename(columns={'chunk_id': 'chunk_uid'}, inplace=True)
                merge_key = 'chunk_uid'
            else:
                merge_key = 'raw_text'
            
            print(f"  Merging on: {merge_key}")
            
            # Select BERTJE columns to merge
            bertje_cols = [col for col in df_bertje.columns if col.startswith('bertje_')]
            bertje_cols_to_merge = [merge_key] + bertje_cols
            
            df_viz = df_viz.merge(
                df_bertje[bertje_cols_to_merge],
                on=merge_key,
                how='left'
            )
            
            has_bertje_scores = True
            print(f"  ✓ Merged BERTJE scores: {len([c for c in df_viz.columns if c.startswith('bertje_score_')])} topic columns")
        else:
            print(f" ⚠ BERTJE predictions not available (run CHECKPOINT 8 first)")
            has_bertje_scores = False
    if VIZ_AVAILABLE and df_viz is not None:

        # --------------------------------------------------------

        # Load Dictionary

        # --------------------------------------------------------



        dict_path = source_fs.folders.get('Dictionary') / 'Curated_dictionary.csv'

        if dict_path.exists():

            df_dict_viz = pd.read_csv(dict_path)

            has_is_seed = 'is_seed' in df_dict_viz.columns

            print(f"\n  ✓ Loaded {len(df_dict_viz)} dictionary terms")

            if has_is_seed:

                n_seed = df_dict_viz['is_seed'].sum()

                n_expanded = len(df_dict_viz) - n_seed

                print(f"    - {n_seed} seed terms, {n_expanded} expanded terms")

        else:

            print(f"\n  ⚠ Dictionary not found: {dict_path}")

            df_dict_viz = None



        # --------------------------------------------------------

        # Load Training Metrics (if available)

        # --------------------------------------------------------



        metrics_path = source_fs.folders.get('Model_finetuning')

        if metrics_path and (metrics_path / 'training_metrics.json').exists():

            with open(metrics_path / 'training_metrics.json', 'r') as f:

                training_metrics_viz = json.load(f)

            print(f"\n  ✓ Training metrics loaded")

        else:

            training_metrics_viz = None

            print(f"\n  ⚠ Training metrics not available")



        # --------------------------------------------------------

        # Identify Topic Columns

        # --------------------------------------------------------



        print(f"\n{'='*70}")

        print("IDENTIFYING TOPICS")

        print(f"{'='*70}")



        # Detect cosine score columns (format: "score_Topic Name")

        topic_cols_cosine_viz = [col for col in df_viz.columns if col.startswith('score_')]



        if len(topic_cols_cosine_viz) == 0:

            print(f"\n  ✗ ERROR: No topic score columns found")

            print(f"    Expected columns starting with 'score_'")

            print(f"    Available columns: {df_viz.columns.tolist()[:10]}...")

            VIZ_AVAILABLE = False

        else:

            # Extract clean topic names

            topics_viz = []

            for col in topic_cols_cosine_viz:

                topic_name = col.replace('score_', '').strip()

                topics_viz.append(topic_name)



            print(f"\n  ✓ Identified {len(topics_viz)} topics:")

            for i, (col, topic) in enumerate(zip(topic_cols_cosine_viz, topics_viz), 1):

                n_high = (df_viz[col] > MIN_SCORE_THRESHOLD).sum()

                pct_high = n_high / len(df_viz) * 100

                mean_score = df_viz[col].mean()

                print(f"    {i}. {topic}")

                print(f"       - Column: {col}")

                print(f"       - Mean score: {mean_score:.3f}")

                print(f"       - High-scoring (>{MIN_SCORE_THRESHOLD}): {n_high} ({pct_high:.1f}%)")



            # --------------------------------------------------------

            # Detect BERTJE Score Columns (if available)

            # --------------------------------------------------------



            if has_bertje_scores:

                # Format: "bertje_score_Topic Name" (abbreviated, no & suffix)

                topic_cols_bertje_viz = [col for col in df_viz.columns if col.startswith('bertje_score_')]

                print(f"\n  ✓ Found {len(topic_cols_bertje_viz)} BERTJE score columns:")

                for col in topic_cols_bertje_viz:

                    topic_abbrev = col.replace('bertje_score_', '')

                    print(f"    - {col} ({topic_abbrev})")

            else:

                topic_cols_bertje_viz = []



            # --------------------------------------------------------

            # Check for Confidence/Margin Columns

            # --------------------------------------------------------



            # Cosine confidence columns

            has_cosine_margin = 'margin_score' in df_viz.columns

            has_cosine_significance = 'significance_score' in df_viz.columns

            has_cosine_cv = 'cv' in df_viz.columns

            has_primary_topic = 'primary_topic' in df_viz.columns



            # BERTJE confidence columns

            has_bertje_margin = 'bertje_score_margin' in df_viz.columns

            has_bertje_confidence = 'bertje_confidence' in df_viz.columns

            has_bertje_cv = 'bertje_cv' in df_viz.columns



            print(f"\n  Confidence Metrics Available:")

            print(f"    Cosine:")

            print(f"      - Primary topic: {'✓' if has_primary_topic else '✗'}")

            print(f"      - Margin score: {'✓' if has_cosine_margin else '✗'}")

            print(f"      - Significance: {'✓' if has_cosine_significance else '✗'}")

            print(f"      - CV: {'✓' if has_cosine_cv else '✗'}")

            print(f"    BERTJE:")

            print(f"      - Confidence: {'✓' if has_bertje_confidence else '✗'}")

            print(f"      - Margin: {'✓' if has_bertje_margin else '✗'}")

            print(f"      - CV: {'✓' if has_bertje_cv else '✗'}")



            # --------------------------------------------------------

            # Check Required Columns

            # --------------------------------------------------------



            required_cols = ['chunk_uid', 'raw_text']

            optional_cols = ['filename', 'text_for_scoring']



            missing_required = [col for col in required_cols if col not in df_viz.columns]

            if missing_required:

                print(f"\n  ⚠ Warning: Missing required columns: {missing_required}")

                # Try to create them if possible

                if 'chunk_uid' not in df_viz.columns:

                    df_viz['chunk_id'] = range(len(df_viz))

                    print(f"    - Generated chunk_id from index")



            print(f"\n  Data columns:")

            print(f"    - chunk_uid: {'✓' if 'chunk_uid' in df_viz.columns else '✗'}")

            print(f"    - filename: {'✓' if 'filename' in df_viz.columns else '✗'}")

            print(f"    - raw_text: {'✓' if 'raw_text' in df_viz.columns else '✗'}")



    # --------------------------------------------------------

    # Final Summary

    # --------------------------------------------------------



    if VIZ_AVAILABLE:

        print(f"\n{'='*70}")
        print("DATA READY FOR VISUALIZATION")
        print(f"{'='*70}")
        print(f"  Total chunks: {len(df_viz):,}")
        print(f"  Dictionary terms: {len(df_dict_viz) if df_dict_viz is not None else 0}")
        print(f"  Topics: {len(topics_viz)}")
        print(f"  BERTJE scores: {'Available' if has_bertje_scores else 'Not available'}")
        print(f"  Training metrics: {'Available' if training_metrics_viz else 'Not available'}")
        print(f"  Visuals output: {visuals_path_viz.relative_to(fs.root)}")

        print(f"\n✓ Cell 9.2 Complete - Ready for clustering & visualization!")



else:

    print("⚠ Skipping visualization - libraries not available")


CHECKPOINT 9: METHODOLOGY VALIDATION & MODEL COMPARISON

Configuration:
  Workflow: slavery_Short-slavdict_pretrained_slavery_v4

Analysis Parameters:
  Min Score Threshold: 0.3
  K-means Clusters: 5
  PCA Components (2D/3D): 2/3

LOADING SCORED DATA
 Loading cosine scores...
  ✓ Loaded 2840 chunks with cosine scores
 Loading BERTJE predictions...
  Merging on: chunk_uid
  ✓ Merged BERTJE scores: 5 topic columns

  ✓ Loaded 1040 dictionary terms
    - 133 seed terms, 907 expanded terms

  ✓ Training metrics loaded

IDENTIFYING TOPICS

  ✓ Identified 4 topics:
    1. Colonial Systems
       - Column: score_Colonial Systems
       - Mean score: 4.385
       - High-scoring (>0.3): 2840 (100.0%)
    2. Heritage & Memory
       - Column: score_Heritage & Memory
       - Mean score: 4.394
       - High-scoring (>0.3): 2840 (100.0%)
    3. Historical Slavery
       - Column: score_Historical Slavery
       - Mean score: 4.654
       - High-scoring (>0.3): 2840 (100.0%)
    4. Modern Racism &

In [21]:
# ============================================================
# CELL 9.3: CROSS-TOPIC SEMANTIC SPACE VISUALIZATION
# ============================================================

if VIZ_AVAILABLE:
    print(f"\n{'='*70}")
    print("CROSS-TOPIC SEMANTIC SPACE VISUALIZATION")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # Prepare Data: Use Topic Scores as Feature Space
    # --------------------------------------------------------

    print(f"\nPreparing cross-topic feature space...")

    # Each chunk is represented by its scores across ALL topics
    # This creates a semantic space where proximity = similar topic profile
    X = df_viz[topic_cols_cosine_viz].values

    print(f"  Feature matrix: {X.shape[0]} chunks × {X.shape[1]} topics")

    # --------------------------------------------------------
    # Assign Primary Topic to Each Chunk
    # --------------------------------------------------------

    # Primary topic = topic with highest score for each chunk
    primary_topic_idx = df_viz[topic_cols_cosine_viz].values.argmax(axis=1)
    df_viz['primary_topic'] = [topics_viz[i] for i in primary_topic_idx]
    df_viz['primary_score'] = df_viz[topic_cols_cosine_viz].max(axis=1)

    # Calculate confidence metrics
    topic_scores = df_viz[topic_cols_cosine_viz].values
    sorted_scores = np.sort(topic_scores, axis=1)[:, ::-1]
    df_viz['score_margin'] = sorted_scores[:, 0] - sorted_scores[:, 1]  # Margin to 2nd topic
    df_viz['score_cv'] = np.std(topic_scores, axis=1) / np.mean(topic_scores, axis=1)  # Coefficient of variation

    print(f"\n  Primary topic distribution:")
    for topic in topics_viz:
        count = (df_viz['primary_topic'] == topic).sum()
        pct = count / len(df_viz) * 100
        print(f"    {topic:50s}: {count:5d} ({pct:5.1f}%)")

    # --------------------------------------------------------
    # Filter High-Confidence Chunks
    # --------------------------------------------------------

    # Use chunks with clear topic assignment for better visualization
    high_conf_mask = (df_viz['primary_score'] >= MIN_SCORE_THRESHOLD)
    df_high_conf = df_viz[high_conf_mask].copy()
    X_high_conf = X[high_conf_mask]

    print(f"\n  High-confidence chunks (score >= {MIN_SCORE_THRESHOLD}): {len(df_high_conf)} / {len(df_viz)} ({len(df_high_conf)/len(df_viz)*100:.1f}%)")

    if len(df_high_conf) < 10:
        print(f"\n  ⚠ WARNING: Too few high-confidence chunks ({len(df_high_conf)})")
        print(f"    Consider lowering MIN_SCORE_THRESHOLD (currently {MIN_SCORE_THRESHOLD})")
        VIZ_AVAILABLE = False

    # --------------------------------------------------------
    # Calculate Topic Centroids
    # --------------------------------------------------------

    if VIZ_AVAILABLE:
        print(f"\n  Calculating topic centroids...")

        topic_centroids = {}
        for topic in topics_viz:
            topic_mask = df_high_conf['primary_topic'] == topic
            if topic_mask.sum() > 0:
                centroid = X_high_conf[topic_mask.values].mean(axis=0)
                topic_centroids[topic] = centroid
                print(f"    {topic:50s}: {topic_mask.sum():4d} chunks")
            else:
                print(f"    {topic:50s}: NO CHUNKS (skipped)")

        # --------------------------------------------------------
        # Dimensionality Reduction: PCA
        # --------------------------------------------------------

        print(f"\n  Performing PCA for dimensionality reduction...")

        # Standardize features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_high_conf)

        # PCA to 2D
        pca_2d = PCA(n_components=2, random_state=42)
        X_pca_2d = pca_2d.fit_transform(X_scaled)

        var_2d = pca_2d.explained_variance_ratio_
        print(f"    2D PCA: {var_2d.sum():.2%} variance explained")
        print(f"      PC1: {var_2d[0]:.2%}, PC2: {var_2d[1]:.2%}")

        # PCA to 3D
        pca_3d = PCA(n_components=3, random_state=42)
        X_pca_3d = pca_3d.fit_transform(X_scaled)

        var_3d = pca_3d.explained_variance_ratio_
        print(f"    3D PCA: {var_3d.sum():.2%} variance explained")
        print(f"      PC1: {var_3d[0]:.2%}, PC2: {var_3d[1]:.2%}, PC3: {var_3d[2]:.2%}")

        # Add PCA coordinates to dataframe
        df_high_conf['pca_x'] = X_pca_2d[:, 0]
        df_high_conf['pca_y'] = X_pca_2d[:, 1]
        df_high_conf['pca_z'] = X_pca_3d[:, 2]

        # Transform centroids to PCA space
        centroids_pca_2d = {}
        centroids_pca_3d = {}

        for topic, centroid in topic_centroids.items():
            centroid_scaled = scaler.transform([centroid])
            centroids_pca_2d[topic] = pca_2d.transform(centroid_scaled)[0]
            centroids_pca_3d[topic] = pca_3d.transform(centroid_scaled)[0]

        print(f"    ✓ Topic centroids transformed to PCA space")

        # --------------------------------------------------------
        # Calculate Distances to Centroids
        # --------------------------------------------------------

        print(f"\n  Calculating chunk distances to topic centroids...")

        # Distance to own topic centroid
        distances_to_own = []
        for idx in range(len(df_high_conf)):
            row = df_high_conf.iloc[idx]
            chunk_vec = X_high_conf[idx]
            centroid_vec = topic_centroids.get(row['primary_topic'])
            if centroid_vec is not None:
                dist = np.linalg.norm(chunk_vec - centroid_vec)
                distances_to_own.append(dist)
            else:
                distances_to_own.append(np.nan)

        df_high_conf['distance_to_centroid'] = distances_to_own

        print(f"    ✓ Distance to own centroid calculated")
        print(f"      Mean: {np.nanmean(distances_to_own):.3f}")
        print(f"      Std:  {np.nanstd(distances_to_own):.3f}")

        # --------------------------------------------------------
        # Prepare Hover Text
        # --------------------------------------------------------

        print(f"\n  Preparing visualization data...")

        # Create text snippets
        if 'raw_text' in df_high_conf.columns:
            df_high_conf['snippet'] = df_high_conf['raw_text'].apply(
                lambda x: (str(x)[:200] + '...') if pd.notna(x) and len(str(x)) > 200
                         else str(x) if pd.notna(x) else '[No text]'
            )
        else:
            df_high_conf['snippet'] = '[Text not available]'

        # Get file names
        if 'filename' in df_high_conf.columns:
            df_high_conf['file_display'] = df_high_conf['filename']
        else:
            df_high_conf['file_display'] = 'Unknown'

        # Get chunk IDs
        if 'chunk_uid' not in df_high_conf.columns:
            df_high_conf['chunk_uid'] = df_high_conf.index

        # Store results
        cross_topic_viz_data = {
            'df': df_high_conf,
            'pca_2d': pca_2d,
            'pca_3d': pca_3d,
            'scaler': scaler,
            'centroids': topic_centroids,
            'centroids_pca_2d': centroids_pca_2d,
            'centroids_pca_3d': centroids_pca_3d,
            'explained_variance_2d': var_2d,
            'explained_variance_3d': var_3d,
        }

        print(f"\n{'='*70}")
        print(f"✓ CROSS-TOPIC SPACE PREPARED")
        print(f"{'='*70}")
        print(f"  Chunks in visualization: {len(df_high_conf)}")
        print(f"  Topics with centroids: {len(topic_centroids)}")
        print(f"  2D variance explained: {var_2d.sum():.1%}")
        print(f"  3D variance explained: {var_3d.sum():.1%}")

        print(f"\n✓ Cell 9.3 Complete - Ready for visualization!")

else:
    print("⚠ Skipping cross-topic visualization - not available")



CROSS-TOPIC SEMANTIC SPACE VISUALIZATION

Preparing cross-topic feature space...
  Feature matrix: 2840 chunks × 4 topics

  Primary topic distribution:
    Colonial Systems                                  :   564 ( 19.9%)
    Heritage & Memory                                 :   526 ( 18.5%)
    Historical Slavery                                :   879 ( 31.0%)
    Modern Racism & Discrimination                    :   871 ( 30.7%)

  High-confidence chunks (score >= 0.3): 2840 / 2840 (100.0%)

  Calculating topic centroids...
    Colonial Systems                                  :  564 chunks
    Heritage & Memory                                 :  526 chunks
    Historical Slavery                                :  879 chunks
    Modern Racism & Discrimination                    :  871 chunks

  Performing PCA for dimensionality reduction...
    2D PCA: 84.82% variance explained
      PC1: 67.14%, PC2: 17.68%
    3D PCA: 95.08% variance explained
      PC1: 67.14%, PC2: 17.68%, PC3:

In [22]:
# ============================================================
# CELL 9.3A: GENERATE CHUNK EMBEDDINGS (PRE/POST TRAINING)
# ============================================================

if VIZ_AVAILABLE:
    print(f"\n{'='*70}")
    print("GENERATING CHUNK EMBEDDINGS FOR PRE/POST COMPARISON")
    print(f"{'='*70}")

    # Import required libraries
    from tqdm.auto import tqdm
    from transformers import AutoModel, AutoTokenizer
    import torch

    # --------------------------------------------------------
    # Decide: Use Embeddings or Topic Scores
    # --------------------------------------------------------

    # Configuration
    USE_BERTJE_EMBEDDINGS = True  # Set to False to use topic scores only

    if USE_BERTJE_EMBEDDINGS:
        print(f"\n  Mode: BERTJE embeddings (768-dimensional)")
        print(f"  Will generate pre and post-training embeddings for chunks")
    else:
        print(f"\n  Mode: Topic scores ({len(topic_cols_cosine_viz)}-dimensional)")
        print(f"  Will use cosine scores vs BERTJE predictions")

    # --------------------------------------------------------
    # Load BERTJE Models (if using embeddings)
    # --------------------------------------------------------

    chunk_embeddings_data = {}

    if USE_BERTJE_EMBEDDINGS:
        trained_model_path = source_fs.folders.get('Model_finetuning')
        base_model_path = source_fs.folders.get('Model_finetuning') / 'base_encoder' if trained_model_path else None

        has_trained_model = False
        has_base_model = False

        # Try to load trained model
        if trained_model_path and (trained_model_path / 'trained_encoder').exists():
            try:
                print(f"\n  Loading trained (post-training) BERTJE model...")
                tokenizer_trained = AutoTokenizer.from_pretrained(str(trained_model_path / 'trained_encoder'))
                model_trained = AutoModel.from_pretrained(str(trained_model_path / 'trained_encoder'))
                model_trained.eval()
                has_trained_model = True
                print(f"    ✓ Loaded: {trained_model_path / 'trained_encoder'}")
            except Exception as e:
                print(f"    ✗ Failed to load trained model: {e}")

        # Try to load base model
        if base_model_path and base_model_path.exists():
            try:
                print(f"\n  Loading base (pre-training) BERTJE model...")
                tokenizer_base = AutoTokenizer.from_pretrained(str(base_model_path))
                model_base = AutoModel.from_pretrained(str(base_model_path))
                model_base.eval()
                has_base_model = True
                print(f"    ✓ Loaded: {base_model_path}")
            except Exception as e:
                print(f"    ✗ Failed to load base model: {e}")

        # Fallback to default BERTJE
        if not has_base_model:
            try:
                print(f"\n  Loading default GroNLP BERTJE model...")
                tokenizer_base = AutoTokenizer.from_pretrained('GroNLP/bert-base-dutch-cased')
                model_base = AutoModel.from_pretrained('GroNLP/bert-base-dutch-cased')
                model_base.eval()
                has_base_model = True
                print(f"    ✓ Loaded: GroNLP/bert-base-dutch-cased")
            except Exception as e:
                print(f"    ✗ Failed to load default BERTJE: {e}")

        if not (has_trained_model and has_base_model):
            print(f"\n  ⚠ Cannot generate embeddings - switching to topic scores mode")
            USE_BERTJE_EMBEDDINGS = False

    # --------------------------------------------------------
    # Generate Chunk Embeddings or Prepare Topic Scores
    # --------------------------------------------------------

    if USE_BERTJE_EMBEDDINGS and has_base_model and has_trained_model:
        print(f"\n{'='*70}")
        print("GENERATING CHUNK EMBEDDINGS")
        print(f"{'='*70}")

        # --------------------------------------------------------
        # Filter Chunks by BERTJE Confidence
        # --------------------------------------------------------

        print(f"Filtering chunks by BERTJE confidence...")

        # Check if we have BERTJE confidence scores
        if 'bertje_confidence' in df_viz.columns:
            # Use BERTJE confidence (from Checkpoint 8)
            # Confidence levels: 'very_high', 'high', 'medium', 'low', 'very_low'
            valid_confidence = ['very_high', 'high', 'medium']
            df_for_embedding = df_viz[
                df_viz['bertje_confidence'].isin(valid_confidence)
            ].copy()
            print(f"    Using BERTJE confidence filter: {valid_confidence}")
            print(f"    Selected: {len(df_for_embedding)} / {len(df_viz)} chunks")
        else:
            # Fallback to score threshold
            print(f"    BERTJE confidence not available, using score threshold")
            df_for_embedding = df_viz[df_viz['primary_score'] >= MIN_SCORE_THRESHOLD].copy()
            print(f"    Selected: {len(df_for_embedding)} / {len(df_viz)} chunks (score >= {MIN_SCORE_THRESHOLD})")

        # Sample if still too many
        MAX_CHUNKS_FOR_EMBEDDING = 5000
        if len(df_for_embedding) > MAX_CHUNKS_FOR_EMBEDDING:
            print(f"\n  Sampling {MAX_CHUNKS_FOR_EMBEDDING} chunks from {len(df_for_embedding)} (for speed)")
            df_for_embedding = df_for_embedding.sample(n=MAX_CHUNKS_FOR_EMBEDDING, random_state=42)
        else:
            print(f"\n  Using all {len(df_for_embedding)} high-confidence chunks")

        # Get chunk texts
        chunk_texts = df_for_embedding['text_for_scoring'].fillna(df_for_embedding['raw_text']).tolist()

        def get_chunk_embeddings(texts, tokenizer, model, batch_size=16):
            """Generate embeddings for chunks"""
            from tqdm import tqdm
            embeddings = []

            for i in tqdm(range(0, len(texts), batch_size), desc='Embedding batches'):
                if i % 500 == 0:
                    print(f"    Processing batch {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1}")

                batch_texts = texts[i:i+batch_size]

                # Tokenize
                inputs = tokenizer(
                    batch_texts,
                    padding=True,
                    truncation=True,
                    max_length=512,  # Full BERTJE context
                    return_tensors='pt'
                )

                # Generate embeddings
                with torch.no_grad():
                    outputs = model(**inputs)
                    # Use CLS token
                    batch_embeddings = outputs.last_hidden_state[:, 0, :].numpy()
                    embeddings.append(batch_embeddings)

            return np.vstack(embeddings)

        # Generate pre-training embeddings
        print(f"\n  Generating pre-training embeddings...")
        embeddings_base = get_chunk_embeddings(chunk_texts, tokenizer_base, model_base)
        print(f"    ✓ Shape: {embeddings_base.shape}")

        # Generate post-training embeddings
        print(f"\n  Generating post-training embeddings...")
        embeddings_trained = get_chunk_embeddings(chunk_texts, tokenizer_trained, model_trained)
        print(f"    ✓ Shape: {embeddings_trained.shape}")

        # Store embeddings
        chunk_embeddings_data['df'] = df_for_embedding
        chunk_embeddings_data['embeddings_base'] = embeddings_base
        chunk_embeddings_data['embeddings_trained'] = embeddings_trained
        chunk_embeddings_data['mode'] = 'embeddings'

        print(f"\n  ✓ Chunk embeddings generated")

    else:
        # Use topic scores
        print(f"\n{'='*70}")
        print("USING TOPIC SCORES FOR COMPARISON")
        print(f"{'='*70}")

        df_for_comparison = df_viz[df_viz['primary_score'] >= MIN_SCORE_THRESHOLD].copy()

        # Pre-training: Cosine scores
        scores_base = df_for_comparison[topic_cols_cosine_viz].values

        # Post-training: BERTJE scores (if available)
        if has_bertje_scores:
            scores_trained = df_for_comparison[topic_cols_bertje_viz].values
            print(f"\n  Pre-training: Cosine scores ({scores_base.shape})")
            print(f"  Post-training: BERTJE scores ({scores_trained.shape})")
        else:
            print(f"\n  ⚠ BERTJE scores not available - cannot compare pre/post")
            scores_trained = None

        chunk_embeddings_data['df'] = df_for_comparison
        chunk_embeddings_data['scores_base'] = scores_base
        chunk_embeddings_data['scores_trained'] = scores_trained
        chunk_embeddings_data['mode'] = 'scores'

    # --------------------------------------------------------
    # Apply PCA
    # --------------------------------------------------------

    print(f"\n{'='*70}")
    print("DIMENSIONALITY REDUCTION (PCA)")
    print(f"{'='*70}")

    if chunk_embeddings_data['mode'] == 'embeddings':
        # PCA on embeddings
        print(f"\n  Applying PCA to pre-training embeddings...")
        scaler_base = StandardScaler()
        emb_scaled_base = scaler_base.fit_transform(chunk_embeddings_data['embeddings_base'])

        pca_2d_base = PCA(n_components=2, random_state=42)
        coords_2d_base = pca_2d_base.fit_transform(emb_scaled_base)

        pca_3d_base = PCA(n_components=3, random_state=42)
        coords_3d_base = pca_3d_base.fit_transform(emb_scaled_base)

        var_2d_base = pca_2d_base.explained_variance_ratio_
        var_3d_base = pca_3d_base.explained_variance_ratio_

        print(f"    2D: {var_2d_base.sum():.2%} variance")
        print(f"    3D: {var_3d_base.sum():.2%} variance")

        print(f"\n  Applying PCA to post-training embeddings...")
        scaler_trained = StandardScaler()
        emb_scaled_trained = scaler_trained.fit_transform(chunk_embeddings_data['embeddings_trained'])

        pca_2d_trained = PCA(n_components=2, random_state=42)
        coords_2d_trained = pca_2d_trained.fit_transform(emb_scaled_trained)

        pca_3d_trained = PCA(n_components=3, random_state=42)
        coords_3d_trained = pca_3d_trained.fit_transform(emb_scaled_trained)

        var_2d_trained = pca_2d_trained.explained_variance_ratio_
        var_3d_trained = pca_3d_trained.explained_variance_ratio_

        print(f"    2D: {var_2d_trained.sum():.2%} variance")
        print(f"    3D: {var_3d_trained.sum():.2%} variance")

        # Store
        chunk_embeddings_data['base'] = {
            'coords_2d': coords_2d_base,
            'coords_3d': coords_3d_base,
            'variance_2d': var_2d_base,
            'variance_3d': var_3d_base,
        }
        chunk_embeddings_data['trained'] = {
            'coords_2d': coords_2d_trained,
            'coords_3d': coords_3d_trained,
            'variance_2d': var_2d_trained,
            'variance_3d': var_3d_trained,
        }

    else:
        # PCA on topic scores
        print(f"\n  Applying PCA to cosine scores...")
        scaler_base = StandardScaler()
        scores_scaled_base = scaler_base.fit_transform(chunk_embeddings_data['scores_base'])

        pca_2d_base = PCA(n_components=2, random_state=42)
        coords_2d_base = pca_2d_base.fit_transform(scores_scaled_base)

        pca_3d_base = PCA(n_components=3, random_state=42)
        coords_3d_base = pca_3d_base.fit_transform(scores_scaled_base)

        var_2d_base = pca_2d_base.explained_variance_ratio_
        var_3d_base = pca_3d_base.explained_variance_ratio_

        print(f"    2D: {var_2d_base.sum():.2%} variance")
        print(f"    3D: {var_3d_base.sum():.2%} variance")

        chunk_embeddings_data['base'] = {
            'coords_2d': coords_2d_base,
            'coords_3d': coords_3d_base,
            'variance_2d': var_2d_base,
            'variance_3d': var_3d_base,
        }

        if chunk_embeddings_data['scores_trained'] is not None:
            print(f"\n  Applying PCA to BERTJE scores...")
            scaler_trained = StandardScaler()
            scores_scaled_trained = scaler_trained.fit_transform(chunk_embeddings_data['scores_trained'])

            pca_2d_trained = PCA(n_components=2, random_state=42)
            coords_2d_trained = pca_2d_trained.fit_transform(scores_scaled_trained)

            pca_3d_trained = PCA(n_components=3, random_state=42)
            coords_3d_trained = pca_3d_trained.fit_transform(scores_scaled_trained)

            var_2d_trained = pca_2d_trained.explained_variance_ratio_
            var_3d_trained = pca_3d_trained.explained_variance_ratio_

            print(f"    2D: {var_2d_trained.sum():.2%} variance")
            print(f"    3D: {var_3d_trained.sum():.2%} variance")

            chunk_embeddings_data['trained'] = {
                'coords_2d': coords_2d_trained,
                'coords_3d': coords_3d_trained,
                'variance_2d': var_2d_trained,
                'variance_3d': var_3d_trained,
            }
        else:
            chunk_embeddings_data['trained'] = None

    # --------------------------------------------------------
    # Calculate Shifts
    # --------------------------------------------------------

    if 'trained' in chunk_embeddings_data and chunk_embeddings_data['trained'] is not None:
        print(f"\n{'='*70}")
        print("CALCULATING CHUNK SHIFTS")
        print(f"{'='*70}")

        coords_pre = chunk_embeddings_data['base']['coords_3d']
        coords_post = chunk_embeddings_data['trained']['coords_3d']

        shifts = coords_post - coords_pre
        shift_magnitudes = np.linalg.norm(shifts, axis=1)

        chunk_embeddings_data['df']['shift_magnitude'] = shift_magnitudes
        chunk_embeddings_data['df']['shift_x'] = shifts[:, 0]
        chunk_embeddings_data['df']['shift_y'] = shifts[:, 1]
        chunk_embeddings_data['df']['shift_z'] = shifts[:, 2]

        print(f"\n  Shift statistics:")
        print(f"    Mean: {shift_magnitudes.mean():.3f}")
        print(f"    Median: {np.median(shift_magnitudes):.3f}")
        print(f"    Max: {shift_magnitudes.max():.3f}")
        print(f"    Std: {shift_magnitudes.std():.3f}")

        # Top shifters by topic
        print(f"\n  Top shifters by topic:")
        for topic in chunk_embeddings_data['df']['primary_topic'].unique():
            topic_df = chunk_embeddings_data['df'][chunk_embeddings_data['df']['primary_topic'] == topic]
            if len(topic_df) > 0:
                top_shifter = topic_df.nlargest(1, 'shift_magnitude').iloc[0]
                print(f"    {topic:50s}: {top_shifter['shift_magnitude']:.3f}")

    print(f"\n{'='*70}")
    print(f"✓ CHUNK EMBEDDINGS PREPARED")
    print(f"{'='*70}")
    print(f"  Chunks: {len(chunk_embeddings_data['df'])}")
    print(f"  Mode: {chunk_embeddings_data['mode']}")
    print(f"  Pre-training data: Available")
    print(f"  Post-training data: {'Available' if chunk_embeddings_data.get('trained') else 'Not available'}")

    print(f"\n✓ Cell 9.3A Complete - Ready for chunk shift visualization!")

else:
    print("⚠ Skipping chunk embeddings - visualization not available")



GENERATING CHUNK EMBEDDINGS FOR PRE/POST COMPARISON

  Mode: BERTJE embeddings (768-dimensional)
  Will generate pre and post-training embeddings for chunks

  Loading trained (post-training) BERTJE model...
    ✓ Loaded: C:\Users\Home\policy-analysis\workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Model_finetuning\trained_encoder

  Loading base (pre-training) BERTJE model...
    ✓ Loaded: C:\Users\Home\policy-analysis\workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Model_finetuning\base_encoder

GENERATING CHUNK EMBEDDINGS
Filtering chunks by BERTJE confidence...
    Using BERTJE confidence filter: ['very_high', 'high', 'medium']
    Selected: 928 / 2840 chunks

  Using all 928 high-confidence chunks

  Generating pre-training embeddings...


Embedding batches:   0%|          | 0/58 [00:00<?, ?it/s]

    Processing batch 1/58


Embedding batches: 100%|██████████| 58/58 [02:38<00:00,  2.74s/it]


    ✓ Shape: (928, 768)

  Generating post-training embeddings...


Embedding batches:   0%|          | 0/58 [00:00<?, ?it/s]

    Processing batch 1/58


Embedding batches: 100%|██████████| 58/58 [02:38<00:00,  2.74s/it]

    ✓ Shape: (928, 768)

  ✓ Chunk embeddings generated

DIMENSIONALITY REDUCTION (PCA)

  Applying PCA to pre-training embeddings...
    2D: 18.17% variance
    3D: 22.28% variance

  Applying PCA to post-training embeddings...
    2D: 27.98% variance
    3D: 34.58% variance

CALCULATING CHUNK SHIFTS

  Shift statistics:
    Mean: 13.874
    Median: 13.080
    Max: 34.043
    Std: 6.383

  Top shifters by topic:
    Historical Slavery                                : 27.870
    Modern Racism & Discrimination                    : 29.294
    Heritage & Memory                                 : 34.043
    Colonial Systems                                  : 32.169

✓ CHUNK EMBEDDINGS PREPARED
  Chunks: 928
  Mode: embeddings
  Pre-training data: Available
  Post-training data: Available

✓ Cell 9.3A Complete - Ready for chunk shift visualization!


In [23]:
# ============================================================
# CELL 9.4: 2D CROSS-TOPIC VISUALIZATION
# ============================================================

if VIZ_AVAILABLE:
    print(f"\n{'='*70}")
    print("CREATING 2D CROSS-TOPIC VISUALIZATION")
    print(f"{'='*70}")

    df_plot = cross_topic_viz_data['df']
    centroids_2d = cross_topic_viz_data['centroids_pca_2d']
    var_exp = cross_topic_viz_data['explained_variance_2d']

    # --------------------------------------------------------
    # Create Color Mapping for Topics
    # --------------------------------------------------------

    # Use distinct colors for each topic
    color_palette = px.colors.qualitative.Set3 + px.colors.qualitative.Pastel
    topic_colors = {topic: color_palette[i % len(color_palette)]
                    for i, topic in enumerate(topics_viz)}

    # --------------------------------------------------------
    # Create Figure
    # --------------------------------------------------------

    fig = go.Figure()

    print(f"\nAdding chunks to plot...")

    # --------------------------------------------------------
    # Plot Chunks Grouped by Primary Topic
    # --------------------------------------------------------

    for topic in topics_viz:
        topic_df = df_plot[df_plot['primary_topic'] == topic]

        if len(topic_df) == 0:
            print(f"  {topic}: No chunks (skipped)")
            continue

        print(f"  {topic}: {len(topic_df)} chunks")

        # Build hover text
        hover_texts = []
        for idx, row in topic_df.iterrows():
            # Get all topic scores
            all_scores = []
            for tc, tn in zip(topic_cols_cosine_viz, topics_viz):
                score = row.get(tc, 0)
                all_scores.append(f"  • {tn}: {score:.3f}")

            hover_parts = [
                f"<b>Chunk ID:</b> {row.get('chunk_uid', 'N/A')}",
                f"<b>File:</b> {row.get('file_display', 'Unknown')}",
                f"<b>Primary Topic:</b> {row['primary_topic']}",
                f"<b>Primary Score:</b> {row['primary_score']:.3f}",
                f"<b>Margin to 2nd:</b> {row.get('score_margin', 0):.3f}",
                f"<b>Distance to Centroid:</b> {row.get('distance_to_centroid', 0):.3f}",
                "<br><b>All Topic Scores:</b>",
            ] + all_scores

            # Add BERTJE comparison if available
            if has_bertje_scores:
                bertje_topic = row.get('bertje_primary_topic')
                if pd.notna(bertje_topic):
                    hover_parts.append(f"<br><b>BERTJE Topic:</b> {bertje_topic}")
                    hover_parts.append(f"<b>Agreement:</b> {'✓' if bertje_topic == row['primary_topic'] else '✗'}")

            # Add text preview
            hover_parts.append(f"<br><b>Text:</b>")
            hover_parts.append(f"{row.get('snippet', '[No text]')[:200]}")

            hover_texts.append("<br>".join(hover_parts))

        # Add scatter trace
        fig.add_trace(go.Scatter(
            x=topic_df['pca_x'],
            y=topic_df['pca_y'],
            mode='markers',
            name=topic,
            marker=dict(
                size=6,
                color=topic_colors[topic],
                opacity=0.6,
                line=dict(width=0.5, color='white')
            ),
            hovertemplate='%{hovertext}<extra></extra>',
            hovertext=hover_texts,
            legendgroup=topic,
        ))

    # --------------------------------------------------------
    # Plot Topic Centroids
    # --------------------------------------------------------

    print(f"\nAdding topic centroids...")

    centroid_x = []
    centroid_y = []
    centroid_names = []
    centroid_hover = []

    for topic, coords in centroids_2d.items():
        centroid_x.append(coords[0])
        centroid_y.append(coords[1])
        centroid_names.append(topic)

        # Count chunks for this topic
        n_chunks = (df_plot['primary_topic'] == topic).sum()

        hover_text = (
            f"<b>TOPIC CENTROID</b><br>"
            f"<b>Topic:</b> {topic}<br>"
            f"<b>Chunks:</b> {n_chunks}<br>"
            f"<b>Position:</b> ({coords[0]:.2f}, {coords[1]:.2f})"
        )
        centroid_hover.append(hover_text)

    fig.add_trace(go.Scatter(
        x=centroid_x,
        y=centroid_y,
        mode='markers+text',
        name='Topic Centroids',
        text=centroid_names,
        textposition='top center',
        textfont=dict(size=10, color='black', family='Arial Black'),
        marker=dict(
            size=20,
            color='red',
            symbol='star',
            line=dict(width=2, color='darkred')
        ),
        hovertemplate='%{hovertext}<extra></extra>',
        hovertext=centroid_hover,
        showlegend=True,
        legendgroup='centroids',
    ))

    # --------------------------------------------------------
    # Update Layout
    # --------------------------------------------------------

    fig.update_layout(
        title=(
            f'<b>Cross-Topic Semantic Space (2D PCA Projection)</b><br>'
            f'<sub>Each point = chunk colored by primary topic | '
            f'Stars = topic centroids | '
            f'Variance explained: {var_exp.sum():.1%} | '
            f'Hover for details</sub>'
        ),
        xaxis_title=f'PC1 ({var_exp[0]:.1%} variance)',
        yaxis_title=f'PC2 ({var_exp[1]:.1%} variance)',
        height=900,
        width=1400,
        template='plotly_white',
        hovermode='closest',
        font=dict(size=11),
        showlegend=True,
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02,
            font=dict(size=9),
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='lightgray',
            borderwidth=1
        ),
        xaxis=dict(
            showgrid=True,
            gridwidth=1,
            gridcolor='lightgray',
            zeroline=True,
            zerolinewidth=2,
            zerolinecolor='gray'
        ),
        yaxis=dict(
            showgrid=True,
            gridwidth=1,
            gridcolor='lightgray',
            zeroline=True,
            zerolinewidth=2,
            zerolinecolor='gray'
        )
    )

    # --------------------------------------------------------
    # Save Visualization
    # --------------------------------------------------------

    output_path = visuals_path_viz / 'cross_topic_space_2d.html'
    fig.write_html(str(output_path))

    print(f"\n{'='*70}")
    print(f"2D VISUALIZATION COMPLETE")
    print(f"{'='*70}")
    print(f"  Saved to: {output_path.relative_to(fs.root)}")
    print(f"  Chunks plotted: {len(df_plot)}")
    print(f"  Topics: {len(centroids_2d)}")
    print(f"  Variance explained: {var_exp.sum():.1%}")

    # Display in notebook
    try:
        fig.show()
    except:
        print(f"  (Interactive display not available)")

    print(f"\nCell 9.4 Complete!")

else:
    print("Skipping 2D visualization - not available")



CREATING 2D CROSS-TOPIC VISUALIZATION

Adding chunks to plot...
  Colonial Systems: 564 chunks
  Heritage & Memory: 526 chunks
  Historical Slavery: 879 chunks
  Modern Racism & Discrimination: 871 chunks

Adding topic centroids...

2D VISUALIZATION COMPLETE
  Saved to: Visuals\cross_topic_space_2d.html
  Chunks plotted: 2840
  Topics: 4
  Variance explained: 84.8%



Cell 9.4 Complete!


In [24]:
# ============================================================
# CELL 9.4A: CHUNK PRE/POST TRAINING COMPARISON (2D)
# ============================================================

if VIZ_AVAILABLE and 'chunk_embeddings_data' in globals() and chunk_embeddings_data.get('trained'):
    print(f"\n{'='*70}")
    print("2D CHUNK PRE/POST TRAINING COMPARISON")
    print(f"{'='*70}")

    df_chunks = chunk_embeddings_data['df']
    coords_pre = chunk_embeddings_data['base']['coords_2d']
    coords_post = chunk_embeddings_data['trained']['coords_2d']
    var_pre = chunk_embeddings_data['base']['variance_2d']
    var_post = chunk_embeddings_data['trained']['variance_2d']

    # Create side-by-side comparison
    from plotly.subplots import make_subplots

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[
            f'<b>Pre-Training</b> ({chunk_embeddings_data["mode"].title()})',
            f'<b>Post-Training</b> ({chunk_embeddings_data["mode"].title()})'
        ],
        horizontal_spacing=0.08
    )

    print(f"\nCreating side-by-side comparison...")
    print(f"  Mode: {chunk_embeddings_data['mode']}")
    print(f"  Chunks: {len(df_chunks)}")

    # --------------------------------------------------------
    # Plot Pre-Training (Column 1)
    # --------------------------------------------------------

    print(f"\n  Plotting pre-training positions...")

    for topic in topics_viz:
        topic_df = df_chunks[df_chunks['primary_topic'] == topic]
        if len(topic_df) == 0:
            continue

        # Get pre-training coordinates
        topic_indices = topic_df.index.tolist()
        local_indices = [df_chunks.index.get_loc(idx) for idx in topic_indices]

        topic_x = coords_pre[local_indices, 0]
        topic_y = coords_pre[local_indices, 1]

        # Build hover
        hover_texts = []
        for idx, row in topic_df.iterrows():
            hover_parts = [
                f"<b>Chunk {row.get('chunk_uid', idx)}</b> (Pre-training)",
                f"Primary Topic: {row['primary_topic']}",
                f"Primary Score: {row['primary_score']:.3f}",
                f"Shift Magnitude: {row.get('shift_magnitude', 0):.3f}",
            ]
            hover_texts.append("<br>".join(hover_parts))

        fig.add_trace(go.Scatter(
            x=topic_x,
            y=topic_y,
            mode='markers',
            name=topic,
            marker=dict(
                size=5,
                color=topic_colors.get(topic, 'gray'),
                opacity=0.5,
            ),
            hovertemplate='%{hovertext}<extra></extra>',
            hovertext=hover_texts,
            legendgroup=topic,
            showlegend=True
        ), row=1, col=1)

    # --------------------------------------------------------
    # Plot Post-Training (Column 2)
    # --------------------------------------------------------

    print(f"  Plotting post-training positions...")

    for topic in topics_viz:
        topic_df = df_chunks[df_chunks['primary_topic'] == topic]
        if len(topic_df) == 0:
            continue

        topic_indices = topic_df.index.tolist()
        local_indices = [df_chunks.index.get_loc(idx) for idx in topic_indices]

        topic_x = coords_post[local_indices, 0]
        topic_y = coords_post[local_indices, 1]

        hover_texts = []
        for idx, row in topic_df.iterrows():
            hover_parts = [
                f"<b>Chunk {row.get('chunk_uid', idx)}</b> (Post-training)",
                f"Primary Topic: {row['primary_topic']}",
                f"Primary Score: {row['primary_score']:.3f}",
                f"Shift Magnitude: {row.get('shift_magnitude', 0):.3f}",
            ]
            hover_texts.append("<br>".join(hover_parts))

        fig.add_trace(go.Scatter(
            x=topic_x,
            y=topic_y,
            mode='markers',
            name=topic,
            marker=dict(
                size=5,
                color=topic_colors.get(topic, 'gray'),
                opacity=0.8,
            ),
            hovertemplate='%{hovertext}<extra></extra>',
            hovertext=hover_texts,
            legendgroup=topic,
            showlegend=False  # Already shown in col 1
        ), row=1, col=2)

    # --------------------------------------------------------
    # Update Layout
    # --------------------------------------------------------

    fig.update_layout(
        title=(
            f'<b>Corpus Chunks: Pre vs Post Training Comparison (2D)</b><br>'
            f'<sub>Left: Before training | Right: After training | '
            f'Compare clustering patterns</sub>'
        ),
        height=700,
        width=1600,
        template='plotly_white',
        hovermode='closest',
        font=dict(size=11),
        showlegend=True,
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02,
            font=dict(size=9),
            bgcolor='rgba(255,255,255,0.8)'
        )
    )

    # Update axes
    for col in [1, 2]:
        var_exp = var_pre if col == 1 else var_post
        fig.update_xaxes(
            title_text=f'PC1 ({var_exp[0]:.1%})',
            showgrid=True,
            gridcolor='lightgray',
            zeroline=True,
            row=1,
            col=col
        )
        fig.update_yaxes(
            title_text=f'PC2 ({var_exp[1]:.1%})',
            showgrid=True,
            gridcolor='lightgray',
            zeroline=True,
            row=1,
            col=col
        )

    # Add variance annotations
    fig.add_annotation(
        text=f"<i>Variance: {var_pre.sum():.1%}</i>",
        xref='x1',
        yref='y1',
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        showarrow=False,
        font=dict(size=10, color='gray'),
        row=1,
        col=1
    )

    fig.add_annotation(
        text=f"<i>Variance: {var_post.sum():.1%}</i>",
        xref='x2',
        yref='y2',
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        showarrow=False,
        font=dict(size=10, color='gray'),
        row=1,
        col=2
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    output_path = visuals_path_viz / 'chunks_prepost_comparison_2d.html'
    fig.write_html(str(output_path))

    print(f"\n{'='*70}")
    print(f"CHUNK PRE/POST COMPARISON COMPLETE")
    print(f"{'='*70}")
    print(f"  Saved to: {output_path.relative_to(fs.root)}")
    print(f"  Mode: {chunk_embeddings_data['mode']}")

    try:
        fig.show()
    except:
        print(f"  (Interactive display not available)")

    print(f"\n✓ Cell 9.4A Complete!")

else:
    if 'chunk_embeddings_data' not in globals():
        print("⚠ Skipping chunk pre/post comparison - run Cell 9.3A first")
    elif not chunk_embeddings_data.get('trained'):
        print("⚠ Skipping chunk pre/post comparison - only pre-training data available")



2D CHUNK PRE/POST TRAINING COMPARISON

Creating side-by-side comparison...
  Mode: embeddings
  Chunks: 928

  Plotting pre-training positions...
  Plotting post-training positions...

CHUNK PRE/POST COMPARISON COMPLETE
  Saved to: Visuals\chunks_prepost_comparison_2d.html
  Mode: embeddings



✓ Cell 9.4A Complete!


In [25]:
# ============================================================
# CELL 9.4B: CHUNK SHIFT VECTORS (3D)
# ============================================================

if VIZ_AVAILABLE and 'chunk_embeddings_data' in globals() and chunk_embeddings_data.get('trained'):
    print(f"\n{'='*70}")
    print("3D CHUNK SHIFT VISUALIZATION")
    print(f"{'='*70}")

    df_chunks = chunk_embeddings_data['df']
    coords_pre = chunk_embeddings_data['base']['coords_3d']
    coords_post = chunk_embeddings_data['trained']['coords_3d']
    var_exp = chunk_embeddings_data['trained']['variance_3d']

    print(f"\nCreating 3D shift visualization...")
    print(f"  Chunks: {len(df_chunks)}")

    fig_3d = go.Figure()

    # --------------------------------------------------------
    # Plot Pre-Training Positions (Light)
    # --------------------------------------------------------

    print(f"  Adding pre-training positions...")

    for topic in topics_viz:
        topic_df = df_chunks[df_chunks['primary_topic'] == topic]
        if len(topic_df) == 0:
            continue

        topic_indices = topic_df.index.tolist()
        local_indices = [df_chunks.index.get_loc(idx) for idx in topic_indices]

        topic_x = coords_pre[local_indices, 0]
        topic_y = coords_pre[local_indices, 1]
        topic_z = coords_pre[local_indices, 2]

        hover_texts = [
            f"<b>Chunk {row.get('chunk_uid', idx)}</b> (Pre)<br>"
            f"Topic: {row['primary_topic']}<br>"
            f"Shift: {row.get('shift_magnitude', 0):.3f}"
            for idx, row in topic_df.iterrows()
        ]

        fig_3d.add_trace(go.Scatter3d(
            x=topic_x,
            y=topic_y,
            z=topic_z,
            mode='markers',
            name=f'{topic} (Pre)',
            marker=dict(
                size=3,
                color=topic_colors.get(topic, 'gray'),
                opacity=0.3,
            ),
            hovertemplate='%{hovertext}<extra></extra>',
            hovertext=hover_texts,
            legendgroup=topic
        ))

    # --------------------------------------------------------
    # Plot Post-Training Positions (Bold)
    # --------------------------------------------------------

    print(f"  Adding post-training positions...")

    for topic in topics_viz:
        topic_df = df_chunks[df_chunks['primary_topic'] == topic]
        if len(topic_df) == 0:
            continue

        topic_indices = topic_df.index.tolist()
        local_indices = [df_chunks.index.get_loc(idx) for idx in topic_indices]

        topic_x = coords_post[local_indices, 0]
        topic_y = coords_post[local_indices, 1]
        topic_z = coords_post[local_indices, 2]

        hover_texts = [
            f"<b>Chunk {row.get('chunk_uid', idx)}</b> (Post)<br>"
            f"Topic: {row['primary_topic']}<br>"
            f"Shift: {row.get('shift_magnitude', 0):.3f}"
            for idx, row in topic_df.iterrows()
        ]

        fig_3d.add_trace(go.Scatter3d(
            x=topic_x,
            y=topic_y,
            z=topic_z,
            mode='markers',
            name=f'{topic} (Post)',
            marker=dict(
                size=4,
                color=topic_colors.get(topic, 'gray'),
                opacity=0.8,
            ),
            hovertemplate='%{hovertext}<extra></extra>',
            hovertext=hover_texts,
            legendgroup=topic
        ))

    # --------------------------------------------------------
    # Add Shift Vectors for Top Shifters
    # --------------------------------------------------------

    print(f"  Adding shift vectors for significant shifts...")

    shift_threshold = np.percentile(df_chunks['shift_magnitude'], 90)  # Top 10%
    significant_shifts = df_chunks[df_chunks['shift_magnitude'] > shift_threshold]

    # Limit to max 200 arrows for performance
    if len(significant_shifts) > 200:
        significant_shifts = significant_shifts.nlargest(200, 'shift_magnitude')

    print(f"    Showing {len(significant_shifts)} shift vectors (top shifters)")

    for idx, row in significant_shifts.iterrows():
        local_idx = df_chunks.index.get_loc(idx)

        fig_3d.add_trace(go.Scatter3d(
            x=[coords_pre[local_idx, 0], coords_post[local_idx, 0]],
            y=[coords_pre[local_idx, 1], coords_post[local_idx, 1]],
            z=[coords_pre[local_idx, 2], coords_post[local_idx, 2]],
            mode='lines',
            line=dict(color='red', width=2, dash='dash'),
            hovertemplate=f"<b>Shift: {row['shift_magnitude']:.3f}</b><extra></extra>",
            showlegend=False
        ))

    # --------------------------------------------------------
    # Update Layout
    # --------------------------------------------------------

    fig_3d.update_layout(
        title=(
            f"<b>Corpus Chunks: Training-Induced Semantic Shifts (3D)</b><br>"
            f"<sub>Variance: {var_exp.sum():.1%} | "
            f"Light = Pre | Bold = Post | "
            f"Red arrows = Top 10% shifters | "
            f"Rotate to explore</sub>"
        ),
        scene=dict(
            xaxis=dict(
                title=f'PC1 ({var_exp[0]:.1%})',
                backgroundcolor="rgb(245, 245, 245)",
                gridcolor="white",
                showbackground=True
            ),
            yaxis=dict(
                title=f'PC2 ({var_exp[1]:.1%})',
                backgroundcolor="rgb(245, 245, 245)",
                gridcolor="white",
                showbackground=True
            ),
            zaxis=dict(
                title=f'PC3 ({var_exp[2]:.1%})',
                backgroundcolor="rgb(245, 245, 245)",
                gridcolor="white",
                showbackground=True
            ),
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.3)
            )
        ),
        width=1400,
        height=900,
        font=dict(size=11),
        showlegend=True,
        legend=dict(
            x=1.0,
            y=1.0,
            font=dict(size=8),
            bgcolor='rgba(255,255,255,0.8)'
        ),
        template='plotly_white'
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    output_path = visuals_path_viz / 'chunks_shifts_3d.html'
    fig_3d.write_html(str(output_path))

    print(f"\n{'='*70}")
    print(f"3D CHUNK SHIFT VISUALIZATION COMPLETE")
    print(f"{'='*70}")
    print(f"  Saved to: {output_path.relative_to(fs.root)}")
    print(f"  Shift vectors shown: {len(significant_shifts)}")

    try:
        fig_3d.show()
    except:
        print(f"  (Interactive display not available)")

    print(f"\n✓ Cell 9.4B Complete!")

else:
    if 'chunk_embeddings_data' not in globals():
        print("⚠ Skipping 3D chunk shifts - run Cell 9.3A first")
    elif not chunk_embeddings_data.get('trained'):
        print("⚠ Skipping 3D chunk shifts - only pre-training data available")



3D CHUNK SHIFT VISUALIZATION

Creating 3D shift visualization...
  Chunks: 928
  Adding pre-training positions...
  Adding post-training positions...
  Adding shift vectors for significant shifts...
    Showing 93 shift vectors (top shifters)

3D CHUNK SHIFT VISUALIZATION COMPLETE
  Saved to: Visuals\chunks_shifts_3d.html
  Shift vectors shown: 93



✓ Cell 9.4B Complete!


In [26]:
# ============================================================
# CELL 9.5: 3D CROSS-TOPIC VISUALIZATION
# ============================================================

if VIZ_AVAILABLE:
    print(f"\n{'='*70}")
    print("CREATING 3D CROSS-TOPIC VISUALIZATION")
    print(f"{'='*70}")

    df_plot = cross_topic_viz_data['df']
    centroids_3d = cross_topic_viz_data['centroids_pca_3d']
    var_exp = cross_topic_viz_data['explained_variance_3d']

    # --------------------------------------------------------
    # Create Figure
    # --------------------------------------------------------

    fig_3d = go.Figure()

    print(f"\nAdding chunks to 3D plot...")

    # --------------------------------------------------------
    # Plot Chunks Grouped by Primary Topic
    # --------------------------------------------------------

    for topic in topics_viz:
        topic_df = df_plot[df_plot['primary_topic'] == topic]

        if len(topic_df) == 0:
            continue

        print(f"  {topic}: {len(topic_df)} chunks")

        # Build hover text (similar to 2D)
        hover_texts = []
        for idx, row in topic_df.iterrows():
            all_scores = []
            for tc, tn in zip(topic_cols_cosine_viz, topics_viz):
                score = row.get(tc, 0)
                all_scores.append(f"  • {tn}: {score:.3f}")

            hover_parts = [
                f"<b>Chunk ID:</b> {row.get('chunk_uid', 'N/A')}",
                f"<b>Primary Topic:</b> {row['primary_topic']}",
                f"<b>Primary Score:</b> {row['primary_score']:.3f}",
                f"<b>Distance to Centroid:</b> {row.get('distance_to_centroid', 0):.3f}",
                "<br><b>All Scores:</b>",
            ] + all_scores[:4]  # Limit to top 4 for readability
            hover_parts.append(f"<br>{row.get('snippet', '')[:250]}")

            hover_texts.append("<br>".join(hover_parts))

        # Add 3D scatter trace
        fig_3d.add_trace(go.Scatter3d(
            x=topic_df['pca_x'],
            y=topic_df['pca_y'],
            z=topic_df['pca_z'],
            mode='markers',
            name=topic,
            marker=dict(
                size=4,
                color=topic_colors[topic],
                opacity=0.7,
                line=dict(width=0.5, color='white')
            ),
            hovertemplate='%{hovertext}<extra></extra>',
            hovertext=hover_texts,
            legendgroup=topic,
        ))

    # --------------------------------------------------------
    # Plot Topic Centroids in 3D
    # --------------------------------------------------------

    print(f"\nAdding topic centroids to 3D plot...")

    centroid_x = []
    centroid_y = []
    centroid_z = []
    centroid_names = []
    centroid_hover = []

    for topic, coords in centroids_3d.items():
        centroid_x.append(coords[0])
        centroid_y.append(coords[1])
        centroid_z.append(coords[2])
        centroid_names.append(topic[:20])  # Shorten for 3D display

        n_chunks = (df_plot['primary_topic'] == topic).sum()
        hover_text = (
            f"<b>CENTROID: {topic}</b><br>"
            f"<b>Chunks:</b> {n_chunks}<br>"
            f"<b>Position:</b> ({coords[0]:.2f}, {coords[1]:.2f}, {coords[2]:.2f})"
        )
        centroid_hover.append(hover_text)

    fig_3d.add_trace(go.Scatter3d(
        x=centroid_x,
        y=centroid_y,
        z=centroid_z,
        mode='markers+text',
        name='Centroids',
        text=centroid_names,
        textposition='top center',
        textfont=dict(size=8, color='black'),
        marker=dict(
            size=10,
            color='red',
            symbol='diamond',
            line=dict(width=2, color='darkred')
        ),
        hovertemplate='%{hovertext}<extra></extra>',
        hovertext=centroid_hover,
        showlegend=True,
        legendgroup='centroids',
    ))

    # --------------------------------------------------------
    # Update Layout
    # --------------------------------------------------------

    fig_3d.update_layout(
        title=(
            f'<b>Cross-Topic Semantic Space (3D PCA Projection)</b><br>'
            f'<sub>Variance explained: {var_exp.sum():.1%} '
            f'(PC1: {var_exp[0]:.1%}, PC2: {var_exp[1]:.1%}, PC3: {var_exp[2]:.1%}) | '
            f'Rotate to explore</sub>'
        ),
        scene=dict(
            xaxis=dict(
                title=f'PC1 ({var_exp[0]:.1%})',
                backgroundcolor="rgb(245, 245, 245)",
                gridcolor="white",
                showbackground=True
            ),
            yaxis=dict(
                title=f'PC2 ({var_exp[1]:.1%})',
                backgroundcolor="rgb(245, 245, 245)",
                gridcolor="white",
                showbackground=True
            ),
            zaxis=dict(
                title=f'PC3 ({var_exp[2]:.1%})',
                backgroundcolor="rgb(245, 245, 245)",
                gridcolor="white",
                showbackground=True
            ),
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.3)
            )
        ),
        width=1400,
        height=900,
        font=dict(size=11),
        showlegend=True,
        legend=dict(
            x=1.0,
            y=1.0,
            xanchor='right',
            yanchor='top',
            font=dict(size=9),
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='lightgray',
            borderwidth=1
        ),
        template='plotly_white'
    )

    # --------------------------------------------------------
    # Save Visualization
    # --------------------------------------------------------

    output_path = visuals_path_viz / 'cross_topic_space_3d.html'
    fig_3d.write_html(str(output_path))

    print(f"\n{'='*70}")
    print(f"3D VISUALIZATION COMPLETE")
    print(f"{'='*70}")
    print(f"  Saved to: {output_path.relative_to(fs.root)}")
    print(f"  Chunks plotted: {len(df_plot)}")
    print(f"  Variance explained: {var_exp.sum():.1%}")

    # Display in notebook
    try:
        fig_3d.show()
    except:
        print(f"  (Interactive display not available)")

    print(f"\nCell 9.5 Complete!")

else:
    print("Skipping 3D visualization - not available")



CREATING 3D CROSS-TOPIC VISUALIZATION

Adding chunks to 3D plot...
  Colonial Systems: 564 chunks
  Heritage & Memory: 526 chunks
  Historical Slavery: 879 chunks
  Modern Racism & Discrimination: 871 chunks

Adding topic centroids to 3D plot...

3D VISUALIZATION COMPLETE
  Saved to: Visuals\cross_topic_space_3d.html
  Chunks plotted: 2840
  Variance explained: 95.1%



Cell 9.5 Complete!


In [27]:
# ============================================================
# CELL 9.6: DICTIONARY TERMS IN SEMANTIC SPACE
# ============================================================

if VIZ_AVAILABLE:
    print(f"\n{'='*70}")
    print("DICTIONARY TERMS IN SEMANTIC SPACE")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # Load Dictionary
    # --------------------------------------------------------

    print(f"\nLoading dictionary...")

    dict_path = source_fs.folders.get('Dictionary') / 'Curated_dictionary.csv'
    if not dict_path.exists():
        print(f"  ✗ Dictionary not found: {dict_path}")
        VIZ_AVAILABLE = False
    else:
        df_dict = pd.read_csv(dict_path)
        print(f"  ✓ Loaded {len(df_dict)} dictionary terms")

        # Identify columns
        has_is_seed = 'is_seed' in df_dict.columns
        has_weight = 'weight' in df_dict.columns
        has_parent = 'parent_seed' in df_dict.columns

        if has_is_seed:
            n_seed = df_dict['is_seed'].sum()
            n_expanded = len(df_dict) - n_seed
            print(f"    - Seed terms: {n_seed}")
            print(f"    - Expanded terms: {n_expanded}")

    # --------------------------------------------------------
    # Load BERTJE Models for Embeddings
    # --------------------------------------------------------

    if VIZ_AVAILABLE:
        print(f"\n{'='*70}")
        print("LOADING BERTJE MODELS FOR EMBEDDINGS")
        print(f"{'='*70}")

        from transformers import AutoModel, AutoTokenizer
        import torch

        # Check for trained model
        trained_model_path = source_fs.folders.get('Model_finetuning')
        base_model_path = source_fs.folders.get('Model_finetuning') / 'base_encoder' if trained_model_path else None

        has_trained_model = False
        has_base_model = False

        # Try to load trained (post-training) model
        if trained_model_path and (trained_model_path / 'trained_encoder').exists():
            try:
                print(f"\n  Loading trained (post-training) BERTJE model...")
                tokenizer_trained = AutoTokenizer.from_pretrained(str(trained_model_path / 'trained_encoder'))
                model_trained = AutoModel.from_pretrained(str(trained_model_path / 'trained_encoder'))
                model_trained.eval()
                has_trained_model = True
                print(f"    ✓ Loaded: {trained_model_path / 'trained_encoder'}")
            except Exception as e:
                print(f"    ✗ Failed to load trained model: {e}")

        # Try to load base (pre-training) model
        if base_model_path and base_model_path.exists():
            try:
                print(f"\n  Loading base (pre-training) BERTJE model...")
                tokenizer_base = AutoTokenizer.from_pretrained(str(base_model_path))
                model_base = AutoModel.from_pretrained(str(base_model_path))
                model_base.eval()
                has_base_model = True
                print(f"    ✓ Loaded: {base_model_path}")
            except Exception as e:
                print(f"    ✗ Failed to load base model: {e}")

        # If neither available, try default GroNLP/bert-base-dutch-cased
        if not has_trained_model and not has_base_model:
            try:
                print(f"\n  Loading default GroNLP BERTJE model...")
                tokenizer_base = AutoTokenizer.from_pretrained('GroNLP/bert-base-dutch-cased')
                model_base = AutoModel.from_pretrained('GroNLP/bert-base-dutch-cased')
                model_base.eval()
                has_base_model = True
                print(f"    ✓ Loaded: GroNLP/bert-base-dutch-cased")
            except Exception as e:
                print(f"    ✗ Failed to load default BERTJE: {e}")
                VIZ_AVAILABLE = False

        if has_trained_model and has_base_model:
            print(f"\n  ✓ Both pre and post-training models available for comparison")
        elif has_trained_model:
            print(f"\n  ⚠ Only post-training model available (no comparison)")
        elif has_base_model:
            print(f"\n  ⚠ Only pre-training model available (no comparison)")
        else:
            print(f"\n  ✗ No models available for embedding")
            VIZ_AVAILABLE = False

    # --------------------------------------------------------
    # Generate Dictionary Term Embeddings
    # --------------------------------------------------------

    if VIZ_AVAILABLE:
        print(f"\n{'='*70}")
        print("GENERATING DICTIONARY EMBEDDINGS")
        print(f"{'='*70}")

        def get_embeddings(texts, tokenizer, model, batch_size=32):
            """Generate embeddings for list of texts using BERTJE model"""
            embeddings = []

            for i in range(0, len(texts), batch_size):
                batch_texts = texts[i:i+batch_size]

                # Tokenize
                inputs = tokenizer(
                    batch_texts,
                    padding=True,
                    truncation=True,
                    max_length=128,
                    return_tensors='pt'
                )

                # Generate embeddings
                with torch.no_grad():
                    outputs = model(**inputs)
                    # Use CLS token embedding
                    batch_embeddings = outputs.last_hidden_state[:, 0, :].numpy()
                    embeddings.append(batch_embeddings)

            return np.vstack(embeddings)

        # Get dictionary terms
        dict_terms = df_dict['term'].tolist() if 'term' in df_dict.columns else df_dict.iloc[:, 0].tolist()

        print(f"\n  Embedding {len(dict_terms)} dictionary terms...")

        # Generate embeddings with available models
        embeddings_base = None
        embeddings_trained = None

        if has_base_model:
            print(f"\n  Generating pre-training embeddings...")
            embeddings_base = get_embeddings(dict_terms, tokenizer_base, model_base)
            print(f"    ✓ Shape: {embeddings_base.shape}")

        if has_trained_model:
            print(f"\n  Generating post-training embeddings...")
            embeddings_trained = get_embeddings(dict_terms, tokenizer_trained, model_trained)
            print(f"    ✓ Shape: {embeddings_trained.shape}")

        # --------------------------------------------------------
        # Dimensionality Reduction for Dictionary Terms
        # --------------------------------------------------------

        print(f"\n{'='*70}")
        print("DIMENSIONALITY REDUCTION (PCA)")
        print(f"{'='*70}")

        # We'll project dictionary terms into same PCA space as chunks
        # This allows direct comparison

        dict_viz_data = {}

        if has_base_model and embeddings_base is not None:
            print(f"\n  Projecting pre-training embeddings to PCA space...")

            # Note: We need to use the same scaler/PCA from chunk visualization
            # But dictionary embeddings are in different space than topic scores
            # So we'll do separate PCA for dictionary terms

            scaler_dict_base = StandardScaler()
            emb_scaled_base = scaler_dict_base.fit_transform(embeddings_base)

            pca_dict_2d_base = PCA(n_components=2, random_state=42)
            emb_pca_2d_base = pca_dict_2d_base.fit_transform(emb_scaled_base)

            pca_dict_3d_base = PCA(n_components=3, random_state=42)
            emb_pca_3d_base = pca_dict_3d_base.fit_transform(emb_scaled_base)

            var_2d_base = pca_dict_2d_base.explained_variance_ratio_
            var_3d_base = pca_dict_3d_base.explained_variance_ratio_

            print(f"    2D PCA: {var_2d_base.sum():.2%} variance")
            print(f"    3D PCA: {var_3d_base.sum():.2%} variance")

            dict_viz_data['base'] = {
                'embeddings': embeddings_base,
                'pca_2d': emb_pca_2d_base,
                'pca_3d': emb_pca_3d_base,
                'pca_model_2d': pca_dict_2d_base,
                'pca_model_3d': pca_dict_3d_base,
                'scaler': scaler_dict_base,
                'variance_2d': var_2d_base,
                'variance_3d': var_3d_base,
            }

        if has_trained_model and embeddings_trained is not None:
            print(f"\n  Projecting post-training embeddings to PCA space...")

            scaler_dict_trained = StandardScaler()
            emb_scaled_trained = scaler_dict_trained.fit_transform(embeddings_trained)

            pca_dict_2d_trained = PCA(n_components=2, random_state=42)
            emb_pca_2d_trained = pca_dict_2d_trained.fit_transform(emb_scaled_trained)

            pca_dict_3d_trained = PCA(n_components=3, random_state=42)
            emb_pca_3d_trained = pca_dict_3d_trained.fit_transform(emb_scaled_trained)

            var_2d_trained = pca_dict_2d_trained.explained_variance_ratio_
            var_3d_trained = pca_dict_3d_trained.explained_variance_ratio_

            print(f"    2D PCA: {var_2d_trained.sum():.2%} variance")
            print(f"    3D PCA: {var_3d_trained.sum():.2%} variance")

            dict_viz_data['trained'] = {
                'embeddings': embeddings_trained,
                'pca_2d': emb_pca_2d_trained,
                'pca_3d': emb_pca_3d_trained,
                'pca_model_2d': pca_dict_2d_trained,
                'pca_model_3d': pca_dict_3d_trained,
                'scaler': scaler_dict_trained,
                'variance_2d': var_2d_trained,
                'variance_3d': var_3d_trained,
            }

        # --------------------------------------------------------
        # Prepare Dictionary DataFrame for Visualization
        # --------------------------------------------------------

        print(f"\n  Preparing dictionary data for visualization...")

        # Add term info
        df_dict['term_text'] = dict_terms
        df_dict['term_length'] = df_dict['term_text'].str.len()

        # Identify seed vs expanded
        if has_is_seed:
            df_dict['term_type'] = df_dict['is_seed'].apply(lambda x: 'Seed' if x else 'Expanded')
        else:
            df_dict['term_type'] = 'Unknown'

        # Get topic assignment
        if 'topic' in df_dict.columns:
            df_dict['topic_assigned'] = df_dict['topic']
        elif 'category' in df_dict.columns:
            df_dict['topic_assigned'] = df_dict['category']
        else:
            df_dict['topic_assigned'] = 'Unknown'

        # Store in dict_viz_data
        dict_viz_data['df'] = df_dict

        print(f"\n{'='*70}")
        print(f"✓ DICTIONARY EMBEDDINGS PREPARED")
        print(f"{'='*70}")
        print(f"  Terms: {len(df_dict)}")
        print(f"  Pre-training embeddings: {'Available' if 'base' in dict_viz_data else 'Not available'}")
        print(f"  Post-training embeddings: {'Available' if 'trained' in dict_viz_data else 'Not available'}")

        print(f"\n✓ Cell 9.6 Complete - Ready for dictionary visualization!")

else:
    print("⚠ Skipping dictionary embeddings - visualization not available")



DICTIONARY TERMS IN SEMANTIC SPACE

Loading dictionary...
  ✓ Loaded 1040 dictionary terms
    - Seed terms: 133
    - Expanded terms: 907

LOADING BERTJE MODELS FOR EMBEDDINGS

  Loading trained (post-training) BERTJE model...
    ✓ Loaded: C:\Users\Home\policy-analysis\workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Model_finetuning\trained_encoder

  Loading base (pre-training) BERTJE model...
    ✓ Loaded: C:\Users\Home\policy-analysis\workflow_data\slavery_Short-slavdict_pretrained_slavery_v4\Model_finetuning\base_encoder

  ✓ Both pre and post-training models available for comparison

GENERATING DICTIONARY EMBEDDINGS

  Embedding 1040 dictionary terms...

  Generating pre-training embeddings...
    ✓ Shape: (1040, 768)

  Generating post-training embeddings...
    ✓ Shape: (1040, 768)

DIMENSIONALITY REDUCTION (PCA)

  Projecting pre-training embeddings to PCA space...
    2D PCA: 13.57% variance
    3D PCA: 18.75% variance

  Projecting post-training embeddings to PC

In [28]:
# ============================================================
# CELL 9.7: 2D DICTIONARY TERMS VISUALIZATION
# ============================================================

if VIZ_AVAILABLE and 'dict_viz_data' in globals():
    print(f"\n{'='*70}")
    print("2D DICTIONARY VISUALIZATION")
    print(f"{'='*70}")

    df_dict_plot = dict_viz_data['df']

    # Color mapping for topics (same as chunks)
    topic_colors = {topic: color_palette[i % len(color_palette)]
                    for i, topic in enumerate(topics_viz)}

    # Additional colors for unknown/multi-topic
    topic_colors['Unknown'] = 'gray'
    topic_colors['Multi-topic'] = 'purple'

    # --------------------------------------------------------
    # Create Comparison Figure (Pre vs Post Training)
    # --------------------------------------------------------

    has_both = 'base' in dict_viz_data and 'trained' in dict_viz_data
    has_base_only = 'base' in dict_viz_data and 'trained' not in dict_viz_data
    has_trained_only = 'trained' in dict_viz_data and 'base' not in dict_viz_data

    if has_both:
        print(f"\n  Creating side-by-side comparison (pre vs post training)...")

        # Create subplots
        from plotly.subplots import make_subplots

        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=['<b>Pre-Training</b> (Base BERTJE)', '<b>Post-Training</b> (Fine-tuned)'],
            horizontal_spacing=0.10
        )

        models = [('base', 1), ('trained', 2)]

    else:
        print(f"\n  Creating single visualization...")
        fig = go.Figure()
        if has_base_only:
            models = [('base', None)]
            title_suffix = " (Pre-Training - Base BERTJE)"
        else:
            models = [('trained', None)]
            title_suffix = " (Post-Training - Fine-tuned)"

    # --------------------------------------------------------
    # Plot Dictionary Terms for Each Model
    # --------------------------------------------------------

    for model_type, col in models:
        data = dict_viz_data[model_type]
        coords_2d = data['pca_2d']
        var_exp = data['variance_2d']

        print(f"\n  Plotting {model_type} model...")

        # Add coordinates to dataframe
        df_plot = df_dict_plot.copy()
        df_plot['pca_x'] = coords_2d[:, 0]
        df_plot['pca_y'] = coords_2d[:, 1]

        # Group by topic
        for topic in df_plot['topic_assigned'].unique():
            topic_df = df_plot[df_plot['topic_assigned'] == topic]

            if len(topic_df) == 0:
                continue

            # Separate seed and expanded terms
            if 'term_type' in topic_df.columns:
                seed_df = topic_df[topic_df['term_type'] == 'Seed']
                expanded_df = topic_df[topic_df['term_type'] == 'Expanded']
            else:
                seed_df = topic_df
                expanded_df = pd.DataFrame()

            # Build hover texts
            def build_hover(row):
                parts = [
                    f"<b>Term:</b> {row['term_text']}",
                    f"<b>Topic:</b> {row['topic_assigned']}",
                    f"<b>Type:</b> {row.get('term_type', 'Unknown')}",
                ]
                if 'weight' in row and pd.notna(row['weight']):
                    parts.append(f"<b>Weight:</b> {row['weight']:.2f}")
                if 'parent_seed' in row and pd.notna(row['parent_seed']):
                    parts.append(f"<b>Parent:</b> {row['parent_seed']}")
                return "<br>".join(parts)

            # Plot seed terms (larger, solid)
            if len(seed_df) > 0:
                hover_texts = [build_hover(row) for _, row in seed_df.iterrows()]

                trace = go.Scatter(
                    x=seed_df['pca_x'],
                    y=seed_df['pca_y'],
                    mode='markers+text',
                    name=f'{topic} (Seed)',
                    text=seed_df['term_text'].str[:15],  # Abbreviated labels
                    textposition='top center',
                    textfont=dict(size=8),
                    marker=dict(
                        size=12,
                        color=topic_colors.get(topic, 'gray'),
                        opacity=0.9,
                        symbol='diamond',
                        line=dict(width=2, color='black')
                    ),
                    hovertemplate='%{hovertext}<extra></extra>',
                    hovertext=hover_texts,
                    legendgroup=topic,
                    showlegend=(col == 1 or col is None),  # Show legend only once
                )

                if col is not None:
                    fig.add_trace(trace, row=1, col=col)
                else:
                    fig.add_trace(trace)

            # Plot expanded terms (smaller, hollow)
            if len(expanded_df) > 0:
                hover_texts = [build_hover(row) for _, row in expanded_df.iterrows()]

                trace = go.Scatter(
                    x=expanded_df['pca_x'],
                    y=expanded_df['pca_y'],
                    mode='markers',
                    name=f'{topic} (Expanded)',
                    marker=dict(
                        size=6,
                        color=topic_colors.get(topic, 'gray'),
                        opacity=0.6,
                        line=dict(width=1, color=topic_colors.get(topic, 'gray'))
                    ),
                    hovertemplate='%{hovertext}<extra></extra>',
                    hovertext=hover_texts,
                    legendgroup=topic,
                    showlegend=False,  # Don't clutter legend
                )

                if col is not None:
                    fig.add_trace(trace, row=1, col=col)
                else:
                    fig.add_trace(trace)

        print(f"    ✓ Plotted {len(df_plot)} terms")

        # Add variance annotation
        if col is not None:
            fig.add_annotation(
                text=f"<i>Variance: {var_exp.sum():.1%}</i>",
                xref=f'x{col}',
                yref=f'y{col}',
                x=0.02,
                y=0.98,
                xanchor='left',
                yanchor='top',
                showarrow=False,
                font=dict(size=10, color='gray'),
                row=1,
                col=col
            )

    # --------------------------------------------------------
    # Update Layout
    # --------------------------------------------------------

    if has_both:
        title_text = (
            '<b>Dictionary Terms: Pre vs Post Training Comparison</b><br>'
            '<sub>Diamonds = Seed terms | Circles = Expanded terms | '
            'Size indicates importance | Compare clustering between models</sub>'
        )
        height = 700
        width = 1600
    else:
        title_text = (
            f'<b>Dictionary Terms in Semantic Space{title_suffix}</b><br>'
            '<sub>Diamonds = Seed terms | Circles = Expanded terms | '
            'Hover for details</sub>'
        )
        height = 800
        width = 1200

    fig.update_layout(
        title_text=title_text,
        height=height,
        width=width,
        template='plotly_white',
        hovermode='closest',
        font=dict(size=11),
        showlegend=True,
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02,
            font=dict(size=9),
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='lightgray',
            borderwidth=1
        )
    )

    # Update axes
    if has_both:
        for col in [1, 2]:
            fig.update_xaxes(
                title_text='PC1',
                showgrid=True,
                gridcolor='lightgray',
                zeroline=True,
                row=1,
                col=col
            )
            fig.update_yaxes(
                title_text='PC2',
                showgrid=True,
                gridcolor='lightgray',
                zeroline=True,
                row=1,
                col=col
            )
    else:
        fig.update_xaxes(
            title_text='PC1',
            showgrid=True,
            gridcolor='lightgray',
            zeroline=True
        )
        fig.update_yaxes(
            title_text='PC2',
            showgrid=True,
            gridcolor='lightgray',
            zeroline=True
        )

    # --------------------------------------------------------
    # Save Visualization
    # --------------------------------------------------------

    if has_both:
        output_path = visuals_path_viz / 'dictionary_terms_comparison_2d.html'
    elif has_trained_only:
        output_path = visuals_path_viz / 'dictionary_terms_trained_2d.html'
    else:
        output_path = visuals_path_viz / 'dictionary_terms_base_2d.html'

    fig.write_html(str(output_path))

    print(f"\n{'='*70}")
    print(f"2D DICTIONARY VISUALIZATION COMPLETE")
    print(f"{'='*70}")
    print(f"  Saved to: {output_path.relative_to(fs.root)}")
    print(f"  Terms plotted: {len(df_dict_plot)}")
    if has_both:
        print(f"  Comparison: Pre-training vs Post-training")

    # Display
    try:
        fig.show()
    except:
        print(f"  (Interactive display not available)")

    print(f"\n✓ Cell 9.7 Complete!")

else:
    print("⚠ Skipping 2D dictionary visualization - data not available")



2D DICTIONARY VISUALIZATION

  Creating side-by-side comparison (pre vs post training)...

  Plotting base model...
    ✓ Plotted 1040 terms

  Plotting trained model...
    ✓ Plotted 1040 terms

2D DICTIONARY VISUALIZATION COMPLETE
  Saved to: Visuals\dictionary_terms_comparison_2d.html
  Terms plotted: 1040
  Comparison: Pre-training vs Post-training



✓ Cell 9.7 Complete!


In [29]:
# ============================================================
# CELL 9.8: 3D DICTIONARY TERMS VISUALIZATION (WITH SHIFT ANALYSIS)
# ============================================================

if VIZ_AVAILABLE and 'dict_viz_data' in globals():
    print(f"\n{'='*70}")
    print("3D DICTIONARY VISUALIZATION")
    print(f"{'='*70}")

    df_dict_plot = dict_viz_data['df']
    has_both = 'base' in dict_viz_data and 'trained' in dict_viz_data

    # --------------------------------------------------------
    # Create 3D Visualizations
    # --------------------------------------------------------

    if has_both:
        print(f"\n  Creating 3D comparison with shift vectors...")

        # --------------------------------------------------------
        # Calculate Shifts (Pre to Post Training)
        # --------------------------------------------------------

        coords_pre = dict_viz_data['base']['pca_3d']
        coords_post = dict_viz_data['trained']['pca_3d']

        # Calculate shift vectors
        shifts = coords_post - coords_pre
        shift_magnitudes = np.linalg.norm(shifts, axis=1)

        # Add to dataframe
        df_dict_plot['shift_magnitude'] = shift_magnitudes
        df_dict_plot['shift_x'] = shifts[:, 0]
        df_dict_plot['shift_y'] = shifts[:, 1]
        df_dict_plot['shift_z'] = shifts[:, 2]

        print(f"    Shift statistics:")
        print(f"      Mean: {shift_magnitudes.mean():.3f}")
        print(f"      Median: {np.median(shift_magnitudes):.3f}")
        print(f"      Max: {shift_magnitudes.max():.3f}")

        # Identify top shifters
        top_shifters = df_dict_plot.nlargest(10, 'shift_magnitude')
        print(f"\n    Top 10 terms with largest shifts:")
        for idx, row in top_shifters.iterrows():
            print(f"      {row['term_text']:30s}: {row['shift_magnitude']:.3f}")

        # --------------------------------------------------------
        # Create Figure with Both Positions + Shift Vectors
        # --------------------------------------------------------

        fig_3d = go.Figure()

        # Plot pre-training positions (lighter, smaller)
        print(f"\n  Adding pre-training positions...")

        df_plot = df_dict_plot.copy()
        df_plot['pca_x'] = coords_pre[:, 0]
        df_plot['pca_y'] = coords_pre[:, 1]
        df_plot['pca_z'] = coords_pre[:, 2]

        for topic in df_plot['topic_assigned'].unique():
            topic_df = df_plot[df_plot['topic_assigned'] == topic]
            if len(topic_df) == 0:
                continue

            hover_texts = [
                f"<b>{row['term_text']}</b> (Pre-training)<br>"
                f"Topic: {row['topic_assigned']}<br>"
                f"Type: {row.get('term_type', 'Unknown')}<br>"
                f"Shift: {row['shift_magnitude']:.3f}"
                for _, row in topic_df.iterrows()
            ]

            fig_3d.add_trace(go.Scatter3d(
                x=topic_df['pca_x'],
                y=topic_df['pca_y'],
                z=topic_df['pca_z'],
                mode='markers',
                name=f'{topic} (Pre)',
                marker=dict(
                    size=4,
                    color=topic_colors.get(topic, 'gray'),
                    opacity=0.3,
                    symbol='circle'
                ),
                hovertemplate='%{hovertext}<extra></extra>',
                hovertext=hover_texts,
                legendgroup=topic,
                showlegend=True
            ))

        # Plot post-training positions (bolder, larger)
        print(f"  Adding post-training positions...")

        df_plot['pca_x'] = coords_post[:, 0]
        df_plot['pca_y'] = coords_post[:, 1]
        df_plot['pca_z'] = coords_post[:, 2]

        for topic in df_plot['topic_assigned'].unique():
            topic_df = df_plot[df_plot['topic_assigned'] == topic]
            if len(topic_df) == 0:
                continue

            # Separate seed vs expanded
            seed_df = topic_df[topic_df['term_type'] == 'Seed'] if 'term_type' in topic_df.columns else topic_df

            if len(seed_df) > 0:
                hover_texts = [
                    f"<b>{row['term_text']}</b> (Post-training)<br>"
                    f"Topic: {row['topic_assigned']}<br>"
                    f"Type: {row.get('term_type', 'Unknown')}<br>"
                    f"Shift: {row['shift_magnitude']:.3f}"
                    for _, row in seed_df.iterrows()
                ]

                fig_3d.add_trace(go.Scatter3d(
                    x=seed_df['pca_x'],
                    y=seed_df['pca_y'],
                    z=seed_df['pca_z'],
                    mode='markers+text',
                    name=f'{topic} (Post)',
                    text=seed_df['term_text'].str[:12],
                    textposition='top center',
                    textfont=dict(size=7),
                    marker=dict(
                        size=8,
                        color=topic_colors.get(topic, 'gray'),
                        opacity=0.9,
                        symbol='diamond',
                        line=dict(width=1, color='black')
                    ),
                    hovertemplate='%{hovertext}<extra></extra>',
                    hovertext=hover_texts,
                    legendgroup=topic,
                    showlegend=True
                ))

        # Plot shift vectors for significant shifts
        print(f"  Adding shift vectors...")

        significant_shifts = df_plot[df_plot['shift_magnitude'] > np.median(shift_magnitudes)]

        for _, row in significant_shifts.iterrows():
            # Arrow from pre to post
            fig_3d.add_trace(go.Scatter3d(
                x=[coords_pre[row.name, 0], coords_post[row.name, 0]],
                y=[coords_pre[row.name, 1], coords_post[row.name, 1]],
                z=[coords_pre[row.name, 2], coords_post[row.name, 2]],
                mode='lines',
                line=dict(color='red', width=2, dash='dash'),
                hovertemplate=f"<b>{row['term_text']}</b><br>Shift: {row['shift_magnitude']:.3f}<extra></extra>",
                showlegend=False
            ))

        print(f"    ✓ Added {len(significant_shifts)} shift vectors (above median)")

        var_exp = dict_viz_data['trained']['variance_3d']
        title_text = (
            f"<b>Dictionary Terms: Training-Induced Semantic Shifts (3D)</b><br>"
            f"<sub>Variance: {var_exp.sum():.1%} | "
            f"Light = Pre-training | Bold = Post-training | "
            f"Red arrows = Significant shifts | "
            f"Rotate to explore</sub>"
        )

    else:
        # Single model visualization
        print(f"\n  Creating single 3D visualization...")

        fig_3d = go.Figure()

        model_type = 'base' if 'base' in dict_viz_data else 'trained'
        data = dict_viz_data[model_type]
        coords_3d = data['pca_3d']
        var_exp = data['variance_3d']

        df_plot = df_dict_plot.copy()
        df_plot['pca_x'] = coords_3d[:, 0]
        df_plot['pca_y'] = coords_3d[:, 1]
        df_plot['pca_z'] = coords_3d[:, 2]

        for topic in df_plot['topic_assigned'].unique():
            topic_df = df_plot[df_plot['topic_assigned'] == topic]
            if len(topic_df) == 0:
                continue

            hover_texts = [
                f"<b>{row['term_text']}</b><br>"
                f"Topic: {row['topic_assigned']}<br>"
                f"Type: {row.get('term_type', 'Unknown')}"
                for _, row in topic_df.iterrows()
            ]

            fig_3d.add_trace(go.Scatter3d(
                x=topic_df['pca_x'],
                y=topic_df['pca_y'],
                z=topic_df['pca_z'],
                mode='markers+text',
                name=topic,
                text=topic_df['term_text'].str[:12],
                textposition='top center',
                textfont=dict(size=7),
                marker=dict(
                    size=6,
                    color=topic_colors.get(topic, 'gray'),
                    opacity=0.8
                ),
                hovertemplate='%{hovertext}<extra></extra>',
                hovertext=hover_texts,
                legendgroup=topic
            ))

        model_name = "Pre-training" if model_type == 'base' else "Post-training"
        title_text = (
            f"<b>Dictionary Terms in 3D Semantic Space ({model_name})</b><br>"
            f"<sub>Variance: {var_exp.sum():.1%} | Rotate to explore</sub>"
        )

    # --------------------------------------------------------
    # Update Layout
    # --------------------------------------------------------

    fig_3d.update_layout(
        title=title_text,
        scene=dict(
            xaxis=dict(
                title=f'PC1 ({var_exp[0]:.1%})',
                backgroundcolor="rgb(245, 245, 245)",
                gridcolor="white",
                showbackground=True
            ),
            yaxis=dict(
                title=f'PC2 ({var_exp[1]:.1%})',
                backgroundcolor="rgb(245, 245, 245)",
                gridcolor="white",
                showbackground=True
            ),
            zaxis=dict(
                title=f'PC3 ({var_exp[2]:.1%})',
                backgroundcolor="rgb(245, 245, 245)",
                gridcolor="white",
                showbackground=True
            ),
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.3)
            )
        ),
        width=1400,
        height=900,
        font=dict(size=11),
        showlegend=True,
        legend=dict(
            x=1.0,
            y=1.0,
            font=dict(size=9),
            bgcolor='rgba(255,255,255,0.8)'
        ),
        template='plotly_white'
    )

    # --------------------------------------------------------
    # Save Visualization
    # --------------------------------------------------------

    if has_both:
        output_path = visuals_path_viz / 'dictionary_terms_shifts_3d.html'
    elif 'trained' in dict_viz_data:
        output_path = visuals_path_viz / 'dictionary_terms_trained_3d.html'
    else:
        output_path = visuals_path_viz / 'dictionary_terms_base_3d.html'

    fig_3d.write_html(str(output_path))

    print(f"\n{'='*70}")
    print(f"3D DICTIONARY VISUALIZATION COMPLETE")
    print(f"{'='*70}")
    print(f"  Saved to: {output_path.relative_to(fs.root)}")
    if has_both:
        print(f"  Shows: Pre/Post training comparison with shift vectors")
        print(f"  Significant shifts: {len(significant_shifts)}")

    # Display
    try:
        fig_3d.show()
    except:
        print(f"  (Interactive display not available)")

    print(f"\n✓ Cell 9.8 Complete!")

else:
    print("⚠ Skipping 3D dictionary visualization - data not available")



3D DICTIONARY VISUALIZATION

  Creating 3D comparison with shift vectors...
    Shift statistics:
      Mean: 21.201
      Median: 20.714
      Max: 40.033

    Top 10 terms with largest shifts:
      handelsondernemingen          : 40.033
      west-indisch                  : 40.018
      west-indisch                  : 40.018
      westindische                  : 39.943
      west-indische                 : 39.921
      west-indische                 : 39.921
      handelsactiviteiten           : 39.792
      handelsactiviteiten           : 39.792
      slavenhandelaar               : 39.181
      handelsforten                 : 39.140

  Adding pre-training positions...
  Adding post-training positions...
  Adding shift vectors...
    ✓ Added 520 shift vectors (above median)

3D DICTIONARY VISUALIZATION COMPLETE
  Saved to: Visuals\dictionary_terms_shifts_3d.html
  Shows: Pre/Post training comparison with shift vectors
  Significant shifts: 520



✓ Cell 9.8 Complete!


In [33]:
# ============================================================

# CELL 9.6: TRAINING METRICS VISUALIZATION & CHECKPOINT SAVE

# ============================================================



if VIZ_AVAILABLE:

    print(f"\n{'='*70}")

    print("TRAINING METRICS VISUALIZATION")

    print(f"{'='*70}")



    if training_metrics_viz is not None:

        print("\n✓ Training metrics available, creating visualizations...")



        # --------------------------------------------------------

        # Extract Metrics

        # --------------------------------------------------------



        final_eval = training_metrics_viz.get('final_eval', {})

        num_train = training_metrics_viz.get('num_train_samples', 'N/A')

        num_val = training_metrics_viz.get('num_val_samples', 'N/A')



        # Get per-topic correlations and MAE

        topic_metrics = {}

        for topic_name in topics_viz:

            # Find matching metric keys (abbreviated topic names in keys)

            topic_abbrev = topic_name.split()[0]  # First word of topic



            for key in final_eval.keys():

                if 'eval_corr_' in key and topic_abbrev in key.replace('eval_corr_', ''):

                    corr = final_eval.get(key, None)

                    mae_key = key.replace('corr', 'mae')

                    mae = final_eval.get(mae_key, None)



                    if corr is not None:

                        topic_metrics[topic_name] = {

                            'correlation': corr,

                            'mae': mae if mae is not None else 0

                        }

                        break



        # Overall metrics

        mean_corr = final_eval.get('eval_mean_correlation', None)



        print(f"\n  Model Performance Summary:")

        print(f"    Training samples: {num_train}")

        print(f"    Validation samples: {num_val}")

        if mean_corr is not None:

            print(f"    Mean correlation: {mean_corr:.4f}")



        # --------------------------------------------------------

        # Visualization 1: Per-Topic Performance Bar Chart

        # --------------------------------------------------------



        if topic_metrics:

            fig_perf = go.Figure()



            topics_list = list(topic_metrics.keys())

            correlations = [topic_metrics[t]['correlation'] for t in topics_list]

            maes = [topic_metrics[t]['mae'] for t in topics_list]



            # Correlation bars

            fig_perf.add_trace(go.Bar(

                name='Correlation (r)',

                x=topics_list,

                y=correlations,

                marker_color='#3498db',

                text=[f'{v:.3f}' for v in correlations],

                textposition='outside',

                hovertemplate='<b>%{x}</b><br>Correlation: %{y:.4f}<extra></extra>'

            ))



            # MAE bars (inverted scale for comparison)

            fig_perf.add_trace(go.Bar(

                name='MAE (inverted)',

                x=topics_list,

                y=[-mae for mae in maes],  # Negative for visual comparison

                marker_color='#e74c3c',

                text=[f'{mae:.3f}' for mae in maes],

                textposition='outside',

                hovertemplate='<b>%{x}</b><br>MAE: %{text}<extra></extra>'

            ))



            # Add mean correlation line

            if mean_corr is not None:

                fig_perf.add_hline(

                    y=mean_corr,

                    line_dash="dash",

                    line_color="green",

                    annotation_text=f"Mean: {mean_corr:.3f}",

                    annotation_position="right"

                )



            fig_perf.update_layout(

                title=(

                    f'<b>BERTJE Model Performance by Topic</b><br>'

                    f'<sub>Continuous regression task | '

                    f'Train: {num_train} samples, Val: {num_val} samples</sub>'

                ),

                xaxis_title='Topic',

                yaxis_title='Metric Value',

                barmode='group',

                height=500,

                width=1200,

                template='plotly_white',

                font=dict(size=11),

                showlegend=True,

                legend=dict(

                    orientation="h",

                    yanchor="bottom",

                    y=1.02,

                    xanchor="center",

                    x=0.5

                ),

                yaxis=dict(

                    range=[-0.5, 1.1]  # Adjusted for inverted MAE

                )

            )



            # Update x-axis labels to wrap

            fig_perf.update_xaxes(tickangle=-45)



            output_path = visuals_path_viz / 'training_metrics_performance.html'

            fig_perf.write_html(str(output_path))

            print(f"\n  ✓ Saved: {output_path.name}")



            # Display in notebook

            try:

                fig_perf.show()

            except:

                pass



        # --------------------------------------------------------

        # Visualization 2: Metrics Summary Table

        # --------------------------------------------------------



        # Create summary table data

        topics_list = list(topic_metrics.keys())
        
        table_data = []

        for topic_name in topics_list:

            metrics = topic_metrics[topic_name]

            table_data.append([

                topic_name,

                f"{metrics['correlation']:.4f}",

                f"{metrics['mae']:.4f}"

            ])



        # Add mean row

        if mean_corr is not None:

            mean_mae = sum(maes) / len(maes) if maes else 0

            table_data.append([

                '<b>MEAN</b>',

                f"<b>{mean_corr:.4f}</b>",

                f"<b>{mean_mae:.4f}</b>"

            ])



        fig_table = go.Figure(data=[go.Table(

            header=dict(

                values=['<b>Topic</b>', '<b>Correlation (r)</b>', '<b>MAE</b>'],

                fill_color='#3498db',

                font=dict(color='white', size=12),

                align='left'

            ),

            cells=dict(

                values=list(zip(*table_data)),  # Transpose

                fill_color=[['white', 'white', 'white', 'lightgray']],

                font=dict(size=11),

                align='left',

                height=30

            )

        )])



        fig_table.update_layout(

            title=(

                f'<b>Training Metrics Summary</b><br>'

                f'<sub>Model training performance on validation set</sub>'

            ),

            height=300 + len(table_data) * 30,

            width=800,

            template='plotly_white'

        )



        output_path_table = visuals_path_viz / 'training_metrics_table.html'

        fig_table.write_html(str(output_path_table))

        print(f"  ✓ Saved: {output_path_table.name}")



        # --------------------------------------------------------

        # Print Summary to Console

        # --------------------------------------------------------



        print(f"\n  Performance Details:")

        for topic_name in topics_list:

            metrics = topic_metrics[topic_name]

            print(f"    {topic_name:40s}: r={metrics['correlation']:.4f}, MAE={metrics['mae']:.4f}")



    else:

        print("\n  ⚠ No training metrics available")

        print("    Run CHECKPOINT 8 (BERTJE training) to generate metrics")



    # --------------------------------------------------------

    # Save Checkpoint Configuration

    # --------------------------------------------------------



    print(f"\n{'='*70}")

    print("SAVING CHECKPOINT")

    print(f"{'='*70}")



    fs.save_config("checkpoint9_visuals")

    print("  ✓ Checkpoint saved: checkpoint9_visuals")



    # --------------------------------------------------------

    # Final Summary

    # --------------------------------------------------------



    print(f"\n{'='*70}")

    print("CHECKPOINT 9 COMPLETE - ALL VISUALIZATIONS GENERATED")

    print(f"{'='*70}")



    print(f"\n📊 Visualization Summary:")

    print(f"  Output folder: {visuals_path_viz.relative_to(fs.root)}")

    print(f"  Topics analyzed: {len(topic_metrics)}")

    # REMOVED: This line assumed a different data structure
    # print(f"  Total chunks visualized: {sum(len(r['data']) for r in topic_metrics.values())}")



    print(f"\n📁 Generated Files:")

    html_files = sorted(visuals_path_viz.glob('*.html'))

    if html_files:

        for file in html_files:

            size_kb = file.stat().st_size / 1024

            print(f"    • {file.name:50s} ({size_kb:6.1f} KB)")

    else:

        print(f"    (No HTML files found)")



    print(f"\n✓ Cell 9.6 Complete - Checkpoint 9 finished successfully!")



else:

    print("⚠ Skipping training metrics visualization - not available")



TRAINING METRICS VISUALIZATION

✓ Training metrics available, creating visualizations...

  Model Performance Summary:
    Training samples: 1587
    Validation samples: 568
  ✓ Saved: training_metrics_table.html

  Performance Details:

SAVING CHECKPOINT
✓ Config saved: config_checkpoint9_visuals_20260102_130238.json
  ✓ Checkpoint saved: checkpoint9_visuals

CHECKPOINT 9 COMPLETE - ALL VISUALIZATIONS GENERATED

📊 Visualization Summary:
  Output folder: Visuals
  Topics analyzed: 0

📁 Generated Files:
    • chunks_prepost_comparison_2d.html                  (4929.3 KB)
    • chunks_shifts_3d.html                              (4859.3 KB)
    • cross_topic_space_2d.html                          (7977.3 KB)
    • cross_topic_space_3d.html                          (6871.3 KB)
    • dictionary_terms_comparison_2d.html                (5086.9 KB)
    • dictionary_terms_shifts_3d.html                    (4947.1 KB)
    • training_metrics_table.html                        (4556.5 KB)

✓ Cell 

✅ **CHECKPOINT 9 COMPLETE** - Visualizations generated

**All checkpoints complete!** Check `Visuals/` folder for interactive plots.

---
# Workflow Complete! 🎉
---

## Summary

All checkpoints have been executed:

✅ **CHECKPOINT 0**: Setup & Configuration
✅ **CHECKPOINT 1**: Text Processing
✅ **CHECKPOINT 2**: Vocabulary Building
✅ **CHECKPOINT 3**: Dictionary Expansion
✅ **CHECKPOINT 4**: Topic Vectors
✅ **CHECKPOINT 5**: Chunk Scoring
✅ **CHECKPOINT 6**: Training Data Prep
✅ **CHECKPOINT 7**: Model Training
✅ **CHECKPOINT 8**: BERTJE Labeling
✅ **CHECKPOINT 9**: Visualizations

## Output Location

All outputs saved to: `{workflow_root}`

```
{ModelType}-{Topic}_{Date}_{Version}/
├── config/              # Config snapshots at each checkpoint
├── Dictionary/          # Input, expanded, curated dictionaries
│   └── Dictionary_suggestions/
├── Model_finetuning/    # Trained model + metrics
├── Cosine_labeling/     # Confidence-classified scores
├── Bertje_labeling/     # Model predictions
├── Visuals/             # Interactive HTML visualizations
└── Other_data/          # Chunks, vocabulary, topic vectors
```

## Next Steps

1. **Review Results**: Check visualizations in `Visuals/`
2. **Analyze Model**: Review training metrics
3. **Use Model**: Load trained model for predictions
4. **Iterate**: Adjust config and re-run from any checkpoint

## Using the Trained Model

To use this model in a new workflow:

```python
CONFIG['model']['use_pretrained'] = True
CONFIG['paths']['pretrained_model_path'] = 'path/to/Model_finetuning'
CONFIG['workflow']['model_type'] = 'Finetuned_{Source}'
```

See `WORKFLOW_GUIDE_v3.md` for complete documentation!